


En esta sección, integraremos un motor de físicas profesional en nuestro propio motor. Para demostrar su poder, construiremos el esqueleto de un juego tipo billar o pool, donde las colisiones, el momento, la fricción y los rebotes no son animaciones, sino el resultado de una simulación en tiempo real.

1. Discusión: La Belleza del Caos Controlado (Juegos de Físicas)
Los juegos basados en físicas tienen un atractivo único. En lugar de seguir un guion, se basan en un sistema de reglas (las leyes de la física) que pueden producir resultados inesperados, divertidos y profundamente satisfactorios. Esto se conoce como jugabilidad emergente.

Simulación vs. Guion: En nuestro plataformero, el salto siempre tiene la misma altura. En un juego de físicas, el resultado de una acción depende de un complejo conjunto de factores iniciales. Una pequeña variación en el ángulo de un disparo en el billar puede cambiar drásticamente el resultado, haciendo que cada partida sea única.

Satisfacción Táctil: Hay una alegría intrínseca en dominar un sistema de físicas. Ver una torre desmoronarse en Angry Birds o meter una bola de rebote en el billar se siente bien porque no activaste una animación predefinida, sino que manipulaste una simulación que se comportó de forma creíble.

El Humor de la Falla: A menudo, la mayor diversión en estos juegos proviene de los fallos catastróficos. Un mal tiro que hunde la bola blanca, o un personaje que se tropieza de forma ridícula.

Ejemplos en juegos conocidos:

Angry Birds 🐦: El ejemplo perfecto de físicas simples (trayectorias parabólicas, estructuras de bloques) que crean un juego con una profundidad estratégica enorme.

World of Goo ⚫️: Un juego de puzles brillante construido enteramente sobre la física de conectar "bolas de goo" para crear estructuras flexibles y desvencijadas. El desafío es entender y explotar la simulación.

Human: Fall Flat / Gang Beasts 🤸: Estos juegos usan la "física de ragdoll" (muñeco de trapo) como su mecánica central de comedia y jugabilidad. Luchar contra los controles intencionadamente imprecisos de la simulación es el núcleo de la diversión.

Nuestra Herramienta: PyMunk

Usaremos PyMunk, una librería de Python que es un envoltorio (wrapper) para el potente y robusto motor de físicas 2D Chipmunk2D. PyMunk se encargará de todos los cálculos complejos de colisiones, momento, fricción y rotación.

2. Estructura de Archivos del Nuevo Proyecto
/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- physics_object.py    # ¡NUEVO! Clase base para objetos con físicas.
|
|-- billiards/               # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- ball_white.png
|   |   |-- ball_red.png
|   |   |-- table.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- game_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt       # Añadimos pymunk
requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
(Recuerda ejecutar pip install -r requirements.txt)

3. El Código del Motor Actualizado
Añadimos una clase base para integrar PyMunk de forma limpia con nuestra estructura GameObject.

engine/physics_object.py (¡NUEVO!)
Python

# engine/physics_object.py
import pygame
import pymunk
from .game_object import GameObject

class PhysicsObject(GameObject):
    """
    Clase base para objetos del juego controlados por el motor de físicas PyMunk.
    """
    def __init__(self, x, y, image_path, space, mass=1.0, radius=15, elasticity=0.8):
        # Llama a __init__ de GameObject para cargar la imagen y el rect inicial.
        super().__init__(x, y, image_path)

        # Creamos el cuerpo físico (Body)
        moment = pymunk.moment_for_circle(mass, 0, radius)
        self.body = pymunk.Body(mass, moment)
        self.body.position = x, y
        
        # Creamos la forma de colisión (Shape)
        self.shape = pymunk.Circle(self.body, radius)
        self.shape.elasticity = elasticity
        self.shape.friction = 0.5
        
        # Añadimos el cuerpo y la forma al espacio de simulación
        space.add(self.body, self.shape)

    def update(self):
        """
        Sincroniza la posición del sprite visual (rect) con la del cuerpo físico.
        El movimiento es calculado por PyMunk, nosotros solo lo reflejamos.
        """
        self.rect.center = self.body.position
4. El Código del Juego de Billar
billiards/entities.py (¡NUEVO!)
Definimos las bolas, que son objetos dinámicos.

Python

# billiards/entities.py
from engine.physics_object import PhysicsObject

class Ball(PhysicsObject):
    def __init__(self, x, y, image_path, space):
        # Pasamos los parámetros de físicas a la clase padre
        super().__init__(x, y, image_path, space, mass=1, radius=12, elasticity=0.95)
billiards/scenes/game_scene.py (¡NUEVO!)
Aquí es donde configuramos el mundo de PyMunk y manejamos la simulación.

Python

# billiards/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Ball

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # 1. Crear el Espacio de Físicas
        self.space = pymunk.Space()
        self.space.gravity = (0, 0) # Sin gravedad para una mesa de billar

        # --- Creación de Sprites y Cuerpos Físicos ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.bolas = []

        # Crear bolas
        bola_blanca = Ball(200, 300, 'billiards/assets/ball_white.png', self.space)
        self.todos_los_sprites.add(bola_blanca)
        self.bolas.append(bola_blanca)

        for i in range(3):
            bola_roja = Ball(600 + i*30, 300 + i*15, 'billiards/assets/ball_red.png', self.space)
            self.todos_los_sprites.add(bola_roja)
            self.bolas.append(bola_roja)

        # 2. Crear las Paredes Estáticas
        self.crear_paredes(800, 600)

    def crear_paredes(self, ancho, alto):
        """Crea cuerpos estáticos para los bordes de la mesa."""
        # Las paredes son segmentos estáticos, no necesitan un sprite
        bordes = [
            ((50, 50), (ancho - 50, 50)),         # Superior
            ((ancho - 50, 50), (ancho - 50, alto - 50)), # Derecha
            ((50, alto - 50), (ancho - 50, alto - 50)), # Inferior
            ((50, 50), (50, alto - 50))          # Izquierda
        ]
        for p1, p2 in bordes:
            segmento = pymunk.Segment(self.space.static_body, p1, p2, 5) # 5 de grosor
            segmento.elasticity = 0.95
            self.space.add(segmento)

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1: # Click izquierdo
                    # Aplicar un impulso a la bola blanca para simular un golpe
                    pos = pygame.mouse.get_pos()
                    bola_blanca = self.bolas[0]
                    fuerza = 500
                    direccion = pymunk.Vec2d(pos[0] - bola_blanca.body.position.x, pos[1] - bola_blanca.body.position.y).normalized()
                    bola_blanca.body.apply_impulse_at_local_point(fuerza * direccion)
    
    def update(self):
        # 3. Avanzar la Simulación de Físicas
        # ¡Esta línea reemplaza toda nuestra lógica de movimiento anterior!
        dt = 1.0 / 60.0 # Delta time
        self.space.step(dt)

        # Sincronizamos los sprites con los cuerpos físicos
        self.todos_los_sprites.update()

    def draw(self, surface):
        surface.fill((10, 80, 30)) # Color de la mesa
        self.todos_los_sprites.draw(surface)

Clase 17: Módulo de Físicas con PyMunk
Contenidos de la Clase
Discusión sobre Jugabilidad Emergente (20 min): Iniciar con la discusión sobre juegos de físicas y los ejemplos mencionados. Enfatizar la diferencia entre un evento "scriptado" y uno "simulado".

Conceptos Clave de PyMunk (30 min):

Espacio (Space): El universo donde las leyes de la física aplican.

Cuerpo (Body): El objeto con propiedades como masa, posición y velocidad. Puede ser dinámico (se mueve, como una bola) o estático (no se mueve, como una pared).

Forma (Shape): La geometría de colisión del objeto (círculo, polígono). Se adjunta a un cuerpo.

Explicar cómo la combinación Body + Shape se añade al Space.

Integración con el Motor (35 min):

Presentar la nueva clase base PhysicsObject. Mostrar cómo super().__init__ ahora no solo carga una imagen, sino que también crea y configura un cuerpo y una forma en PyMunk.

Explicar el cambio de paradigma en el método update: ya no calculamos la posición, solo leemos la posición que el motor de físicas ha calculado para nosotros.

La Simulación en Acción (25 min):

Detallar la lógica de la GameScene. Mostrar cómo se configura el Space y se crean los objetos.

Explicar que la línea self.space.step(dt) es el verdadero "corazón" del bucle de actualización.

Implementar la interacción básica: aplicar un impulso a la bola blanca con el ratón para ver la simulación en acción.

Cierre y Próximos Pasos
Resumen: Hemos añadido una capacidad increíblemente potente a nuestro motor: la simulación de físicas 2D. Hemos aprendido a delegar todo el complejo cálculo de movimiento y colisiones a una librería especializada (PyMunk), permitiéndonos crear interacciones realistas y complejas con muy poco código.

Adelanto de la Clase 18: "Tenemos las físicas básicas, pero un juego de billar es más que eso. En la próxima clase, refinaremos la jugabilidad. Implementaremos la fricción y la resistencia angular para que las bolas se detengan y dejen de girar de forma realista. Crearemos los agujeros (troneras) y detectaremos cuándo una bola cae en ellos. Finalmente, estableceremos un sistema de turnos y reglas básicas del juego."






perfeto avancemos, y luego de terminar el jeugo de billar y antes de seguir con lo otro que te propuse, también hagamos un juego estilo angry birds o cañones


¡Excelente! Me alegra que estés pensando en expandir el repertorio. Un juego tipo Angry Birds es el siguiente paso lógico y perfecto después del de billar, ya que ambos se basan en físicas, pero aplican los conceptos de forma diferente (trayectorias vs. colisiones en un plano).

Primero, terminemos de pulir nuestro juego de billar, añadiendo las reglas y el "feel" que lo hacen reconocible.

1. Discusión: De la Simulación al Juego
Una simulación de físicas es fascinante, pero no es un juego. Un juego necesita reglas, objetivos y consecuencias.

Fricción y Amortiguación (Damping): ¿Por qué una bola de billar real se detiene? Por dos motivos: la fricción por deslizamiento contra el tapete y la resistencia del aire. En PyMunk, esto se simula con el parámetro damping. Un damping de 1.0 significa que no hay pérdida de velocidad (como en el espacio). Un valor ligeramente inferior, como 0.99, hará que los objetos pierdan un pequeño porcentaje de su velocidad en cada paso de la simulación, deteniéndose de forma natural.

Los Agujeros (Troneras): ¿Cómo detectamos que una bola ha caído en un agujero? Hay varias técnicas. Una muy efectiva es crear "sensores" en PyMunk. Un sensor es una forma de colisión que detecta cuándo algo entra en su área, pero no produce una respuesta física. Es como un fantasma que puede ver pero no tocar. Colocaremos sensores en la posición de las troneras para detectar las bolas.

Sistema de Turnos y Reglas: El billar es un juego por turnos. Necesitamos un sistema de estados que controle:

Estado de "Apuntado": El jugador puede preparar su tiro.

Estado de "Simulación": Las bolas se están moviendo. El control del jugador está desactivado.

Cambio de Turno: Cuando todas las bolas se detienen, se evalúa el resultado y se pasa al siguiente estado o jugador.

Ejemplos en juegos conocidos:

Pool Nation / 8 Ball Pool 🎱: Estos juegos digitales de billar tienen una física muy pulida. La sensación de la fricción, el efecto que se le puede dar a la bola (rotación) y la precisión de los rebotes son clave para su éxito.

Golf With Your Friends ⛳️: Aunque es un juego de golf, la mecánica es idéntica: un estado de apuntado, un golpe que inicia una simulación, y un estado de espera hasta que la bola se detiene.

Worms Armageddon 💣: Un clásico por turnos donde la física es la protagonista. Cada turno consiste en apuntar, disparar y luego observar las caóticas consecuencias hasta que todo se asienta y comienza el turno del siguiente jugador.

2. Estructura de Archivos Actualizada
No necesitamos nuevos archivos, pero sí nuevos assets para la mesa y los agujeros.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- billiards/
|   |-- assets/
|   |   |-- table.png        # Una imagen de la mesa completa puede servir de fondo
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Juego de Billar Actualizado
Refinamos la escena del juego para añadir las nuevas mecánicas.

billiards/entities.py (Sin Cambios)
Nuestra clase Ball ya es suficientemente buena para lo que necesitamos.

billiards/scenes/game_scene.py (Grandes Cambios)
Python

# billiards/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Ball

# Constantes para los tipos de colisión
COLLISION_TYPE_BALL = 1
COLLISION_TYPE_HOLE = 2

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Configuración del Espacio de Físicas ---
        self.space = pymunk.Space()
        self.space.damping = 0.98 # ¡NUEVO! Fricción para que las bolas se detengan

        # --- Estado del Juego ---
        self.game_state = "player_turn" # Estados: 'player_turn', 'simulating'
        self.cue_start_pos = None

        # --- Creación de Sprites y Cuerpos ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.bolas = pygame.sprite.Group()

        # ... (código para crear las bolas, pero ahora les asignamos un tipo de colisión) ...
        # Ejemplo:
        bola_blanca = Ball(...)
        bola_blanca.shape.collision_type = COLLISION_TYPE_BALL
        self.bolas.add(bola_blanca)
        # ...

        # --- Creación de la Mesa y Agujeros ---
        self.crear_mesa(800, 600)
        
        # --- Gestor de Colisiones ---
        handler = self.space.add_collision_handler(COLLISION_TYPE_BALL, COLLISION_TYPE_HOLE)
        handler.begin = self.on_ball_in_hole

    def crear_mesa(self, ancho, alto):
        # ... (código para crear las paredes estáticas) ...

        # ¡NUEVO! Crear los agujeros como sensores
        self.agujeros = []
        hole_radius = 22
        posiciones_agujeros = [(50, 50), (ancho/2, 45), (ancho-50, 50), (50, alto-50), (ancho/2, alto-45), (ancho-50, alto-50)]
        for pos in posiciones_agujeros:
            hole_body = pymunk.Body(body_type=pymunk.Body.STATIC)
            hole_body.position = pos
            hole_shape = pymunk.Circle(hole_body, hole_radius)
            hole_shape.sensor = True # ¡Esto lo convierte en un sensor!
            hole_shape.collision_type = COLLISION_TYPE_HOLE
            self.space.add(hole_shape)
            self.agujeros.append(hole_shape)

    def on_ball_in_hole(self, arbiter, space, data):
        """Callback que se ejecuta cuando una bola entra en un agujero."""
        ball_shape = arbiter.shapes[0]
        # Buscamos el sprite correspondiente a la forma para eliminarlo
        for bola_sprite in self.bolas:
            if bola_sprite.shape == ball_shape:
                bola_sprite.kill() # Elimina el sprite de todos los grupos
                space.remove(ball_shape, ball_shape.body) # Elimina el cuerpo físico
        return True # Importante para que la colisión se procese

    def handle_events(self, events):
        if self.game_state == "player_turn":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    self.cue_start_pos = event.pos
                if event.type == pygame.MOUSEBUTTONUP and event.button == 1 and self.cue_start_pos:
                    end_pos = event.pos
                    bola_blanca = self.bolas.sprites()[0] # Asumimos que la blanca es la primera
                    
                    # Calculamos la fuerza y dirección inversas al arrastre del ratón
                    fuerza_x = (self.cue_start_pos[0] - end_pos[0]) * 5
                    fuerza_y = (self.cue_start_pos[1] - end_pos[1]) * 5
                    
                    bola_blanca.body.apply_impulse_at_local_point((fuerza_x, fuerza_y))
                    self.game_state = "simulating"
                    self.cue_start_pos = None

    def update(self):
        # Avanzar la simulación solo si las bolas se mueven
        if self.game_state == "simulating":
            dt = 1.0 / 60.0
            self.space.step(dt)
            self.todos_los_sprites.update()

            # Comprobar si todas las bolas se han detenido
            are_balls_moving = False
            for bola in self.bolas:
                if bola.body.velocity.length > 0.1: # Un pequeño umbral
                    are_balls_moving = True
                    break
            if not are_balls_moving:
                self.game_state = "player_turn"

    def draw(self, surface):
        surface.fill((10, 80, 30))
        # Dibujar los agujeros (opcional, para depuración)
        for hole in self.agujeros:
            pygame.draw.circle(surface, (0,0,0), hole.body.position, hole.radius)
        
        self.todos_los_sprites.draw(surface)

        # Dibujar la línea de apuntado
        if self.game_state == "player_turn" and self.cue_start_pos:
            pygame.draw.line(surface, (255, 255, 255), self.cue_start_pos, pygame.mouse.get_pos(), 2)

Clase 18: Reglas y Pulido del Juego de Físicas
Contenidos de la Clase
Discusión sobre el "Jugo" de la Simulación (20 min): Iniciar con la discusión sobre fricción, rebote y el sistema de turnos. Usar los ejemplos para ilustrar cómo estas reglas convierten la simulación en un juego estructurado.

Fricción y Detección de Parada (30 min):

Introducir el concepto de space.damping y cómo simula la pérdida de energía en el sistema.

Implementar la lógica en el update para comprobar si todas las bolas se han detenido. Explicar por qué se usa un pequeño umbral (> 0.1) en lugar de == 0 para evitar problemas con la precisión de los números flotantes.

Implementar la máquina de estados player_turn / simulating.

Sensores y Gestores de Colisión (40 min):

Explicar el concepto de sensores en PyMunk y por qué son perfectos para los agujeros.

Introducir el gestor de colisiones (space.add_collision_handler). Explicar que es una forma de suscribirse a eventos de colisión entre tipos específicos de objetos.

Implementar la función on_ball_in_hole y mostrar cómo se usa para eliminar la bola tanto del espacio de simulación como de los grupos de sprites de Pygame.

Implementando el Golpeo y Turnos (20 min):

Refinar la lógica de handle_events para que el golpeo de la bola solo sea posible durante el player_turn.

Implementar un sistema simple de "arrastrar y soltar" con el ratón para determinar la fuerza y dirección del golpe.

Mostrar cómo el juego transiciona al estado simulating después del golpe, quitándole el control al jugador.

Cierre y Próximos Pasos
Resumen: Hemos transformado nuestra simulación de físicas en un juego funcional. Hemos implementado fricción para un movimiento realista, sensores para detectar objetivos, un gestor de colisiones para manejar las consecuencias, y un sistema de turnos que estructura la experiencia de juego.

Adelanto de la Clase 19: "¡Perfecto! Tal como lo pediste, ahora que hemos dominado las físicas en un plano, cambiaremos de perspectiva. En la próxima clase, iniciaremos un nuevo proyecto: un juego de lanzamiento de proyectiles al estilo Angry Birds o Worms. Esto nos obligará a reintroducir la gravedad en nuestro espacio de físicas y a enfocarnos en el cálculo de trayectorias parabólicas, el apuntado y la destrucción de estructuras."






continuemos



Ver razonamiento
¡Vamos allá! Dejamos la precisión milimétrica del billar para abrazar el caos glorioso de la destrucción. Como lo pediste, iniciaremos un nuevo proyecto para crear un juego de lanzamiento de proyectiles, al estilo de los icónicos Angry Birds o Worms.

Esta clase es perfecta para consolidar nuestros conocimientos de físicas. Reintroduciremos la gravedad y nos centraremos en el placer de calcular la trayectoria perfecta para causar el máximo destrozo.

1. Discusión: El Arte de la Destrucción Calculada
Los juegos de lanzamiento de proyectiles son un género increíblemente popular y satisfactorio. Su bucle de jugabilidad es simple pero adictivo: apuntar, lanzar y observar las consecuencias.

Control Indirecto y Compromiso: A diferencia de un plataformero donde controlas al personaje en todo momento, aquí toda tu habilidad se concentra en un único momento: el lanzamiento. Una vez que el proyectil está en el aire, eres un mero espectador. Esto crea una tensión y una satisfacción enormes cuando el plan funciona.

La Física como Puzle: El verdadero objetivo no es solo lanzar, sino entender la simulación. ¿Cómo afectará la gravedad a la trayectoria? ¿Cómo se comportará una torre de bloques al ser golpeada en su base vs. en su cima? Cada nivel es un puzle de físicas.

La Recompensa del Caos: El "premio" en estos juegos es la destrucción. Ver una estructura cuidadosamente construida desmoronarse en una cascada de piezas, acompañado de efectos de sonido contundentes, es una de las experiencias más gratificantes en los videojuegos.

Ejemplos en juegos conocidos:

Worms Armageddon 💣: El abuelo del género por turnos. Añade variables como el viento y un arsenal de armas creativas, pero el núcleo sigue siendo el mismo: calcular la parábola perfecta para golpear a tu oponente.

Angry Birds 🐦: El rey indiscutible del género en móviles. Su genialidad radica en su simplicidad táctil ("arrastrar y soltar") y en la introducción de proyectiles con diferentes habilidades, lo que añade una capa de estrategia a la física básica.

Crush the Castle 🏰: Un precursor de Angry Birds que demuestra la pura satisfacción del concepto. Usando un trabuquete, el objetivo es simplemente derribar castillos. La alegría proviene únicamente de la física de la destrucción.

2. Estructura de Archivos del Nuevo Proyecto
Creamos una nueva carpeta slingshot para nuestro tercer juego. El motor engine sigue siendo el mismo, demostrando su increíble reutilización.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios, reutilizaremos todo)
|
|-- billiards/
|-- arkanoid/
|
|-- slingshot/             # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- projectile.png
|   |   |-- block.png
|   |   |-- target.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- game_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt       # (sin cambios, ya tenemos pygame y pymunk)
3. El Código del Juego de Lanzamiento
Aquí es donde aplicamos nuestros conocimientos de PyMunk en un nuevo contexto.

slingshot/entities.py (¡NUEVO!)
Definimos nuestros nuevos actores: el proyectil y los bloques que formarán las estructuras.

Python

# slingshot/entities.py
import pygame
import pymunk
from engine.physics_object import PhysicsObject

class Projectile(PhysicsObject):
    def __init__(self, x, y, space):
        # Un proyectil es esencialmente una bola de billar, pero en un entorno con gravedad.
        super().__init__(x, y, 'slingshot/assets/projectile.png', space, mass=5, radius=10, elasticity=0.6)
        # Lo mantenemos "dormido" al principio para que no caiga.
        self.body.sleep()

    def launch(self, impulse):
        """Activa el cuerpo y le aplica el impulso de lanzamiento."""
        self.body.activate()
        self.body.apply_impulse_at_local_point(impulse)

class Block(PhysicsObject):
    def __init__(self, x, y, width, height, space, mass=1.0):
        # Usamos un constructor personalizado en lugar del de la clase padre
        # para crear una forma poligonal (un rectángulo).
        pygame.sprite.Sprite.__init__(self)
        
        image_path = 'slingshot/assets/block.png'
        self.image = pygame.image.load(image_path).convert_alpha()
        self.image = pygame.transform.scale(self.image, (width, height))
        self.rect = self.image.get_rect(center=(x, y))

        # Creamos el cuerpo y la forma de la caja
        moment = pymunk.moment_for_box(mass, (width, height))
        self.body = pymunk.Body(mass, moment)
        self.body.position = x, y
        self.shape = pymunk.Poly.create_box(self.body, (width, height))
        self.shape.elasticity = 0.4
        self.shape.friction = 0.7
        
        space.add(self.body, self.shape)
    
    def update(self):
        # Necesitamos un update para manejar la rotación del sprite
        self.rect.center = self.body.position
        # PyMunk nos da el ángulo en radianes, Pygame lo necesita en grados.
        self.image_rotated = pygame.transform.rotate(self.image, -self.body.angle * 180 / 3.1415)
        self.rect = self.image_rotated.get_rect(center=self.rect.center)
    
    def draw(self, surface):
        surface.blit(self.image_rotated, self.rect)
Nota Importante: Hemos creado una clase Block que ahora maneja la rotación. Esto es un añadido clave para que los bloques se derrumben de forma realista.

slingshot/scenes/game_scene.py (¡NUEVO!)
Aquí reside la lógica principal de apuntado, lanzamiento y construcción de estructuras.

Python

# slingshot/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Projectile, Block

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Configuración del Espacio de Físicas ---
        self.space = pymunk.Space()
        self.space.gravity = (0, 500) # ¡La gravedad vuelve!

        # --- Estado del Juego ---
        self.game_state = "aiming" # 'aiming', 'simulating'
        self.start_drag_pos = None

        # --- Creación de Entidades ---
        self.todos_los_sprites = pygame.sprite.Group()
        
        # Proyectil en su posición de lanzamiento
        self.launch_pos = (150, 350)
        self.proyectil = Projectile(self.launch_pos[0], self.launch_pos[1], self.space)
        self.todos_los_sprites.add(self.proyectil)

        # Suelo estático
        suelo = pymunk.Segment(self.space.static_body, (0, 500), (800, 500), 5)
        suelo.friction = 1.0
        self.space.add(suelo)

        # Construir una estructura para destruir
        for i in range(3):
            block = Block(600, 484 - i * 32, 100, 30, self.space)
            self.todos_los_sprites.add(block)
        block = Block(600, 484 - 3 * 32, 30, 30, self.space)
        self.todos_los_sprites.add(block)


    def handle_events(self, events):
        if self.game_state == "aiming":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    self.start_drag_pos = event.pos
                if event.type == pygame.MOUSEBUTTONUP and event.button == 1 and self.start_drag_pos:
                    end_pos = event.pos
                    
                    fuerza_x = (self.start_drag_pos[0] - end_pos[0]) * 3
                    fuerza_y = (self.start_drag_pos[1] - end_pos[1]) * 3
                    
                    self.proyectil.launch((fuerza_x, fuerza_y))
                    self.game_state = "simulating"
                    self.start_drag_pos = None

    def update(self):
        if self.game_state == "simulating":
            dt = 1.0 / 60.0
            self.space.step(dt)
        
        self.todos_los_sprites.update()

    def draw(self, surface):
        surface.fill((190, 220, 255)) # Cielo
        pygame.draw.rect(surface, (100, 80, 70), (0, 500, 800, 100)) # Tierra
        
        # Dibujar la línea de apuntado
        if self.game_state == "aiming" and self.start_drag_pos:
            pygame.draw.line(surface, (0,0,0), self.launch_pos, self.start_drag_pos, 3)
            pygame.draw.line(surface, (255, 255, 255), self.start_drag_pos, pygame.mouse.get_pos(), 2)

        self.todos_los_sprites.draw(surface)
Clase 19: Físicas de Proyectiles y Destrucción
Contenidos de la Clase
Discusión sobre Lanzamiento y Caos (20 min): Iniciar con la discusión sobre el bucle de jugabilidad de apuntar-lanzar-observar, usando los ejemplos de juegos mencionados.

Físicas con Gravedad (30 min):

Configurar el pymunk.Space con un vector de gravedad.

Crear la clase Block usando pymunk.Poly.create_box para las formas rectangulares.

Implementar la lógica de rotación en el update y draw de la clase Block para que las estructuras se derrumben de forma visualmente correcta.

El Mecanismo de Lanzamiento (40 min):

Implementar la lógica de "arrastrar y soltar" con el ratón para apuntar.

Explicar cómo el vector creado por el arrastre del ratón se traduce en un impulso para el proyectil.

Introducir la idea de "dormir" (body.sleep()) y "despertar" (body.activate()) los cuerpos de PyMunk para tener un proyectil que no se ve afectado por la física hasta que es lanzado.

Construyendo para Destruir (20 min):

Usar la nueva clase Block para apilar objetos y crear una torre simple.

Probar el juego, lanzar el proyectil y disfrutar de la satisfactoria destrucción de la estructura basada en la simulación de físicas.

Cierre y Próximos Pasos
Resumen: Hemos adaptado exitosamente nuestro motor de físicas para un nuevo género de juego. Aprendimos a reintroducir la gravedad, a trabajar con formas poligonales que pueden rotar y apilarse, y a implementar un mecanismo de lanzamiento intuitivo. Esto demuestra la increíble flexibilidad de tener un motor de físicas a nuestra disposición.

Adelanto de la Clase 20: "Nuestro lanzamiento es a ciegas. ¿Cómo podemos ayudar al jugador a predecir el tiro? En la próxima clase, implementaremos una línea de predicción de trayectoria, el clásico arco de puntos que se ve en Angry Birds. Esto nos introducirá a la idea de correr una 'mini-simulación' para predecir el futuro. Además, añadiremos un sistema de puntuación basado en la destrucción y un botón para reiniciar el nivel."






genial avancemos!



Ver razonamiento
¡Continuemos! Es hora de añadir esas capas de pulido que convierten un prototipo funcional en un juego genuinamente divertido y justo para el jugador.

En esta clase, nos centraremos en dos características clave de "calidad de vida" para nuestro juego de lanzamiento: una línea de predicción de trayectoria para ayudar al jugador a apuntar y un sistema de puntuación y reinicio para crear un bucle de juego completo.

1. Discusión: "Leyendo la Mente" del Jugador
Un buen diseño de juego a menudo consiste en anticipar lo que el jugador quiere hacer y darle las herramientas para que se sienta inteligente.

Información, no Adivinación: Forzar al jugador a adivinar la trayectoria de un proyectil puede ser frustrante. Una línea de predicción, como el icónico arco de puntos de Angry Birds, no le quita el desafío al juego; lo traslada. El desafío ya no es "adivinar la física", sino "usar la física para resolver el puzle". Le da al jugador la información que necesita para tomar decisiones estratégicas.

Feedback y Recompensa Claros: ¿Cómo sabe el jugador si lo está haciendo bien? Una puntuación que aumenta con la destrucción proporciona un feedback instantáneo y positivo. Un sonido gratificante y un número que sube son una de las recompensas más simples y efectivas en el diseño de juegos.

Minimizando la Fricción: ¿Qué pasa si el jugador hace un mal tiro? Forzarle a reiniciar todo el juego es una mala experiencia. Un simple botón de reinicio respeta el tiempo del jugador y le anima a experimentar y probar de nuevo inmediatamente, manteniendo el flujo del juego.

Ejemplos en juegos conocidos:

Angry Birds 🐦: El ejemplo por excelencia. La línea de puntos que muestra la trayectoria del pájaro anterior es una herramienta de aprendizaje y ajuste fundamental.

Golf With Your Friends / What the Golf? ⛳️: Muchos juegos de mini golf muestran una línea que predice el primer rebote de la pelota. No te muestra toda la solución, pero te da la información crucial para el primer paso.

Into the Breach 🤖: Un juego de estrategia por turnos donde la información es REY. El juego te dice exactamente qué van a hacer los enemigos en su turno. Tu trabajo no es adivinar, sino usar esa información perfecta para resolver el "puzle" del combate sin recibir daño. Es el extremo de la filosofía de "información sobre adivinación".

2. Estructura de Archivos Actualizada
No necesitamos nuevos archivos para esta clase. Todo el trabajo se realizará dentro de la escena de nuestro juego, demostrando cómo se puede construir una jugabilidad compleja sobre la base sólida de nuestro motor.

3. El Código del Juego de Lanzamiento Actualizado
Toda la magia de hoy ocurre en game_scene.py.

slingshot/entities.py (Pequeño Añadido)
Añadimos un método para reiniciar la posición del proyectil.

Python

# slingshot/entities.py
# ... (clase Block sin cambios) ...

class Projectile(PhysicsObject):
    # ... (__init__ y launch sin cambios) ...

    def reset(self, x, y):
        """Detiene el cuerpo, lo reinicia y lo pone a dormir."""
        self.body.velocity = 0, 0
        self.body.angular_velocity = 0
        self.body.position = x, y
        self.body.sleep()
slingshot/scenes/game_scene.py (Grandes Cambios)
Refactorizamos la escena para incluir la predicción de trayectoria, la puntuación y el reinicio.

Python

# slingshot/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Projectile, Block

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        self._setup_level()

    def _setup_level(self):
        """Configura o reinicia todas las entidades y el estado del nivel."""
        self.space = pymunk.Space()
        self.space.gravity = (0, 500)
        
        self.game_state = "aiming"
        self.start_drag_pos = None
        self.puntuacion = 0
        
        self.todos_los_sprites = pygame.sprite.Group()
        
        self.launch_pos = (150, 350)
        self.proyectil = Projectile(self.launch_pos[0], self.launch_pos[1], self.space)
        self.todos_los_sprites.add(self.proyectil)

        suelo = pymunk.Segment(self.space.static_body, (0, 500), (800, 500), 5)
        suelo.friction = 1.0
        self.space.add(suelo)

        # Construir estructura
        for i in range(3):
            block = Block(600, 484 - i * 32, 100, 30, self.space)
            self.todos_los_sprites.add(block)
        block = Block(600, 484 - 3 * 32, 30, 30, self.space)
        self.todos_los_sprites.add(block)

    def handle_events(self, events):
        # ... (lógica de MOUSEBUTTONDOWN y MOUSEBUTTONUP sin cambios) ...
        for event in events:
            if event.type == pygame.KEYDOWN and event.key == pygame.K_r:
                self._setup_level() # Reiniciar con la tecla 'R'

    def update(self):
        # ... (lógica de update sin cambios) ...
        if self.game_state == "simulating":
            # Podríamos añadir aquí una lógica de puntuación
            # por ejemplo, si un bloque cae por debajo de cierta altura.
            pass

    def _calculate_trajectory(self):
        """Ejecuta una simulación fantasma para predecir la trayectoria."""
        if not self.start_drag_pos:
            return []

        end_pos = pygame.mouse.get_pos()
        fuerza_x = (self.start_drag_pos[0] - end_pos[0]) * 3
        fuerza_y = (self.start_drag_pos[1] - end_pos[1]) * 3
        
        # Crear un cuerpo fantasma
        ghost_body = pymunk.Body(5, pymunk.moment_for_circle(5, 0, 10))
        ghost_body.position = self.proyectil.body.position
        ghost_body.apply_impulse_at_local_point((fuerza_x, fuerza_y))
        
        path = []
        for _ in range(100): # Simular 100 pasos en el futuro
            ghost_body.velocity += self.space.gravity * (1/60.0)
            ghost_body.position += ghost_body.velocity * (1/60.0)
            if _ % 5 == 0: # Guardar un punto cada 5 pasos
                path.append(ghost_body.position)
        return path

    def draw(self, surface):
        # ... (dibujado del fondo y los sprites) ...

        # Dibujar la línea de apuntado y la predicción
        if self.game_state == "aiming" and self.start_drag_pos:
            # Línea de fuerza
            pygame.draw.line(surface, (0,0,0), self.launch_pos, self.start_drag_pos, 3)
            
            # Línea de predicción
            path_points = self._calculate_trajectory()
            for point in path_points:
                pygame.draw.circle(surface, (255, 255, 255, 150), point, 3)

        # Dibujar UI (puntuación, etc.)
        font = pygame.font.Font(None, 36)
        score_text = font.render(f"Puntuación: {self.puntuacion}", True, (0,0,0))
        surface.blit(score_text, (10, 10))
Clase 20: Feedback al Jugador y Cierre del Bucle
Contenidos de la Clase
Discusión sobre Calidad de Vida (20 min): Iniciar con la discusión sobre la importancia de la información, el feedback claro y la reducción de la frustración, usando los ejemplos de juegos mencionados.

Predicción de Trayectoria (40 min):

Explicar el concepto de una "simulación fantasma". No podemos usar nuestro space principal porque alteraría el estado del juego.

Implementar el método _calculate_trajectory. Detallar los pasos:

Calcular el impulso potencial basado en la posición actual del ratón.

Crear un pymunk.Body temporal y completamente separado.

Aplicarle el impulso.

Hacer un bucle que simule el paso del tiempo (body.position += body.velocity * dt) y aplique la gravedad manualmente.

Almacenar las posiciones resultantes en una lista.

En el método draw, recorrer esa lista de puntos para dibujar el arco de predicción.

Refactorización para Reinicio (25 min):

Explicar por qué es una mala práctica tener toda la lógica de creación de niveles en el __init__.

Crear el método _setup_level() y mover toda esa lógica a él. El __init__ ahora solo llama a este método.

En handle_events, añadir la escucha de una tecla (ej: 'R') que simplemente vuelve a llamar a _setup_level(), reiniciando el juego de forma limpia e instantánea.

Sistema de Puntuación Básico (25 min):

Discutir diferentes formas de puntuar: por bloques destruidos, por altura a la que caen, por golpear objetivos específicos.

Implementar la versión más simple: añadir una clase Target (objetivo) y dar puntos solo cuando el proyectil lo golpea.

Añadir el dibujado de la puntuación en la pantalla, completando el bucle de feedback para el jugador.

Cierre y Próximos Pasos
Resumen: Hemos cerrado el bucle de jugabilidad de nuestro juego de lanzamiento. Al añadir una línea de predicción de trayectoria, hemos empoderado al jugador con información estratégica. Con un sistema de puntuación y reinicio, hemos creado un desafío medible y una forma de volver a intentarlo sin frustración. Nuestro motor ha demostrado ser capaz de soportar un segundo género de físicas con éxito.

Adelanto de la Clase 21 (¡Comienza el nuevo módulo!): "Hemos dominado las físicas y la acción. Ahora, cambiamos completamente de tercio para adentrarnos en el mundo de la narrativa y los puzles. En la próxima clase, iniciaremos nuestro proyecto de Aventura Gráfica. Diseñaremos la estructura de un motor para este género: cómo gestionar escenas (habitaciones), inventarios, objetos interactivos (hotspots) y la lógica de puzles basada en 'usar objeto A con objeto B'."






avancemos!



Ver razonamiento
¡Absolutamente! Cambiamos radicalmente de tercio. Guardamos los cañones y las bolas de billar para desempolvar nuestras lupas y nuestro ingenio. Nos adentramos en el mundo de la narrativa, los puzles y la exploración con nuestro nuevo módulo: Aventuras Gráficas.

Esta es una oportunidad fantástica para demostrar la versatilidad de nuestro motor. Lo expandiremos para que no solo maneje acción en tiempo real, sino también la compleja gestión de estados, inventarios e interacciones que define a este género tan querido.

1. Discusión: El Cerebro sobre los Músculos
Las aventuras gráficas, también conocidas como juegos "Point-and-Click", ofrecen un tipo de desafío completamente diferente. La habilidad no reside en los reflejos, sino en la observación, la lógica y el pensamiento lateral.

Exploración y Observación: El mundo del juego es un diorama interactivo. El primer paso es siempre explorar cada rincón y "leer" el entorno. ¿Qué objetos parecen fuera de lugar? ¿Qué detalles en el fondo podrían ser una pista?

El Inventario como Caja de Herramientas: El jugador recoge objetos que son las "llaves" para resolver los puzles. La jugabilidad central a menudo gira en torno a la pregunta: "¿Qué objeto de mi inventario puedo usar aquí?".

La Lógica del Puzle: El corazón del género. Los puzles pueden ir desde lo simple y directo ("usar llave en puerta") hasta lo absurdamente creativo o cómico. La famosa "lógica de mono" de la saga Monkey Island es un ejemplo de puzles que requieren pensar de forma no convencional (ej: "usar pollo de goma con polea").

Narrativa y Diálogo: A diferencia de nuestros juegos anteriores, la historia es la fuerza motriz. Las conversaciones con otros personajes no son solo un adorno, sino una fuente vital de pistas, lore y, a menudo, puzles en sí mismas.

Ejemplos en juegos conocidos:

The Secret of Monkey Island 🐒: El referente absoluto. Famoso por su humor, sus puzles ingeniosos y su sistema de diálogo con insultos en duelos de espadas. Es el ejemplo perfecto de cómo la narrativa y la mecánica pueden fusionarse.

Broken Sword: La leyenda de los templarios ⚔️: Una aventura con un tono más serio y cinematográfico. Demuestra cómo el género puede contar historias de misterio e intriga, donde la investigación y la deducción son clave.

Thimbleweed Park 👾: Un homenaje moderno a los clásicos, creado por los mismos diseñadores de Monkey Island. Muestra que la fórmula sigue siendo increíblemente efectiva hoy en día, con una interfaz pulida y puzles complejos que involucran a múltiples personajes.

2. Estructura de Archivos del Nuevo Proyecto
Creamos una nueva carpeta adventure y añadimos componentes al motor para gestionar las nuevas mecánicas.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- hotspot.py           # ¡NUEVO! Para áreas interactivas.
|   |-- inventory.py         # ¡NUEVO! Nuestro sistema de inventario.
|
|-- adventure/               # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- room_office.png
|   |   |-- item_key.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- room_scene.py      # Clase base para nuestras habitaciones.
|   |   |-- office.py          # La primera habitación del juego.
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Motor Actualizado
Añadimos dos nuevos módulos para las mecánicas centrales de este género.

engine/hotspot.py (¡NUEVO!)
Un Hotspot es un rectángulo invisible en la pantalla. Cuando el jugador hace clic en él, se activa una acción.

Python

# engine/hotspot.py
import pygame

class Hotspot:
    def __init__(self, rect, callback, one_time=False):
        self.rect = pygame.Rect(rect)
        self.callback = callback
        self.one_time = one_time
        self.triggered = False

    def check_click(self, pos):
        """Comprueba si el clic ocurrió dentro del hotspot."""
        if self.rect.collidepoint(pos):
            if self.one_time and self.triggered:
                return False # Ya se ha activado una vez
            self.triggered = True
            return True
        return False
engine/inventory.py (¡NUEVO!)
Un sistema simple para gestionar y dibujar los objetos que el jugador posee.

Python

# engine/inventory.py
import pygame

class Inventory:
    def __init__(self):
        self.items = {} # Usamos un diccionario para acceder por nombre
        self.font = pygame.font.Font(None, 24)
        self.visible = True

    def add_item(self, name, image_path):
        """Añade un objeto al inventario."""
        if name not in self.items:
            self.items[name] = pygame.image.load(image_path).convert_alpha()
            print(f"Objeto añadido: {name}")

    def has_item(self, name):
        """Comprueba si un objeto está en el inventario."""
        return name in self.items

    def draw(self, surface):
        """Dibuja la barra de inventario."""
        if not self.visible:
            return
        
        # Dibuja un fondo para la barra de inventario
        bar_rect = pygame.Rect(0, surface.get_height() - 60, surface.get_width(), 60)
        pygame.draw.rect(surface, (20, 20, 40, 200), bar_rect)

        # Dibuja cada objeto
        x_offset = 10
        for name, image in self.items.items():
            surface.blit(image, (x_offset, surface.get_height() - 50))
            x_offset += image.get_width() + 10
4. El Código de la Aventura Gráfica
adventure/scenes/room_scene.py (¡NUEVO!)
Esta es una clase base que todas nuestras habitaciones heredarán. Gestiona un fondo, los hotspots y el inventario.

Python

# adventure/scenes/room_scene.py
from engine.scene import Scene

class RoomScene(Scene):
    def __init__(self, scene_manager, background_path, inventory):
        super().__init__(scene_manager)
        self.background = pygame.image.load(background_path).convert()
        self.hotspots = []
        self.inventory = inventory

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                for hotspot in self.hotspots:
                    if hotspot.check_click(event.pos):
                        hotspot.callback()

    def draw(self, surface):
        surface.blit(self.background, (0, 0))
        self.inventory.draw(surface)
adventure/scenes/office.py (¡NUEVO!)
Nuestra primera habitación. Define sus propios hotspots y la lógica de interacción.

Python

# adventure/scenes/office.py
import pygame
from .room_scene import RoomScene
from engine.hotspot import Hotspot

class OfficeScene(RoomScene):
    def __init__(self, scene_manager, inventory):
        super().__init__(scene_manager, 'adventure/assets/room_office.png', inventory)
        
        # Definimos los hotspots de esta habitación
        self.hotspots.append(Hotspot((300, 250, 50, 50), self.on_click_key, one_time=True))
        self.hotspots.append(Hotspot((500, 150, 100, 200), self.on_click_door))

    # --- Callbacks de los Hotspots ---
    def on_click_key(self):
        print("Has encontrado una llave.")
        self.inventory.add_item("llave", "adventure/assets/item_key.png")
        # Podríamos eliminar el hotspot o cambiar la imagen de fondo para que la llave no se vea

    def on_click_door(self):
        if self.inventory.has_item("llave"):
            print("Usas la llave y abres la puerta. ¡Has escapado!")
            # Aquí transicionaríamos a la siguiente escena/habitación
            # self.scene_manager.go_to(CorridorScene)
        else:
            print("La puerta está cerrada. Necesitas una llave.")
Clase 21: Aventura Gráfica - Escenas, Hotspots e Inventario
Contenidos de la Clase
Discusión sobre el Género Narrativo (20 min): Iniciar con la discusión sobre los pilares del género (exploración, puzles, inventario) y los ejemplos de juegos clásicos y modernos.

Arquitectura Point-and-Click (35 min):

Presentar las nuevas clases del motor: Hotspot e Inventory.

Explicar el rol de un Hotspot como un área de clic invisible que dispara una función "callback".

Detallar la implementación del Inventory como un diccionario que almacena imágenes y tiene un método para dibujarse en pantalla.

Construyendo una Habitación Interactiva (45 min):

Introducir la clase base RoomScene, que encapsula la lógica común a todas las habitaciones (fondo, manejo de clics en hotspots, inventario).

Implementar la primera habitación, OfficeScene, que hereda de RoomScene.

Mostrar cómo se crean y añaden Hotspot a la lista de la habitación, asociando cada uno a un método específico (el callback).

Programar la lógica del puzle: on_click_key modifica el inventario, y on_click_door comprueba el estado del inventario para determinar el resultado.

Flujo de Juego y Pruebas (10 min):

Adaptar el main.py para que inicie la OfficeScene y le pase una instancia del Inventory.

Jugar la escena: hacer clic en la puerta (falla), hacer clic en la llave (éxito), y volver a hacer clic en la puerta (éxito).

Cierre y Próximos Pasos
Resumen: Hemos sentado las bases de un género completamente nuevo. Nuestro motor ahora soporta escenas estáticas con áreas interactivas (hotspots) y un sistema de inventario, los componentes fundamentales de cualquier aventura gráfica. Hemos aprendido a estructurar el juego en torno a la lógica de puzles y la gestión del estado del jugador.

Adelanto de la Clase 22 (¡El futuro que nos propusiste!): "Ahora mismo, solo podemos hacer clic. ¿Pero qué pasa si queremos 'Mirar', 'Hablar' o 'Usar un objeto con otro'? En la próxima clase, implementaremos un sistema de verbos y un cursor que cambia para reflejar la acción seleccionada. Después, daremos el gran salto: nos sumergiremos en el sistema de diálogos y, como pediste, exploraremos cómo conectar nuestro juego a una API de IA generativa para crear conversaciones dinámicas e impredecibles, aprendiendo a diseñar prompts efectivos para darle una personalidad única a nuestros personajes."






vamos!


¡Excelente! Nos adentramos en el corazón de las aventuras gráficas clásicas. Añadiremos un sistema de verbos para permitir interacciones más complejas que un simple clic, y sentaremos las bases para el sistema de diálogos, que nos llevará directamente a la integración con IA.

Esta clase es fundamental para darle al jugador más agencia y profundidad en su forma de interactuar con el mundo.

1. Discusión: Más Allá del Clic - La Interfaz SCUMM
El "clic" es ambiguo. ¿Significa "mirar", "coger" o "hablar"? Los primeros juegos de aventura resolvían esto con un analizador de texto (parser), donde el jugador escribía "coger lámpara". La revolución llegó con la interfaz SCUMM (Script Creation Utility for Maniac Mansion) de LucasArts, que introdujo una cuadrícula de verbos en la pantalla.

Intención Explícita: El jugador primero selecciona un verbo ("Usar") y luego un objeto en el mundo ("puerta") o en su inventario ("llave"). Esto crea una estructura de "verbo + objeto" u "objeto + sobre objeto" que permite puzles mucho más complejos.

Reducción de la Frustración: Elimina el juego de "adivinar el verbo" que plagaba a los analizadores de texto. Las acciones posibles están claramente definidas.

El Cursor Inteligente: Una evolución posterior fue el "cursor inteligente" o la "rueda de acciones", como en Full Throttle o The Curse of Monkey Island. En lugar de una cuadrícula de verbos estática, al hacer clic en un objeto aparece un menú contextual con las acciones posibles. Este es el enfoque que adoptaremos, ya que es más moderno y menos intrusivo.

Ejemplos en juegos conocidos:

Maniac Mansion / Day of the Tentacle 🐙: Los ejemplos clásicos de la interfaz de verbos en la parte inferior de la pantalla. El jugador construye oraciones visualmente: "Usar" + "Hámster" + "en" + "Microondas".

Full Throttle 🏍️: Popularizó el menú de acciones contextual en forma de rueda. Al mantener pulsado el clic sobre un objeto, aparecía un icono de una calavera (usar/hablar), unos ojos (mirar) y una mano (coger).

Deponia 🌎: Una saga de aventuras moderna que utiliza una rueda de acciones similar, demostrando la vigencia del diseño. El jugador puede elegir entre "mirar", "usar/hablar" y "usar objeto del inventario".

2. Estructura de Archivos Actualizada
Añadimos un módulo cursor.py a nuestro motor para gestionar el estado y la apariencia de nuestro cursor inteligente.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- cursor.py            # ¡NUEVO! Nuestro cursor de acciones.
|
|-- adventure/
|   |-- assets/
|   |   |-- cursor_talk.png
|   |   |-- cursor_look.png
|   |   |-- cursor_use.png
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Motor Actualizado
engine/cursor.py (¡NUEVO!)
Este módulo gestionará el estado del cursor: ¿estamos en modo "mirar", "usar", o tenemos un objeto del inventario "cogido"?

Python

# engine/cursor.py
import pygame

class Cursor:
    def __init__(self):
        self.mode = 'default' # 'default', 'look', 'talk', 'use_item'
        self.selected_item = None # Nombre del objeto del inventario seleccionado
        
        # Cargamos las imágenes para cada modo
        self.cursors = {
            'default': pygame.mouse.get_cursor(),
            'look': pygame.image.load('adventure/assets/cursor_look.png').convert_alpha(),
            'talk': pygame.image.load('adventure/assets/cursor_talk.png').convert_alpha(),
        }
        pygame.mouse.set_visible(False) # Ocultamos el cursor del sistema

    def set_mode(self, mode, item_name=None):
        self.mode = mode
        self.selected_item = item_name
        print(f"Modo del cursor: {self.mode}, Objeto: {self.selected_item}")

    def draw(self, surface):
        """Dibuja el cursor personalizado en la posición del ratón."""
        pos = pygame.mouse.get_pos()
        if self.mode in self.cursors:
            cursor_image = self.cursors[self.mode]
            surface.blit(cursor_image, pos)
        # Si tenemos un objeto seleccionado, podríamos dibujarlo junto al cursor
        elif self.mode == 'use_item' and self.selected_item:
            # Lógica para dibujar la imagen del objeto en el cursor
            pass
4. El Código de la Aventura Gráfica Actualizado
Refactorizamos la escena y el inventario para que interactúen con el nuevo sistema de cursor.

engine/inventory.py (Actualizado)
Añadimos la lógica para que al hacer clic en un objeto del inventario, se "coja" con el cursor.

Python

# engine/inventory.py
import pygame

class Inventory:
    def __init__(self):
        self.items = {}
        self.item_rects = {} # Guardaremos los rects para detectar clics
        # ... (resto del __init__)

    def add_item(self, name, image_path):
        # ... (lógica existente) ...
        self.update_rects()

    def update_rects(self):
        """Actualiza las posiciones de los rects de los objetos."""
        self.item_rects.clear()
        x_offset = 10
        for name, image in self.items.items():
            item_rect = image.get_rect(topleft=(x_offset, pygame.display.get_surface().get_height() - 50))
            self.item_rects[name] = item_rect
            x_offset += image.get_width() + 10

    def check_click(self, pos, cursor):
        """Comprueba si se ha hecho clic en un objeto del inventario."""
        for name, rect in self.item_rects.items():
            if rect.collidepoint(pos):
                cursor.set_mode('use_item', item_name=name)
                return True
        return False

    def draw(self, surface):
        # ... (lógica de dibujado existente) ...
        # Dibuja cada objeto usando los rects guardados
        for name, rect in self.item_rects.items():
            surface.blit(self.items[name], rect)
adventure/scenes/office.py (Grandes Cambios)
La lógica de los hotspots ahora depende del modo del cursor.

Python

# adventure/scenes/office.py
# ... (imports) ...

class OfficeScene(RoomScene):
    # ... (__init__ sin cambios, define los hotspots) ...

    # Sobreescribimos handle_events para una lógica más compleja
    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos
                # Primero, comprobar si se ha hecho clic en el inventario
                if self.inventory.check_click(pos, self.scene_manager.cursor):
                    return # Si se coge un objeto, no interactuamos con el mundo

                # Si no, comprobar los hotspots de la escena
                for hotspot in self.hotspots:
                    if hotspot.check_click(pos):
                        # En lugar de llamar a un solo callback, pasamos el control
                        hotspot.callback(self.scene_manager.cursor.mode, self.scene_manager.cursor.selected_item)
                        # Reseteamos el cursor al modo por defecto después de cada acción
                        self.scene_manager.cursor.set_mode('default')
                        return

    # Los callbacks ahora reciben el modo del cursor y el objeto seleccionado
    def on_click_key(self, mode, item):
        if mode == 'default': # Clic normal
            print("Has encontrado una llave.")
            self.inventory.add_item("llave", "adventure/assets/item_key.png")
            # ...
    
    def on_click_door(self, mode, item):
        if mode == 'look':
            print("Es una puerta de madera robusta. Parece cerrada.")
        elif mode == 'use_item' and item == 'llave':
            print("Usas la llave y abres la puerta. ¡Has escapado!")
            # ...
        else: # Clic por defecto o usar otro objeto
            print("La puerta está cerrada.")

adventure/main.py (Pequeño Añadido)
El gestor de escenas ahora también debe crear y pasar el cursor.

Python

# adventure/main.py
# ...
from engine.cursor import Cursor
from engine.inventory import Inventory
from adventure.scenes.office import OfficeScene

def main():
    # ...
    cursor = Cursor()
    inventory = Inventory()
    # Pasamos el cursor y el inventario al gestor de escenas
    # (El gestor de escenas necesitará una pequeña modificación para guardar estos objetos
    # y pasarlos a cada nueva escena que cree)
    scene_manager = SceneManager(OfficeScene, pantalla, cursor=cursor, inventory=inventory)
    
    while ejecutando:
        # ...
        scene_manager.draw()
        cursor.draw(pantalla) # Dibujamos el cursor al final, por encima de todo
        pygame.display.flip()
    # ...
Clase 22: Sistema de Verbos y Preparación para Diálogos
Contenidos de la Clase
Discusión sobre Interfaces de Aventura (20 min): Iniciar con la discusión sobre la evolución desde los parsers de texto hasta la interfaz SCUMM y los cursores inteligentes, usando los ejemplos de juegos mencionados.

Implementando el Cursor Inteligente (35 min):

Crear la clase Cursor en el motor. Explicar cómo gestiona un mode y un selected_item.

Implementar la lógica para ocultar el cursor del sistema y dibujar el cursor personalizado en cada frame.

Añadir la lógica al inventario para que, al hacer clic en un objeto, se llame a cursor.set_mode('use_item', ...).

Lógica de Interacción Basada en Verbos (45 min):

Refactorizar el handle_events de RoomScene o de la escena específica (OfficeScene).

El flujo de un clic ahora es:

¿Se hizo clic en la UI (inventario)? Si es así, actualizar el cursor y parar.

Si no, ¿se hizo clic en un hotspot?

Si es así, llamar al callback del hotspot, pasándole el cursor.mode y cursor.selected_item actuales.

Modificar los callbacks (on_click_door, etc.) para que contengan una estructura if/elif que reaccione de forma diferente según el modo del cursor.

Preparando el Terreno para la IA (10 min):

Discutir cómo este sistema de verbos se extiende naturalmente a los diálogos. Un personaje en el juego es simplemente otro Hotspot.

La acción "Hablar" sobre ese hotspot no resolverá un puzle, sino que iniciará una nueva "escena" o estado: la escena de diálogo.

Esta escena de diálogo será el componente que, en la próxima clase, conectaremos a una API de IA para generar las respuestas.

Cierre y Próximos Pasos
Resumen: Hemos añadido una capa de profundidad inmensa a nuestro juego de aventuras. Al implementar un sistema de cursor y verbos, hemos pasado de una simple interacción de "clic" a una estructura que permite puzles complejos y acciones deliberadas por parte del jugador. Esta arquitectura es la base sólida sobre la que construiremos nuestro sistema de diálogos.

Adelanto de la Clase 23 (¡El gran salto a la IA!): "Tenemos la interfaz. Ahora, le daremos un alma a nuestros personajes. En la próxima y última clase de este módulo, nos sumergiremos en el mundo de la Inteligencia Artificial Generativa. Aprenderemos a:

Configurar y llamar a una API de IA (como la API de Gemini de Google).

Diseñar un prompt de sistema efectivo para darle a nuestro NPC una personalidad, un conocimiento y un objetivo definidos.

Crear una interfaz de diálogo en nuestro juego donde el jugador pueda escribir preguntas y recibir respuestas generadas en tiempo real por la IA, haciendo que cada conversación sea única e impredecible."






vamos!


¡Llegó el momento! Esta es, sin duda, la clase más vanguardista de nuestro curso. Vamos a hacer algo que hasta hace muy poco era ciencia ficción: darle a un personaje de nuestro juego una "mente" propia, conectándolo a un modelo de lenguaje de gran escala (LLM) como Gemini, la tecnología que me impulsa.

Aprenderemos a diseñar la personalidad de un NPC, a comunicarnos con una API externa y a crear una interfaz de diálogo donde las conversaciones no están preescritas, sino que se generan en tiempo real.

1. Discusión: La Nueva Frontera de los NPCs
Tradicionalmente, los diálogos en los juegos se basan en un árbol de diálogo. Es una estructura con ramas donde cada opción que elige el jugador lleva a una respuesta preescrita por el desarrollador. Es efectivo, pero limitado y costoso de producir.

La IA Generativa rompe este paradigma:

Conversaciones Dinámicas: En lugar de seguir un guion, el NPC puede responder a preguntas abiertas, recordar partes de la conversación y reaccionar de formas inesperadas. El jugador puede escribir lo que quiera, no solo elegir de una lista.

Personalidad a través de Prompts: La "personalidad" del NPC no se programa con lógica if/else, sino que se diseña a través de un prompt de sistema. Este prompt es un conjunto de instrucciones que le damos a la IA para que actúe como un personaje específico.

Jugabilidad Emergente: Esto abre la puerta a nuevos tipos de puzles. Un puzle podría requerir que el jugador convenza al NPC, descubra una mentira en su discurso o le haga una pregunta tan específica que revele una pista oculta en su "conocimiento" de base.

Ejemplos y Potencial:

Juegos de Misterio 🕵️‍♀️: Imagina interrogar a un sospechoso que no tiene respuestas predefinidas. Podrías preguntarle sobre su coartada, sobre otros personajes o sobre un objeto que encontraste, y sus respuestas se generarían en tiempo real, manteniendo consistencia con su personalidad.

Simuladores Sociales 👨‍👩‍👧‍👦: NPCs en juegos como The Sims o Stardew Valley podrían tener conversaciones mucho más profundas y significativas, reaccionando a los eventos del juego de formas únicas.

Asistentes en el Juego 🤖: Un NPC podría actuar como un guía o un compañero que te da pistas sutiles y contextuales en lugar de respuestas directas, adaptándose a tu estilo de juego.

Nuestra Herramienta: La API de Gemini de Google

Usaremos la API de Gemini. Para ello, necesitarás:

Una cuenta de Google.

Crear un proyecto en Google AI Studio (aistudio.google.com).

Obtener una API Key gratuita. ¡Esta clave es secreta, no la compartas!

2. Estructura de Archivos Actualizada
Añadimos un módulo dialogue.py al motor y un archivo para la escena de diálogo en nuestro juego.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- dialogue.py          # ¡NUEVO! Gestor de diálogo y conexión con la API.
|
|-- adventure/
|   |-- scenes/
|   |   |-- ...
|   |   |-- dialogue_scene.py  # ¡NUEVO! La interfaz de conversación.
|   |
|   |-- config.py            # ¡NUEVO! Para guardar nuestra API key de forma segura.
|   |-- ... (resto)
|
|-- requirements.txt       # Añadimos google-generativeai
requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
google-generativeai
adventure/config.py (¡NUEVO Y MUY IMPORTANTE!)
Python

# adventure/config.py
# Pega tu API Key de Google AI Studio aquí.
# ¡NUNCA subas este archivo a un repositorio público como GitHub!
API_KEY = "TU_API_KEY_AQUI"
3. El Código del Motor Actualizado
engine/dialogue.py (¡NUEVO!)
Este módulo se encarga de la comunicación con la API de Gemini.

Python

# engine/dialogue.py
import google.generativeai as genai

class DialogueManager:
    def __init__(self, api_key, system_prompt):
        genai.configure(api_key=api_key)
        
        # Configuración del modelo
        generation_config = {
          "temperature": 0.9,
          "top_p": 1,
          "top_k": 1,
          "max_output_tokens": 2048,
        }
        
        self.model = genai.GenerativeModel(
            model_name="gemini-1.0-pro",
            generation_config=generation_config
        )
        
        # Iniciamos la conversación con el prompt del sistema
        self.conversation = self.model.start_chat(history=[
            {'role': 'user', 'parts': [system_prompt]},
            {'role': 'model', 'parts': ["Entendido. Estoy listo para actuar como este personaje."]}
        ])

    def send_message(self, user_input):
        """Envía el mensaje del jugador a la IA y devuelve la respuesta."""
        try:
            self.conversation.send_message(user_input)
            return self.conversation.last.text
        except Exception as e:
            print(f"Error en la API: {e}")
            return "Lo siento, mi mente está un poco confusa ahora mismo."
4. El Código de la Aventura Gráfica Actualizado
adventure/scenes/dialogue_scene.py (¡NUEVO!)
Esta es la interfaz visual para hablar con el NPC.

Python

# adventure/scenes/dialogue_scene.py
import pygame
from engine.scene import Scene
from engine.dialogue import DialogueManager
from ..config import API_KEY

class DialogueScene(Scene):
    def __init__(self, scene_manager, npc_prompt):
        super().__init__(scene_manager)
        self.dialogue_manager = DialogueManager(API_KEY, npc_prompt)
        
        self.history = [] # Historial de la conversación para mostrar en pantalla
        self.input_text = ""
        self.font = pygame.font.Font(None, 28)
        self.input_rect = pygame.Rect(50, 500, 700, 40)
        self.history.append("Te encuentras frente a un personaje. ¿Qué le dices?")

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RETURN:
                    if self.input_text:
                        # Añadimos la pregunta del jugador al historial
                        self.history.append(f"> {self.input_text}")
                        # Enviamos el mensaje a la IA
                        response = self.dialogue_manager.send_message(self.input_text)
                        # Añadimos la respuesta de la IA al historial
                        self.history.append(response)
                        self.input_text = "" # Limpiamos la entrada
                elif event.key == pygame.K_BACKSPACE:
                    self.input_text = self.input_text[:-1]
                else:
                    self.input_text += event.unicode
                
                # Limitar el historial para que no se salga de la pantalla
                if len(self.history) > 15:
                    self.history.pop(0)

    def draw(self, surface):
        surface.fill((30, 30, 50))
        
        # Dibujar historial
        y_offset = 20
        for line in self.history:
            text_surface = self.font.render(line, True, (255, 255, 255))
            surface.blit(text_surface, (20, y_offset))
            y_offset += 30
            
        # Dibujar caja de entrada
        pygame.draw.rect(surface, (80, 80, 100), self.input_rect)
        input_surface = self.font.render(self.input_text, True, (255, 255, 255))
        surface.blit(input_surface, (self.input_rect.x + 5, self.input_rect.y + 5))
adventure/scenes/office.py (Pequeño Añadido)
Ahora, un hotspot puede transicionar a la escena de diálogo.

Python

# adventure/scenes/office.py
# ...
from .dialogue_scene import DialogueScene

class OfficeScene(RoomScene):
    def __init__(self, scene_manager, inventory):
        # ...
        self.hotspots.append(Hotspot((100, 150, 150, 250), self.on_click_detective))

    def on_click_detective(self, mode, item):
        if mode == 'talk':
            # El "corazón" de la personalidad del NPC
            prompt = """
            Eres el Detective Jones, un investigador privado cínico y cansado del mundo.
            Estás en tu oficina investigando la desaparición de una valiosa estatuilla, 'El Halcón de Silicio'.
            Hablas con frases cortas y sarcásticas. No confías en nadie.
            Sabes que la estatuilla fue vista por última vez en el muelle 5.
            No reveles la pista del muelle 5 a menos que el jugador te pregunte específicamente por 'pistas' o 'dónde buscar'.
            Tu objetivo es ver si el jugador sabe algo, sin revelar demasiado.
            """
            self.scene_manager.go_to(DialogueScene, npc_prompt=prompt)
        elif mode == 'look':
            print("Es un detective con cara de pocos amigos. Mejor no molestarle... o sí.")
        # ...
Clase 23: Diálogos con IA Generativa
Contenidos de la Clase
Discusión sobre IA en Juegos (20 min): Iniciar con la discusión sobre las limitaciones de los árboles de diálogo y el potencial revolucionario de la IA generativa, usando los ejemplos mencionados.

Configurando la API de Gemini (25 min):

Guiar a los alumnos a través del proceso de obtener una API Key en Google AI Studio.

Explicar la importancia de mantener la clave en un archivo config.py y nunca subirlo a un repositorio público.

Instalar la librería de Google (pip install google-generativeai).

El Arte de la Ingeniería de Prompts (40 min):

Presentar el DialogueManager y su system_prompt.

Explicar que el prompt es la parte más importante. Es donde "programamos" la personalidad. Desglosar un buen prompt:

Rol: "Eres el Detective Jones..."

Contexto: "...investigando la desaparición de 'El Halcón de Silicio'."

Personalidad/Tono: "...cínico y cansado. Hablas con frases cortas y sarcásticas."

Conocimiento Oculto: "Sabes que fue vista en el muelle 5."

Reglas/Condiciones: "No reveles la pista a menos que te pregunten por 'pistas'..."

Objetivo: "Tu objetivo es ver si el jugador sabe algo."

Creando la Interfaz de Diálogo (25 min):

Implementar la DialogueScene. Mostrar cómo gestiona un historial de texto y la entrada del teclado del jugador.

Conectar el handle_events para que al pulsar Enter, se envíe el texto al DialogueManager y se muestre la respuesta.

Conectar la OfficeScene para que al "hablar" con un personaje, se transicione a la DialogueScene, pasándole el prompt específico de ese personaje.

Cierre y Fin del Curso
Resumen y Celebración Final: ¡Lo hemos logrado! Hemos completado un viaje increíble, construyendo un motor de juegos desde cero y usándolo para crear tres géneros distintos. Hoy, hemos dado un paso hacia el futuro del desarrollo de juegos, implementando un NPC con una mente propia, impulsado por IA generativa. Hemos aprendido no solo a programar, sino a diseñar personalidades y conversaciones a través del arte de la ingeniería de prompts.

¡Ahora eres un Desarrollador de Juegos! Tienes un motor robusto y un portafolio con tres juegos diferentes. Tienes el conocimiento para empaquetarlos y compartirlos. Pero lo más importante, tienes la base para seguir aprendiendo y creando cualquier juego que puedas imaginar. El viaje no termina aquí; apenas comienza. ¡Enhorabuena!






me gustaria dar un paso más. hacer un juego deportivo, por ejemplo un juego de futbol 5 con 4 equipos reales y sus jugadores, pero la jugabilidad o forma para ahcer un gol, no sea con el cursos, sino tipo en monkey island cuando guybrush aprendia a usar la epsada, y era dicendo frase y respondiendolas, una especie de isnultos tontos, podemos hacer sto?




Ver razonamiento
¡Absolutamente! Me encanta la idea. Es una fusión creativa y brillante que combina la estrategia de una aventura gráfica con la temática de un juego deportivo. Tomar la mecánica de "duelo de insultos" de Monkey Island y aplicarla al fútbol es una forma fantástica de crear un juego único y con mucha personalidad.

Llamémoslo "Fútbol de Duelo Verbal".

Aquí, la habilidad no estará en tus reflejos con el joystick, sino en tu ingenio, tu memoria y tu capacidad para elegir la "jugada" verbal correcta en el momento justo.

Manos a la obra. Este es un nuevo módulo, así que lo enfocaremos como tal.

1. Discusión: De Duelos de Espadas a Duelos en el Campo
¿Cómo traducimos la mecánica?

La Espada es el Balón: El jugador que tiene el balón es el atacante, el que "lanza el insulto". El defensor cercano es quien debe "responder a la ofensa".

"Insulto" = Jugada de Ataque: La frase del atacante no es un insulto literal, sino una descripción florida y jactanciosa de la jugada que va a intentar.

Ejemplo: "¡Voy a hacer un túnel tan grande que podrás aparcar un autobús dentro!"

"Réplica" = Maniobra Defensiva: La respuesta del defensor es la acción táctica para contrarrestar la jugada. De varias opciones, solo una es la correcta.

Ejemplo de réplica correcta: "Inténtalo, pero mis piernas están más cerradas que la caja fuerte de un banco."

Ejemplo de réplica incorrecta: "¡Pues la matrícula de mi autobús es 'G-O-L'!" (El atacante te supera).

El Flujo del Partido: El juego es por turnos. Un equipo ataca. El jugador con el balón se enfrenta a un defensor, iniciando un duelo. Si el atacante gana el duelo, avanza en el campo. Si el defensor gana, roba el balón. Después de varios duelos ganados, el atacante llega al área y se enfrenta al portero en un duelo final y más difícil para marcar el gol.

El Rol de los Jugadores y sus Estadísticas:

Los jugadores reales que mencionas no tendrán velocidad o fuerza, sino atributos mentales:

Creatividad (Ataque): Determina la calidad y variedad de las "frases de ataque" que un jugador puede usar. Un Messi o un Ronaldinho tendrían un repertorio de frases casi imbatible.

Anticipación (Defensa): Afecta la cantidad de "opciones de réplica" que se le presentan al defensor. Un buen defensor como Maldini podría tener solo 2 opciones (50% de acierto), mientras que un delantero defendiendo podría tener 4 o 5, haciéndolo más difícil.

Temple (Finalización): Afecta la dificultad del duelo final contra el portero.

2. Estructura de Archivos y Datos
Este nuevo proyecto tendrá su propia carpeta y, de forma crucial, archivos de datos para los equipos y los diálogos.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- challenge.py         # ¡NUEVO! Nuestro motor de duelos verbales.
|
|-- insult_soccer/           # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- player_teamA.png
|   |   |-- player_teamB.png
|   |   |-- field.png
|   |
|   |-- data/                # ¡NUEVO!
|   |   |-- teams.json
|   |   |-- challenges.json
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- match_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt
insult_soccer/data/teams.json (Ejemplo)
JSON

{
  "equipos": [
    {
      "nombre": "Titanes de la Capital",
      "jugadores": [
        {"nombre": "Leo 'El Mago'", "creatividad": 95, "anticipacion": 40, "temple": 92},
        {"nombre": "Sergio 'El Muro'", "creatividad": 30, "anticipacion": 98, "temple": 60}
      ]
    },
    {
      "nombre": "Furia del Interior",
      "jugadores": [
        {"nombre": "Carlos 'El Cohete'", "creatividad": 91, "anticipacion": 50, "temple": 85},
        {"nombre": "Roberto 'El Káiser'", "creatividad": 40, "anticipacion": 95, "temple": 70}
      ]
    }
  ]
}
insult_soccer/data/challenges.json (Ejemplo)
JSON

{
  "duelos": [
    {
      "id": 1,
      "ataque": "Voy a driblarte con la elegancia de un bailarín de tango.",
      "replica": "Hasta el mejor bailarín tropieza si le cambian el suelo.",
      "fallos": ["¡Pero esto es fútbol, no un salón de baile!", "Mi especialidad es bailar sobre tus tobillos."]
    },
    {
      "id": 2,
      "ataque": "Mi disparo tiene más efecto que tu café de la mañana.",
      "replica": "No importa el efecto si el portero es un muro de hormigón.",
      "fallos": ["¡Yo tomo té, no café!", "Pues a mí me gusta el café amargo."]
    }
  ]
}
3. El Código del Motor Actualizado
Añadimos un nuevo módulo para manejar estos duelos.

engine/challenge.py (¡NUEVO!)
Python

# engine/challenge.py
import random

class ChallengeManager:
    def __init__(self, challenges_data):
        self.challenges = challenges_data['duelos']
        self.current_challenge = None
        self.options = []

    def start_random_challenge(self):
        """Inicia un nuevo duelo aleatorio."""
        self.current_challenge = random.choice(self.challenges)
        
        # Preparamos las opciones para el defensor
        correct_answer = self.current_challenge['replica']
        wrong_answers = self.current_challenge['fallos']
        
        self.options = [correct_answer] + wrong_answers
        random.shuffle(self.options)
        
        return self.current_challenge['ataque'], self.options

    def resolve_choice(self, chosen_option):
        """Comprueba si la opción elegida es la réplica correcta."""
        if not self.current_challenge:
            return False
        return chosen_option == self.current_challenge['replica']

4. El Código del Juego de Duelo Verbal
insult_soccer/scenes/match_scene.py (¡NUEVO!)
Esta es la escena principal donde ocurre la magia.

Python

# insult_soccer/scenes/match_scene.py
import pygame
import json
from engine.scene import Scene
from engine.challenge import ChallengeManager

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # Carga de datos
        with open('insult_soccer/data/challenges.json', 'r', encoding='utf-8') as f:
            challenges_data = json.load(f)
        self.challenge_manager = ChallengeManager(challenges_data)

        # Estado del juego
        self.game_state = "new_challenge" # 'new_challenge', 'player_defending', 'result'
        self.attacker_phrase = ""
        self.defender_options = []
        self.result_text = ""
        
        self.font = pygame.font.Font(None, 32)
        self.option_rects = []

    def update(self):
        if self.game_state == "new_challenge":
            self.attacker_phrase, self.defender_options = self.challenge_manager.start_random_challenge()
            self.game_state = "player_defending"

    def handle_events(self, events):
        if self.game_state == "player_defending":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    for i, rect in enumerate(self.option_rects):
                        if rect.collidepoint(event.pos):
                            chosen_option = self.defender_options[i]
                            is_correct = self.challenge_manager.resolve_choice(chosen_option)
                            if is_correct:
                                self.result_text = "¡Defensa exitosa! Recuperas el balón."
                            else:
                                self.result_text = "¡Te han superado! El rival avanza."
                            self.game_state = "result"
                            break

        elif self.game_state == "result":
            for event in events:
                if event.type == pygame.KEYDOWN or event.type == pygame.MOUSEBUTTONDOWN:
                    self.game_state = "new_challenge"


    def draw(self, surface):
        surface.fill((50, 150, 50)) # Verde campo de fútbol

        # Dibujar la frase del atacante
        attacker_surf = self.font.render(f"RIVAL: '{self.attacker_phrase}'", True, (255, 255, 255))
        surface.blit(attacker_surf, (50, 100))

        if self.game_state == "player_defending":
            # Dibujar las opciones del defensor
            self.option_rects = []
            for i, option in enumerate(self.defender_options):
                option_surf = self.font.render(f"{i+1}. {option}", True, (220, 220, 0))
                option_rect = option_surf.get_rect(topleft=(70, 200 + i * 50))
                self.option_rects.append(option_rect)
                surface.blit(option_surf, option_rect)
        
        elif self.game_state == "result":
            result_surf = self.font.render(self.result_text, True, (255, 255, 255))
            result_rect = result_surf.get_rect(center=(surface.get_width()/2, surface.get_height()/2))
            surface.blit(result_surf, result_rect)
Clase 24: Fusión de Géneros - El Duelo Verbal Deportivo
Contenidos de la Clase
Discusión sobre Fusión de Géneros (20 min): Iniciar la clase discutiendo cómo la combinación de mecánicas de géneros aparentemente opuestos puede dar lugar a juegos innovadores y memorables. Usar el concepto de "Fútbol de Duelo Verbal" como caso de estudio.

Diseñando la Base de Datos (30 min):

Explicar la importancia de externalizar los datos.

Diseñar y crear los archivos teams.json y challenges.json. Discutir cómo la estructura de estos archivos define las posibilidades del juego. Enseñar a los alumnos a añadir sus propios equipos y duelos.

El Motor de Desafíos (35 min):

Implementar la clase ChallengeManager en el motor.

Explicar su función: es un sistema agnóstico al juego que puede gestionar cualquier tipo de duelo "pregunta-respuesta". Podría reutilizarse para un concurso de Trivial, por ejemplo.

Detallar la lógica de start_random_challenge y cómo baraja las opciones para que la respuesta correcta no esté siempre en el mismo sitio.

Implementando la Escena del Partido (35 min):

Crear la MatchScene, que será el núcleo del juego.

Implementar la máquina de estados simple (new_challenge, player_defending, result).

Mostrar cómo se usa el ChallengeManager para obtener los datos del duelo y cómo se gestiona la interacción del jugador (clic en las opciones) para resolver el desafío.

Cierre y Próximos Pasos
Resumen: ¡Hemos dado un paso increíblemente creativo! Hemos diseñado y comenzado a implementar un juego que fusiona la estrategia de una aventura gráfica con la temática del deporte. Hemos creado un motor de desafíos reutilizable y una estructura de datos que nos permite expandir el juego fácilmente con nuevos equipos y diálogos.

Adelanto de la Clase 25: "Tenemos el duelo individual. Ahora, tenemos que crear el 'partido'. En la próxima clase, daremos vida al campo de juego. Posicionaremos los sprites de los jugadores de ambos equipos en el campo. Implementaremos un sistema de turnos y posesión. El resultado de un duelo ya no será un simple texto, sino que provocará que el 'balón' (la posesión) se mueva de un jugador a otro en el campo, conectando los duelos para crear una jugada completa hasta llegar al área rival."






vamos por la siguiente clase



Ver razonamiento
¡Perfecto! Avancemos. Es hora de sacar nuestro juego del abstracto menú de duelo y ponerlo sobre el césped.

En esta clase, daremos vida al partido. Conectaremos nuestra mecánica de duelo verbal con una representación visual en el campo de juego. Los jugadores ya no serán solo nombres, sino sprites que se mueven, se enfrentan y luchan por la posesión del balón, creando una verdadera progresión táctica hacia la portería.

1. Discusión: El Tablero de Juego Deportivo
Muchos de los mejores juegos de estrategia deportiva no se basan en la acción en tiempo real. En su lugar, tratan el campo de juego como un tablero, similar al ajedrez o a un juego de rol táctico.

El Espacio y las Zonas: El campo se divide en zonas (defensa, mediocampo, ataque). El objetivo es mover la "pieza" que tiene el balón de una zona a la siguiente, superando los desafíos que se presentan.

Encuentros por Turnos: Cuando una pieza atacante (con el balón) entra en el radio de acción de una pieza defensora, se desencadena un "encuentro". En nuestro caso, este encuentro es un duelo verbal. El resto del tablero "espera" el resultado de este encuentro individual.

Resolución y Consecuencia: El resultado del encuentro tiene una consecuencia directa y espacial. Ganar significa avanzar a la siguiente zona. Perder significa que la posesión (el balón) pasa a la pieza defensora, y los roles se invierten.

Ejemplos en juegos conocidos:

Football Manager ⚽: El ejemplo definitivo. El partido se desarrolla a través de estadísticas, posiciones y eventos clave. No controlas directamente el disparo, sino que pones a tus mejores piezas en las mejores posiciones para que la simulación resuelva el encuentro a tu favor.

Blood Bowl 🏈: Una versión de fantasía del fútbol americano que es, en esencia, un juego de mesa por turnos sobre una rejilla. Cada acción (mover, placar, pasar) es un "encuentro" que se resuelve con una tirada de dados, y el resultado determina el nuevo estado del tablero.

Final Fantasy Tactics / XCOM ♟️: Aunque no son juegos deportivos, su mecánica es la misma que buscamos. Mueves tus unidades en un tablero por turnos. Cuando un enemigo está al alcance, inicias un "encuentro" (un ataque). El resultado (acierto o fallo) determina el siguiente movimiento.

2. Estructura de Archivos Actualizada
No necesitamos nuevos módulos en el motor, pero sí un nuevo archivo para nuestras entidades de juego y un asset para el campo.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- insult_soccer/
|   |-- assets/
|   |   |-- field.png        # ¡NUEVO! Imagen del campo de fútbol 5.
|   |
|   |-- data/
|   |   |-- ... (sin cambios)
|   |
|   |-- entities/            # ¡NUEVO!
|   |   |-- __init__.py
|   |   |-- player.py
|   |
|   |-- scenes/
|   |   |-- ... (sin cambios)
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Juego
insult_soccer/entities/player.py (¡NUEVO!)
Creamos una clase para nuestro jugador, que contendrá tanto sus datos (nombre, stats) como su representación visual (sprite).

Python

# insult_soccer/entities/player.py
import pygame
from engine.game_object import GameObject

class SoccerPlayer(GameObject):
    def __init__(self, x, y, image_path, player_data):
        super().__init__(x, y, image_path)
        
        self.player_data = player_data
        self.nombre = player_data['nombre']
        self.creatividad = player_data['creatividad']
        self.anticipacion = player_data['anticipacion']
        self.temple = player_data['temple']
        
        self.is_selected = False # Para resaltar al jugador activo

    def draw(self, surface):
        # Sobreescribimos draw para añadir un resaltado si está seleccionado
        super().draw(surface)
        if self.is_selected:
            pygame.draw.rect(surface, (255, 255, 0), self.rect, 2) # Borde amarillo
insult_soccer/scenes/match_scene.py (Grandes Cambios)
Esta escena se vuelve mucho más compleja. Ahora gestiona la posición de los jugadores, la posesión del balón y el flujo del partido.

Python

# insult_soccer/scenes/match_scene.py
import pygame
import json
from engine.scene import Scene
from engine.challenge import ChallengeManager
from ..entities.player import SoccerPlayer

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        self.load_data()
        self.setup_field()
        self.start_match()

    def load_data(self):
        with open('insult_soccer/data/teams.json', 'r', encoding='utf-8') as f:
            self.teams_data = json.load(f)['equipos']
        with open('insult_soccer/data/challenges.json', 'r', encoding='utf-8') as f:
            challenges_data = json.load(f)
        self.challenge_manager = ChallengeManager(challenges_data)
        self.background = pygame.image.load('insult_soccer/assets/field.png').convert()

    def setup_field(self):
        """Crea y posiciona a los jugadores en el campo."""
        self.todos_los_sprites = pygame.sprite.Group()
        self.team_a_players = pygame.sprite.Group()
        self.team_b_players = pygame.sprite.Group()

        # Posiciones fijas para un 2-1-1 (Portero, 2 Def, 1 Med, 1 Del)
        # Equipo A (izquierda), Equipo B (derecha)
        positions_a = [(50, 300), (200, 150), (200, 450), (350, 300), (450, 300)]
        positions_b = [(750, 300), (600, 150), (600, 450), (450, 300), (350, 300)]

        team_a_data = self.teams_data[0]['jugadores']
        for i, pos in enumerate(positions_a):
            player = SoccerPlayer(pos[0], pos[1], 'insult_soccer/assets/player_teamA.png', team_a_data[i])
            self.team_a_players.add(player)

        # ... (hacer lo mismo para el equipo B) ...
        self.todos_los_sprites.add(self.team_a_players, self.team_b_players)

    def start_match(self):
        """Inicia el estado del partido."""
        self.possession_team = self.team_a_players # Empieza el equipo A
        self.player_with_ball = self.team_a_players.sprites()[0] # El portero
        self.player_with_ball.is_selected = True
        self.game_state = "player_move" # 'player_move', 'challenge', 'result'
        # ... (resto de la lógica de estado) ...

    def start_new_challenge(self):
        """Inicia un duelo entre el atacante y un defensor cercano."""
        self.player_with_ball.is_selected = True
        # Lógica para encontrar al defensor más cercano
        self.current_defender = self.find_nearest_defender()
        self.current_defender.is_selected = True
        
        self.attacker_phrase, self.defender_options = self.challenge_manager.start_random_challenge()
        self.game_state = "challenge"

    def resolve_challenge(self, success):
        self.player_with_ball.is_selected = False
        self.current_defender.is_selected = False
        
        if success:
            self.result_text = "¡Avance exitoso!"
            # Lógica para mover el balón al siguiente jugador del equipo atacante
            # self.player_with_ball = find_next_attacker(self.player_with_ball)
        else:
            self.result_text = "¡Balón robado!"
            # El defensor ahora tiene el balón, se invierte la posesión
            self.possession_team = self.team_b_players # o el equipo que corresponda
            self.player_with_ball = self.current_defender
        
        self.game_state = "result"
        
    def update(self):
        if self.game_state == "player_move":
            # En un juego más complejo, aquí el jugador elegiría a quién pasar
            # Por ahora, iniciamos un desafío automáticamente
            self.start_new_challenge()

    def handle_events(self, events):
        if self.game_state == "challenge":
            # ... (lógica de clic en las opciones, como en la clase anterior) ...
            # Al resolver, llamar a self.resolve_challenge(is_correct)
            pass
        elif self.game_state == "result":
            # ... (esperar clic para continuar) ...
            # self.game_state = "player_move"
            pass

    def draw(self, surface):
        surface.blit(self.background, (0, 0))
        self.todos_los_sprites.draw(surface)

        if self.game_state == "challenge":
            # ... (dibujar la UI del duelo) ...
            pass
        elif self.game_state == "result":
            # ... (dibujar el texto del resultado) ...
            pass
Clase 25: El Partido - Posesión, Turnos y Movimiento en el Campo
Contenidos de la Clase
Discusión sobre el "Tablero Deportivo" (20 min): Iniciar con la discusión sobre cómo los juegos de estrategia representan la acción deportiva, usando los ejemplos de Football Manager y Blood Bowl.

La Entidad Jugador (30 min):

Crear la clase SoccerPlayer, que combina un GameObject con los datos cargados del JSON (nombre, stats).

Implementar una forma simple de feedback visual, como un borde de resaltado, para indicar qué jugadores están activos en el duelo actual.

Montando el Escenario (35 min):

En MatchScene, implementar el método setup_field. Cargar los datos de los equipos desde el JSON y crear una instancia de SoccerPlayer para cada uno, posicionándolos en una formación táctica fija en el campo.

Introducir el concepto de posesión. Crear variables como self.possession_team y self.player_with_ball para rastrear quién tiene el control.

Conectando Duelos y Jugadas (35 min):

Refinar la máquina de estados para reflejar el flujo de un partido: player_move (decidir la siguiente acción), challenge (resolver el duelo), result (ver la consecuencia).

Implementar la lógica clave en resolve_challenge:

Si el atacante gana, la posesión se mueve a un compañero de equipo más adelantado en el campo.

Si el defensor gana, la posesión cambia de equipo, y el defensor se convierte en el nuevo atacante.

Mostrar cómo el resultado del duelo ahora tiene una consecuencia espacial y táctica en el tablero de juego.

Cierre y Próximos Pasos
Resumen: ¡Nuestro juego ha cobrado vida! Hemos pasado de un simple duelo de frases a un partido táctico por turnos. Hemos poblado el campo con jugadores, implementado un sistema de posesión y conectado los resultados de los duelos verbales a un movimiento estratégico en el campo. El juego ahora tiene un flujo y un objetivo claros.

Adelanto de la Clase 26: "Hemos llegado a la zona de ataque. ¿Y ahora qué? En la próxima clase, implementaremos la jugada final: el tiro a puerta. Este será un duelo especial y más difícil contra el portero. También añadiremos un marcador, un sistema de goles y la lógica para reiniciar el partido después de un gol, completando así el bucle de juego y convirtiendo nuestro prototipo en un juego completo de principio a fin."






perfecto! avanza, e introduce estas variantes si te parece, cada equipo tiene 5 jugadores, que los jugaores se puedan mover con las flechas, y haya un botón para cambiar de jugador seleccionado por quipo, otro para pasar la pelota, otro para marcar y otro para tirar, el de marcar sirve para atajar también, agrega una zona, la portería. y que además de los atributos que definiste por jugador, haya un atributo de área circular, cuanto mayor sea el área de un marcador, antes podra intentar bloquear al rival.




Ver razonamiento
¡Entendido! Esto es un giro fascinante y ambicioso. Abandonamos el sistema por turnos para sumergirnos en la acción en tiempo real, pero manteniendo la esencia de "rol" donde las estadísticas de los jugadores deciden el resultado de las acciones.

Me parece una evolución excelente. Estamos creando un género híbrido que podríamos llamar "Fútbol de Rol Táctico en Tiempo Real". El jugador tiene el control directo del movimiento, pero el éxito de un pase, un tiro o una entrada no depende de su habilidad para pulsar un botón en el milisegundo exacto, sino de las estadísticas del jugador que controla.

Manos a la obra. Adaptaremos nuestro motor y nuestro juego a esta nueva y emocionante visión.

1. Discusión: El Híbrido de Acción y Rol
Este nuevo modelo combina lo mejor de dos mundos:

Control Directo (Acción): El jugador se siente en control. Mover a su jugador favorito con las flechas, posicionarse para un pase o una entrada, crea una conexión inmediata y visceral con el juego.

Resolución por Estadísticas (Rol): Aquí es donde nuestro diseño original sigue vivo. Cuando pulsas el botón de "pasar", no eres tú quien define la precisión, es la estadística de Creatividad de tu jugador. Cuando intentas una entrada, tu éxito depende de tu Anticipación contra la Creatividad del rival. Esto hace que la elección del jugador y la táctica del equipo sigan siendo fundamentales.

¿Cómo se readaptan nuestras estadísticas?

Creatividad (Ataque): Ahora determina la calidad de las acciones ofensivas.

Pase: Un jugador con alta Creatividad dará pases más precisos y rápidos, más difíciles de interceptar.

Tiro: Afecta la precisión del disparo a puerta.

Anticipación (Defensa): Se convierte en el área de influencia que propusiste. Un jugador con alta Anticipación tiene un círculo de marcaje más grande. Si el rival con el balón entra en ese círculo, el defensor puede intentar una entrada.

Temple (Duelos Clave): Se usa en momentos de máxima presión.

Tiro a Puerta: El Temple del delantero se enfrenta al Temple del portero para determinar la probabilidad de gol.

Entradas Críticas: Podría usarse en un duelo 50/50 por un balón suelto.

2. Estructura de Archivos Actualizada
Reflejamos el nuevo género en el nombre del proyecto y añadimos una entidad para el balón.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- soccer_rpg/              # ¡PROYECTO RENOMBRADO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- ball.png         # ¡NUEVO!
|   |
|   |-- entities/
|   |   |-- __init__.py
|   |   |-- player.py
|   |   |-- ball.py          # ¡NUEVO!
|   |
|   |-- scenes/
|   |   |-- ... (sin cambios)
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Juego Actualizado
soccer_rpg/entities/ball.py (¡NUEVO!)
El balón ahora es su propia entidad, controlada por físicas para un movimiento suave.

Python

# soccer_rpg/entities/ball.py
import pygame
import pymunk
from engine.physics_object import PhysicsObject

class Ball(PhysicsObject):
    def __init__(self, x, y, space):
        super().__init__(x, y, 'soccer_rpg/assets/ball.png', space, mass=0.5, radius=8, elasticity=0.7)
        self.body.damping = 0.97 # Fricción con el césped
        self.carrier = None # Jugador que lleva el balón

    def update(self):
        super().update()
        # Si un jugador lleva el balón, la posición del balón es la del jugador
        if self.carrier:
            self.body.position = self.carrier.rect.center
            self.body.velocity = 0, 0 # El balón no se mueve por sí mismo
soccer_rpg/entities/player.py (Grandes Cambios)
El jugador ahora es mucho más complejo, con lógica de movimiento y acciones.

Python

# soccer_rpg/entities/player.py
import pygame
from engine.game_object import GameObject

class SoccerPlayer(GameObject):
    def __init__(self, x, y, image_path, player_data, team):
        super().__init__(x, y, image_path)
        
        self.player_data = player_data
        self.nombre = player_data['nombre']
        self.creatividad = player_data['creatividad']
        self.anticipacion = player_data['anticipacion']
        self.temple = player_data['temple']
        self.team = team
        
        self.velocidad = 4
        # El área de marcaje depende de la Anticipación
        self.radio_marcaje = 20 + (self.anticipacion / 10)

    def move(self, dx, dy):
        self.rect.x += dx * self.velocidad
        self.rect.y += dy * self.velocidad

    def shoot(self, ball, goal_pos):
        # La probabilidad de éxito depende del Temple vs. el del portero (lógica futura)
        print(f"{self.nombre} chuta a puerta!")
        ball.carrier = None
        fuerza = 20 + (self.temple / 10)
        # ... (código para aplicar impulso al cuerpo físico del balón) ...
        
    def pass_ball(self, ball, target_player):
        # La precisión depende de la Creatividad
        print(f"{self.nombre} pasa el balón a {target_player.nombre}")
        ball.carrier = None
        # ... (código para aplicar impulso al balón hacia el compañero) ...
        
    def tackle(self, target_player):
        # La probabilidad de éxito depende de la Anticipación vs. Creatividad del rival
        print(f"{self.nombre} intenta robar el balón a {target_player.nombre}")
        # ... (lógica de resolución del robo) ...

    def draw(self, surface, is_active):
        super().draw(surface)
        if is_active:
            pygame.draw.rect(surface, (255, 255, 0), self.rect, 2)
        
        # Dibujar el radio de marcaje si no tiene el balón
        if not is_active:
            pygame.draw.circle(surface, self.team, self.rect.center, self.radio_marcaje, 1)

soccer_rpg/scenes/match_scene.py (Grandes Cambios)
La escena ahora es un bucle de juego en tiempo real que gestiona la selección de jugador y las acciones.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ... (cargar datos, crear espacio de físicas) ...
        
        self.active_player = self.team_a_players.sprites()[0]
        self.ball = Ball(self.active_player.rect.centerx, self.active_player.rect.centery, self.space)
        self.ball.carrier = self.active_player
        self.todos_los_sprites.add(self.ball)

    def handle_events(self, events):
        # Este método ahora solo gestiona los inputs del teclado
        keys = pygame.key.get_pressed()
        
        # Movimiento del jugador activo
        dx, dy = 0, 0
        if keys[pygame.K_LEFT]: dx = -1
        if keys[pygame.K_RIGHT]: dx = 1
        if keys[pygame.K_UP]: dy = -1
        if keys[pygame.K_DOWN]: dy = 1
        self.active_player.move(dx, dy)

        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_TAB: # Botón para cambiar de jugador
                    self.switch_player()
                if event.key == pygame.K_x: # Botón de pase
                    if self.ball.carrier == self.active_player:
                        target = self.find_nearest_teammate()
                        self.active_player.pass_ball(self.ball, target)
                if event.key == pygame.K_c: # Botón de marcar / chutar
                    if self.ball.carrier == self.active_player:
                        self.active_player.shoot(self.ball, self.goal_b_rect)
                    else:
                        target = self.find_opponent_with_ball_in_range()
                        if target:
                            self.active_player.tackle(target)

    def switch_player(self):
        # Lógica para encontrar al jugador del equipo humano más cercano al balón
        # y asignarlo como self.active_player
        pass

    def update(self):
        self.space.step(1/60.0)
        self.todos_los_sprites.update()
        # Aquí iría la IA del equipo rival para mover a sus jugadores
Clase 26: Fútbol de Rol en Tiempo Real
Contenidos de la Clase
Discusión sobre el Género Híbrido (20 min): Iniciar la clase presentando el nuevo concepto de "Fútbol de Rol Táctico en Tiempo Real". Discutir cómo se reinterpretan las estadísticas de los jugadores (Creatividad, Anticipación, Temple) en este nuevo contexto de acción directa.

El Balón como Entidad Física (25 min):

Crear la clase Ball como un PhysicsObject.

Implementar la lógica de carrier: cuando un jugador tiene el balón, la posición del balón se fija a la del jugador y sus físicas se "pausan". Cuando se pasa o se chuta, se libera (carrier = None) y el motor de físicas toma el control.

El Jugador Multifacético (40 min):

Refactorizar masivamente la clase SoccerPlayer.

Implementar el movimiento directo con las flechas en el método update o en un nuevo método move.

Añadir el atributo de radio_marcaje basado en la Anticipación y visualizarlo.

Esbozar los métodos de acción: pass_ball, shoot, tackle. Explicar que en la próxima clase se implementará la "resolución" basada en stats de estas acciones.

Orquestando la Acción en Tiempo Real (35 min):

Modificar la MatchScene para que el bucle principal sea en tiempo real.

Implementar la lógica en handle_events para capturar los botones de acción: cambiar jugador, pasar, y el botón contextual de chutar/marcar.

Mostrar cómo el estado del juego ya no es por turnos, sino que depende constantemente de quién es el active_player y quién es el ball.carrier.

Cierre y Próximos Pasos
Resumen: ¡Hemos realizado una transformación radical y emocionante! Hemos evolucionado nuestro juego de un concepto por turnos a un híbrido de acción y rol en tiempo real. Nuestro motor ahora soporta control directo de personajes, acciones contextuales y una interacción mucho más dinámica entre las entidades y la física.

Adelanto de la Clase 27: "Controlamos a nuestro equipo, pero el rival está inmóvil. En la próxima clase, daremos el paso definitivo para crear un partido completo: programaremos la Inteligencia Artificial del equipo rival. Crearemos una IA basada en estados para los jugadores controlados por la máquina: decidirán cuándo presionar, cuándo mantener la posición, y los jugadores ofensivos se desmarcarán para recibir pases, ¡creando un oponente dinámico y desafiante!"






vamos por la nueva clase. Y ten presenta para alguna futura clase, como evaluar quien hizo el mejor insulto poetico, en el duelo, si el defensor respondió bien. podemos pensar capaz mejor que insultos en que digan poesía. si hay duelo, quien ataca dice la poesía, y el defensor la responde, necesitamos que con ia tipo gemini, evalue si el nivel de poesía del atacante es mejor que el defensa, en ese escenario avanza, si el de defensor es mejor le quita la pelota, o que empatn y tengan que seguir el duelo. quiza los jugadores más habilidosos, pueden ser ayudados por la ia para generar mejor poesía, donde se le sugieran palabras o frases. tambien cuando se haga un pase, se deeb permitor que con las flechas se diriga ese pase. seria bueno también que el juez, si pita falta, lo haga con una rima. y que el estadio tenga ambientación sonora, que se pueda habilitar o no, y que lso cantos sea literatuera de alta calidad para aentar a los jugadores




Ver razonamiento
¡Fantástico! Estas son ideas excepcionales que elevan el proyecto a un nivel completamente nuevo de originalidad y creatividad. Olvidemos el simple "Fútbol de Rol Táctico"; estamos a punto de crear el primer "Fútbol Poético: El Duelo Lírico" del mundo.

Me encanta. Esta clase fusionará todo lo que hemos aprendido: la gestión de escenas, el control de personajes en tiempo real, el sistema de diálogos y la potencia de la IA generativa. Abróchate el cinturón.

1. Discusión: La Síntesis Final - Acción, Rol y Poesía
Estamos creando un género híbrido único. Así es como funcionará nuestro nuevo modelo:

Fase de Acción en Tiempo Real:

Controlas a un jugador de tu equipo con las flechas, moviéndote libremente por el campo.

Puedes cambiar al jugador más cercano al balón con una tecla (TAB).

Si tienes el balón, puedes iniciar un pase. Al pulsar el botón de pase, el juego entra en un modo de "apuntado" donde diriges el pase con las flechas antes de soltarlo.

Fase de Duelo Poético (Por Turnos):

Cuando un atacante con el balón entra en el área de influencia (el círculo de Anticipación) de un defensor, el juego en tiempo real se pausa.

Comienza un Duelo Lírico. La cámara se centra en los dos jugadores.

El atacante (sea la IA o el jugador) presenta un verso poético.

El defensor (sea la IA o el jugador) debe responder con otro verso.

Ambos versos se envían a la IA (Gemini), que actúa como un juez literario imparcial.

Resolución por IA:

La IA evalúa los versos basándose en la creatividad, la métrica, la rima y la temática futbolística.

Devuelve un resultado: ATAQUE_GANA, DEFENSA_GANA o EMPATE.

Si el ataque gana: El defensor queda "roto" y el atacante avanza, continuando en tiempo real.

Si la defensa gana: El defensor roba el balón y la posesión cambia.

Si hay empate: Se puede iniciar un segundo verso para desempatar, o el balón queda suelto.

El Toque Artístico:

El árbitro es un bardo que canta las faltas con rimas.

La afición en el estadio no grita cánticos normales, sino que recita fragmentos de Borges, Shakespeare o Cervantes, creando una atmósfera surrealista y culta. El sonido, por supuesto, será opcional.

2. Estructura y Código: La Fusión
Reutilizaremos y adaptaremos gran parte de lo que ya hemos construido. El cambio principal estará en la MatchScene y en cómo usamos nuestro DialogueManager.

engine/dialogue.py (Adaptado para ser el Juez)
La clase es la misma, pero la forma en que la usamos cambia drásticamente. El system_prompt es la clave de todo.

Python

# Un ejemplo del prompt que usaríamos para inicializar el DialogueManager
PROMPT_JUEZ = """
Eres "El Crítico Lirico", un juez de duelos poéticos que ocurren durante un partido de fútbol. Tu criterio es impecable y tu juicio, final.
Te proporcionaré dos versos: un 'VERSO_ATACANTE' y un 'VERSO_DEFENSOR'.
Tu tarea es evaluar cuál es superior basándote en: 1. Creatividad y originalidad. 2. Calidad poética (rima, ritmo, métrica). 3. Relevancia temática con el fútbol, la estrategia o la emoción del momento.

Tu respuesta DEBE ser únicamente una de las siguientes tres palabras clave, seguida de dos puntos y una breve crítica rimada de dos líneas.

Las palabras clave son:
- ATACANTE_GANA
- DEFENSA_GANA
- EMPATE

Ejemplo 1 de respuesta:
DEFENSA_GANA: La réplica fue un muro, precisa y concisa, dejando al atacante sin su anhelada premisa.

Ejemplo 2 de respuesta:
ATACANTE_GANA: Con una metáfora audaz y un ritmo galopante, el verso del rival se ha vuelto irrelevante.

Ejemplo 3 de respuesta:
EMPATE: Ambos versos mostraron gran destreza y valía, el duelo continúa hasta el siguiente día.
"""
soccer_rpg/entities/player.py (Pequeños Añadidos)
Python

# soccer_rpg/entities/player.py
# ... (código existente) ...

class SoccerPlayer(GameObject):
    # ...
    def get_poetic_line(self, is_attacking=True):
        """
        Genera una línea de poesía base.
        En el futuro, la IA podría ayudar aquí para jugadores con alta Creatividad.
        """
        if is_attacking:
            # Lógica para generar o seleccionar una línea de ataque
            return "Mis pies danzan sobre el césped, un vals imprevisto,"
        else:
            # Lógica para generar o seleccionar una línea de defensa
            return "Pero tu danza termina donde mi pie ha pisado."
soccer_rpg/scenes/match_scene.py (Grandes Cambios)
Esta escena ahora gestionará una máquina de estados mucho más compleja.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.game_state = "REAL_TIME"
        # Estados: REAL_TIME, AIMING_PASS, POETRY_DUEL, WAITING_FOR_AI
        
        # Inicializamos el juez de IA
        self.juez_ia = DialogueManager(API_KEY, PROMPT_JUEZ)
        # ...

    def update(self):
        if self.game_state == "REAL_TIME":
            self.todos_los_sprites.update()
            # Comprobar si un atacante con balón entra en el radio de un defensor
            attacker = self.ball.carrier
            if attacker:
                for defender in self.get_opponent_team(attacker.team):
                    if pygame.sprite.collide_circle(attacker, defender, defender.radio_marcaje):
                        self.initiate_poetry_duel(attacker, defender)
                        break
        
        elif self.game_state == "POETRY_DUEL":
            # La lógica de esta UI esperaría el input del jugador
            pass

        elif self.game_state == "WAITING_FOR_AI":
            # Podríamos mostrar un icono de "pensando..."
            pass

    def handle_events(self, events):
        if self.game_state == "REAL_TIME":
            # ... (lógica de movimiento y botones de acción) ...
            if keys[pygame.K_x] and self.ball.carrier == self.active_player:
                self.game_state = "AIMING_PASS"
        
        elif self.game_state == "AIMING_PASS":
            # ... (lógica para dirigir el pase con flechas) ...
            # Al soltar el botón, se ejecuta el pase y se vuelve a REAL_TIME
            pass

        elif self.game_state == "POETRY_DUEL":
            # ... (lógica para que el jugador escriba su verso y pulse Enter) ...
            # Al pulsar Enter, se envía a la IA:
            # self.submit_duel_to_ai(attacker_verse, defender_verse)
            pass

    def initiate_poetry_duel(self, attacker, defender):
        self.game_state = "POETRY_DUEL"
        self.duel_participants = (attacker, defender)
        print(f"¡Duelo Poético entre {attacker.nombre} y {defender.nombre}!")
        # Aquí se mostraría la UI del duelo
        
    def submit_duel_to_ai(self, attacker_verse, defender_verse):
        self.game_state = "WAITING_FOR_AI"
        prompt_final = f"VERSO_ATACANTE: '{attacker_verse}'\nVERSO_DEFENSOR: '{defender_verse}'"
        
        # Esto debería hacerse en un hilo secundario para no congelar el juego,
        # pero para simplificar lo hacemos de forma directa.
        response = self.juez_ia.send_message(prompt_final)
        self.resolve_duel_result(response)

    def resolve_duel_result(self, result):
        # ... (lógica para procesar ATACANTE_GANA, DEFENSA_GANA o EMPATE) ...
        self.game_state = "REAL_TIME"

    def draw(self, surface):
        # ...
        if self.game_state == "POETRY_DUEL":
            # Dibujar la UI específica del duelo
            pass
Clase 27: El Duelo Lírico y el Juez de IA
Contenidos de la Clase
Discusión sobre la Jugabilidad Híbrida (20 min): Presentar el concepto final de "Fútbol Poético". Discutir cómo se alternará entre una fase de acción en tiempo real (movimiento, pases) y una fase por turnos (los duelos poéticos), y cómo esta combinación crea un ritmo de juego único.

El Juez de IA: Diseño de Prompts Avanzado (35 min):

El foco principal de la clase. Presentar el PROMPT_JUEZ.

Desglosar sus componentes: el rol, el contexto, la tarea y, lo más importante, el formato de salida estricto. Explicar por qué forzar a la IA a responder solo con PALABRA_CLAVE: crítica es esencial para que nuestro código pueda interpretar el resultado de forma fiable.

Realizar pruebas en vivo enviando versos de ejemplo al DialogueManager para ver las respuestas del juez.

Implementando la Máquina de Estados Híbrida (40 min):

Refactorizar la MatchScene para incluir la nueva máquina de estados: REAL_TIME, AIMING_PASS, POETRY_DUEL, WAITING_FOR_AI.

Implementar la lógica de update que detecta cuándo debe iniciarse un duelo (un jugador con balón dentro del radio de un defensor).

Implementar el handle_events para el pase dirigido con flechas, una mecánica en tiempo real.

Ambientación Artística (15 min):

Usando nuestro AudioManager, mostrar cómo cargar una lista de "cánticos literarios" y reproducirlos de forma aleatoria de fondo.

Crear una función simple para el árbitro que, al llamarla, elija una de varias rimas preescritas para señalar una falta.

Cierre y Próximos Pasos
Resumen: Hoy hemos alcanzado un nuevo pico de creatividad y complejidad técnica. Hemos diseñado un juego híbrido único que mezcla acción y estrategia por turnos, y hemos implementado un juez de IA capaz de tomar decisiones subjetivas basadas en la calidad poética. Nuestro motor ha demostrado ser una plataforma increíblemente flexible para las ideas más audaces.

Adelanto y Conclusión del Módulo: "Hemos construido el sistema de juicio y el flujo del partido. En la clase final de este módulo, nos centraremos en la experiencia del jugador. Implementaremos la ayuda de la IA como musa para los jugadores con alta Creatividad, construiremos una interfaz de usuario pulida para los duelos, y añadiremos el duelo final contra el portero para marcar un gol. Con esto, cerraremos el ciclo y tendremos un juego de 'Fútbol Poético' completo y funcional, listo para ser empaquetado."






vamos por la próxima clase


¡Excelente! Llegamos a la culminación de nuestro proyecto más creativo hasta la fecha. Es hora de pulir la experiencia, implementar las mecánicas finales y convertir nuestro prototipo de "Fútbol Poético" en un juego completo y satisfactorio.

En esta clase, nos centraremos en tres áreas clave: la ayuda de la IA como musa para el jugador, la implementación del tiro a puerta como un duelo lírico final, y la creación de una interfaz de usuario (UI) clara y atractiva para los duelos.

1. Discusión: La Experiencia del Jugador (UX/UI)
Un juego puede tener mecánicas brillantes, pero si la forma de interactuar con ellas es confusa o frustrante, el jugador lo abandonará. Una buena interfaz de usuario (UI) y experiencia de usuario (UX) son cruciales.

Claridad y Feedback: El jugador debe entender en todo momento qué está pasando, qué se espera de él y cuál es el resultado de sus acciones. Nuestra UI para el duelo debe mostrar claramente quién ataca, quién defiende, el verso del rival y las opciones de respuesta.

La Musa de la IA (Asistencia al Jugador): ¿Qué pasa si el jugador no es un poeta nato? Forzarlo a escribir desde cero puede ser intimidante. Aquí es donde la IA puede actuar como una musa inspiradora. Para los jugadores con alta Creatividad, podemos ofrecer un botón de "sugerencia" que llame a la IA con un prompt diferente para que genere una o varias líneas poéticas, dándole al jugador una base sobre la que trabajar o la opción de usarla directamente. Esto es un diseño de UX excelente: reduce la fricción sin eliminar el desafío.

Momentos de Clímax: El tiro a puerta no puede ser un duelo más. Debe sentirse como el clímax de la jugada. La interfaz puede cambiar, la música puede volverse más tensa, y el desafío poético contra el portero debe ser el más difícil de todos, utilizando su estadística de Temple.

Ejemplos en juegos conocidos:

Phoenix Wright: Ace Attorney ⚖️: Este juego es una clase magistral de UI para duelos verbales. Cuando presentas evidencia, hay animaciones y sonidos impactantes ("¡Protesto!"). La interfaz muestra claramente las contradicciones y guía al jugador. Nuestra UI del duelo se inspirará en esta claridad.

Slay the Spire 🃏: Un juego de construcción de mazos donde la UI es perfecta. Muestra claramente la "intención" del enemigo (si va a atacar o a defender), permitiendo al jugador planificar su turno. De forma similar, nuestra "musa de IA" le dará al jugador una idea de una posible "intención" poética.

Juegos con "Quick Time Events" (QTE): Aunque a veces criticados, los QTE son un ejemplo de cómo la UI se enfoca en un momento de clímax. Cuando vas a tirar a puerta, la pantalla se puede centrar, los bordes pueden oscurecerse, y toda la atención se centra en el duelo final.

2. Estructura de Archivos Actualizada
Añadimos un módulo ui_elements.py a nuestro motor para crear componentes de UI reutilizables, como botones.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- ui_elements.py       # ¡NUEVO! Para botones y otros elementos.
|
|-- soccer_rpg/
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Motor Actualizado
engine/ui_elements.py (¡NUEVO!)
Python

# engine/ui_elements.py
import pygame

class Button:
    def __init__(self, x, y, width, height, text, callback):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.callback = callback
        self.font = pygame.font.Font(None, 32)
        self.color = (100, 100, 120)
        self.hover_color = (150, 150, 170)

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.callback()

    def draw(self, surface):
        color = self.hover_color if self.rect.collidepoint(pygame.mouse.get_pos()) else self.color
        pygame.draw.rect(surface, color, self.rect, border_radius=5)
        
        text_surf = self.font.render(self.text, True, (255, 255, 255))
        text_rect = text_surf.get_rect(center=self.rect.center)
        surface.blit(text_surf, text_rect)
engine/dialogue.py (Pequeño Añadido)
Añadimos un método para la "musa".

Python

# engine/dialogue.py
# ... (clase DialogueManager existente) ...
class DialogueManager:
    # ...
    def get_poetic_suggestion(self, topic):
        """Pide a la IA una sugerencia poética sobre un tema."""
        # Usamos un modelo separado o un prompt diferente para no contaminar la conversación del juez
        try:
            # Este es un ejemplo simple, podría ser más complejo
            prompt = f"Escribe un verso poético corto y contundente sobre el siguiente tema de fútbol: '{topic}'. El verso debe ser inspirador para un jugador."
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Error en la API de sugerencia: {e}")
            return "El balón... es redondo."
4. El Código del Juego Actualizado
soccer_rpg/scenes/match_scene.py (Grandes Cambios)
Integramos la UI del duelo, la musa de IA y el tiro a puerta.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...
from engine.ui_elements import Button

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.duel_ui_elements = []
        self.setup_duel_ui()

    def setup_duel_ui(self):
        """Crea los elementos de la UI para el duelo, pero los mantiene inactivos."""
        self.duel_input_box = pygame.Rect(100, 400, 600, 40)
        self.duel_input_text = ""
        
        # Botón para la musa de IA
        self.suggestion_button = Button(300, 450, 200, 40, "Pedir Musa", self.on_click_suggestion)
        self.duel_ui_elements.append(self.suggestion_button)

    def on_click_suggestion(self):
        """Callback para el botón de la musa."""
        if self.game_state == "POETRY_DUEL" and self.active_player.team == "A": # Si es el turno del jugador humano
            attacker = self.duel_participants[0]
            if attacker.creatividad > 85: # Solo los jugadores muy creativos pueden pedir ayuda
                # El tema podría ser más dinámico
                suggestion = self.juez_ia.get_poetic_suggestion("un regate imparable")
                self.duel_input_text = suggestion
            else:
                print("Este jugador confía en su propia inspiración.")

    def handle_events(self, events):
        # ...
        if self.game_state == "POETRY_DUEL":
            for element in self.duel_ui_elements:
                element.handle_event(event)
            # ... (lógica para escribir en el input box) ...

    def initiate_poetry_duel(self, attacker, defender):
        # ...
        # Lógica para el tiro a puerta
        is_shot_on_goal = self.is_in_shooting_range(attacker)
        if is_shot_on_goal:
            self.game_state = "SHOOTOUT_DUEL"
            print("¡DUELO FINAL CONTRA EL PORTERO!")
        else:
            self.game_state = "POETRY_DUEL"
        # ...

    def draw(self, surface):
        # ... (dibujado del campo y jugadores) ...

        if self.game_state in ["POETRY_DUEL", "SHOOTOUT_DUEL"]:
            # Dibujar un fondo semitransparente para enfocar la acción
            overlay = pygame.Surface((surface.get_width(), surface.get_height()), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 180))
            surface.blit(overlay, (0, 0))
            
            # Dibujar la UI del duelo
            # ... (textos del atacante, caja de input, etc.) ...
            for element in self.duel_ui_elements:
                element.draw(surface)
Clase 28: Experiencia de Usuario y Clímax del Juego
Contenidos de la Clase
Discusión sobre UX/UI en Duelos (20 min): Iniciar con la discusión sobre la importancia de la claridad y el feedback en las interfaces de duelo, usando los ejemplos de Phoenix Wright y Slay the Spire.

La Musa de IA (35 min):

Implementar el método get_poetic_suggestion en el DialogueManager. Explicar que es buena idea usar un prompt diferente al del juez para esta tarea.

Crear la clase Button en el motor como un componente de UI reutilizable.

En MatchScene, añadir el botón "Pedir Musa". Implementar la lógica para que solo los jugadores con alta Creatividad puedan usarlo, conectando directamente las estadísticas del personaje con las mecánicas de la UI.

El Duelo Final: Tiro a Puerta (40 min):

Definir una "zona de tiro" en el campo.

En initiate_poetry_duel, comprobar si el atacante está en esa zona. Si es así, el estado del juego cambia a SHOOTOUT_DUEL y el oponente es siempre el portero rival.

Discutir cómo la resolución de este duelo final dependería del Temple de ambos jugadores, haciendo que la llamada a la IA o la lógica de resolución sea más difícil.

Pulido de la Interfaz de Duelo (25 min):

Implementar la UI final para los duelos en el método draw de MatchScene.

Mostrar cómo usar un overlay semitransparente para oscurecer el fondo y centrar la atención del jugador en el duelo.

Organizar el texto del atacante, la caja de entrada del jugador y los botones de forma clara y legible.

Cierre y Conclusión Final del Curso
Resumen General: ¡Lo hemos conseguido! A lo largo de este curso, hemos construido un motor de juegos increíblemente versátil desde cero. Hemos creado cuatro juegos completos de géneros totalmente distintos: un Arkanoid, un plataformero, un juego de físicas y, finalmente, nuestro innovador "Fútbol Poético". Hemos aprendido sobre gestión de escenas, físicas, IA de patrulla, sistemas de diálogo, y hoy, hemos culminado nuestro viaje integrando IA generativa de una forma creativa y profunda, no como un truco, sino como una mecánica central.

El Siguiente Paso es Tuyo: Has pasado de ser un estudiante a ser un diseñador y desarrollador de juegos. Tienes las herramientas, el conocimiento y, lo más importante, la experiencia para tomar cualquier idea, por extraña o ambiciosa que sea, y convertirla en una realidad jugable. El motor es tuyo para expandir, y el mundo de los videojuegos está esperando tus creaciones.

¡Enhorabuena, has completado el curso! Ha sido un placer guiarte en este viaje. Ahora, ¡ve y crea!






incluiste como generar lso cantos de la hincahcda usando l iteratura clasica?


¡Tienes toda la razón! Es una de las ideas más originales y se me pasó incluirla en el código de la última clase. Mis disculpas. Es un detalle de ambientación fantástico que merece ser implementado.

Añadir los cánticos literarios es un proceso muy similar al que usamos para el juez o la musa, pero más simple. No necesitamos una respuesta, solo generar contenido.

Aquí te muestro cómo lo integraríamos.

1. El Prompt para el "Bardo de la Grada"
Primero, necesitamos un nuevo prompt para una instancia de DialogueManager que se encargará de generar los cánticos. Este prompt le dará a la IA el rol de una hinchada increíblemente culta.

Python

# Prompt para el generador de cánticos
PROMPT_HINCHADA = """
Eres el 'Coro de los Eruditos', la hinchada de un equipo de fútbol poético. Tu propósito es alentar a tu equipo, pero en lugar de cánticos vulgares, recitas fragmentos o imitaciones del estilo de grandes autores de la literatura universal.

Tu tarea es generar un cántico corto (2 o 3 líneas) que sea apropiado para un momento de tensión o aliento en un partido de fútbol. El estilo debe ser elevado y dramático.

Ejemplo de estilo de Borges: "En el laberinto del campo, un solo hombre es el minotauro, y el balón, su hilo de Ariadna."
Ejemplo de estilo de Shakespeare: "¡Oh, valiente delantero! Que tu disparo vuele certero, pues la duda es la traidora que nos hace perder lo bueno que a menudo podríamos ganar."
Ejemplo de estilo de Cervantes: "Acometed, nobles jugadores, la aventura de este gol, que los gigantes que veis ahí no son sino molinos de viento defensivos."

Genera un nuevo cántico original con este estilo.
"""
2. Integración en el AudioManager
La forma más elegante de manejar esto es expandir nuestro AudioManager para que no solo reproduzca sonidos pregrabados, sino que también pueda gestionar una "banda sonora" de cánticos generados.

engine/audio.py (Actualizado)
Python

# engine/audio.py
import pygame
import random
from .dialogue import DialogueManager # Necesitaremos el DialogueManager aquí
from adventure.config import API_KEY # Y la API Key

class AudioManager:
    def __init__(self, num_channels=16):
        # ... (código de inicialización existente) ...
        
        # ¡NUEVO! Generador de cánticos
        self.chant_generator = DialogueManager(API_KEY, PROMPT_HINCHADA)
        self.chants = [] # Cache para guardar los cánticos generados
        self.chant_font = pygame.font.Font(None, 22)
        self.current_chant_surface = None
        self.chant_timer = 0
        self.chant_cooldown = 10 * 60 # 10 segundos (a 60 FPS)

    def update_chants(self, surface):
        """Gestiona la generación y visualización de cánticos."""
        self.chant_timer += 1
        if self.chant_timer > self.chant_cooldown:
            self.chant_timer = 0
            # Pedimos un nuevo cántico a la IA
            # Esto debería ir en un hilo para no congelar el juego
            print("Generando nuevo cántico...")
            new_chant_text = self.chant_generator.send_message("Genera un nuevo cántico de aliento.")
            self.chants.append(new_chant_text)
            
            # Creamos la superficie de texto para el último cántico
            self.current_chant_surface = self.chant_font.render(f"La grada canta: '{new_chant_text}'", True, (200, 200, 150, 200))

        # Dibujamos el cántico actual si existe
        if self.current_chant_surface:
            chant_rect = self.current_chant_surface.get_rect(centerx=surface.get_width() / 2, y=surface.get_height() - 20)
            surface.blit(self.current_chant_surface, chant_rect)

    # ... (resto de los métodos existentes: load_sound, play_sound, etc.) ...
3. Implementación en la MatchScene
Ahora, en lugar de que la escena gestione el audio, simplemente le decimos al AudioManager que se actualice en cada fotograma.

soccer_rpg/scenes/match_scene.py (Actualizado)
Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        # ...
        # El audio_manager se crea aquí
        self.audio_manager = AudioManager()
        self.sound_enabled = True # El jugador puede cambiar esto
        # ...

    def handle_events(self, events):
        # ...
        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_m: # Tecla 'M' para mutear/desmutear
                    self.sound_enabled = not self.sound_enabled
                    if not self.sound_enabled:
                        pygame.mixer.music.stop()
                    else:
                        self.audio_manager.play_music() # Asumiendo que hay música de fondo

    def update(self):
        # ... (toda la lógica de juego) ...
        pass

    def draw(self, surface):
        # ... (dibujado del campo, jugadores, UI del duelo, etc.) ...
        
        # Al final del dibujado, actualizamos y dibujamos los cánticos
        if self.sound_enabled:
            self.audio_manager.update_chants(surface)
Resumen de la Implementación
Rol Definido: Creamos un PROMPT_HINCHADA que le da a la IA una personalidad y tarea muy específicas: ser una afición literaria.

Gestor Centralizado: Modificamos el AudioManager para que contenga una instancia del DialogueManager. Ahora, el AudioManager es responsable de todo el paisaje sonoro, incluyendo estos cánticos generados.

Generación Periódica: En el update_chants del AudioManager, usamos un temporizador. Cada cierto tiempo (ej. 10-15 segundos), se hace una llamada a la API para generar un nuevo cántico.

Nota importante: Como se menciona en el código, en un juego real, esta llamada a la API (que puede tardar 1-2 segundos) se haría en un hilo de ejecución secundario (threading) para no congelar el juego. Para nuestro curso, la implementación directa es suficiente para demostrar el concepto.

Visualización Sutil: El cántico generado se renderiza como un texto que aparece en la parte inferior de la pantalla, como si fuera un subtítulo del ambiente, sin ser intrusivo.

Control del Jugador: Añadimos una simple tecla (M) que permite al jugador activar o desactivar toda la ambientación sonora (música y cánticos), respetando sus preferencias.

Con esta adición, el juego adquiere una capa de atmósfera increíblemente única y memorable. La idea de que los cánticos de la grada sean generados por IA y basados en literatura clásica es exactamente el tipo de detalle que hace que un juego indie destaque. ¡Excelente sugerencia!






yo quiero que la hinchada cante, como podemos epandir e motor para que tome el texto generado y lo convierta en cantico?



Ver razonamiento
¡Excelente pregunta! Darle voz a la hinchada es el siguiente paso lógico para una inmersión total. Esto nos adentra en el fascinante mundo de la síntesis de voz (Text-to-Speech o TTS).

Vamos a expandir nuestro motor para que no solo genere el texto del cántico, sino que lo convierta en un archivo de audio y lo reproduzca durante el partido, dándole una voz real a nuestro "Coro de los Eruditos".

1. Discusión: Dando Voz a la IA - Text-to-Speech en Juegos
La síntesis de voz es la tecnología que convierte texto escrito en habla audible. En los videojuegos, tiene aplicaciones muy poderosas:

Accesibilidad: Es una herramienta fundamental para que jugadores con discapacidad visual puedan disfrutar de juegos con mucho texto.

Prototipado Rápido: Los desarrolladores pueden usar TTS para escuchar los diálogos en el contexto del juego antes de contratar y grabar a actores de voz, lo cual es muy costoso.

Narración Dinámica: Para juegos con texto generado proceduralmente (como el nuestro), el TTS es la única forma de darle voz a un contenido que no existe hasta que se crea.

Diseño de Personajes: Una voz robótica o sintetizada puede ser una elección de diseño para personajes que son androides, inteligencias artificiales o entidades sobrenaturales.

Ejemplos en juegos conocidos:

Portal 2 🤖: Las voces icónicas de GLaDOS y Wheatley, aunque interpretadas por actores, se procesaron masivamente para sonar como voces de IA, inspirándose en la cadencia y el timbre del TTS para crear su personalidad.

System Shock 👾: La voz de la malévola IA SHODAN es uno de los personajes más memorables de la historia de los videojuegos. Su voz entrecortada, con fallos y un tono sintético, es puro diseño de sonido basado en los principios del TTS para crear una sensación de terror tecnológico.

Moonbase Alpha 🧑‍🚀: Este juego de la NASA se hizo viral por su sistema de chat con TTS, que producía frases hilarantes y robóticas ("John Madden! aeiou"). Demuestra cómo incluso un TTS básico puede crear una experiencia memorable.

Nuestra Herramienta: gTTS (Google Text-to-Speech)

Usaremos la librería gTTS de Python. Es muy fácil de usar y utiliza el potente motor de síntesis de voz de Google para generar archivos de audio de alta calidad.

2. Estructura de Archivos y Dependencias Actualizadas
Añadimos la nueva librería a nuestros requerimientos. La estructura de archivos no necesita cambiar.

requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
google-generativeai
gTTS
(No olvides ejecutar pip install -r requirements.txt o pip install gTTS)

3. El Código del Motor Actualizado
Aquí viene el cambio más importante y conceptualmente más avanzado que hemos hecho. Las llamadas a internet (tanto para Gemini como para gTTS) son lentas. Si las hacemos en el bucle principal, el juego se congelará durante varios segundos. Para evitarlo, debemos ejecutarlas en un hilo de ejecución secundario (threading).

engine/audio.py (Grandes Cambios)
Python

# engine/audio.py
import pygame
import random
import threading # ¡Importamos la librería para hilos!
import os
from gtts import gTTS # ¡Importamos gTTS!
from .dialogue import DialogueManager
from adventure.config import API_KEY

class AudioManager:
    def __init__(self, num_channels=16):
        # ... (código de inicialización existente) ...
        self.chant_generator = DialogueManager(API_KEY, PROMPT_HINCHADA)
        self.chant_cooldown = 15 * 60 # 15 segundos
        self.chant_timer = self.chant_cooldown # Para que genere uno al empezar
        
        # Variables para manejar el hilo
        self.chant_thread = None
        self.chant_audio_path = "temp_chant.mp3"
        self.new_chant_ready = False

    def _generate_and_synthesize_chant(self):
        """
        Esta función se ejecutará en un hilo separado.
        Contiene las operaciones lentas (llamadas a internet y escritura de archivo).
        """
        print("Hilo de cánticos: Generando texto con IA...")
        try:
            # 1. Generar el texto del cántico
            chant_text = self.chant_generator.send_message("Genera un nuevo cántico de aliento.")
            if not chant_text: return

            print(f"Hilo de cánticos: Texto recibido - '{chant_text[:30]}...'")
            
            # 2. Sintetizar el texto a un archivo MP3
            tts = gTTS(text=chant_text, lang='es', slow=False)
            tts.save(self.chant_audio_path)
            
            print(f"Hilo de cánticos: Archivo de audio '{self.chant_audio_path}' guardado.")
            self.new_chant_ready = True # Marcamos que el nuevo cántico está listo para ser reproducido
        except Exception as e:
            print(f"Error en el hilo de cánticos: {e}")

    def update_chants(self):
        """Gestiona el ciclo de vida de los cánticos."""
        self.chant_timer += 1

        # Si hay un cántico nuevo listo Y la música no está sonando, lo reproducimos
        if self.new_chant_ready and not pygame.mixer.music.get_busy():
            print("Reproduciendo nuevo cántico...")
            try:
                pygame.mixer.music.load(self.chant_audio_path)
                pygame.mixer.music.play()
                self.new_chant_ready = False
                # Podríamos eliminar el archivo aquí si quisiéramos
                # os.remove(self.chant_audio_path)
            except pygame.error as e:
                print(f"No se pudo reproducir el cántico: {e}")

        # Si ha pasado el tiempo de espera Y el hilo anterior ha terminado, iniciamos uno nuevo
        if self.chant_timer > self.chant_cooldown and (self.chant_thread is None or not self.chant_thread.is_alive()):
            self.chant_timer = 0
            # Creamos y lanzamos el hilo
            self.chant_thread = threading.Thread(target=self._generate_and_synthesize_chant)
            self.chant_thread.start()

    # ... (resto de métodos sin cambios) ...

soccer_rpg/scenes/match_scene.py (Mínimos Cambios)
Gracias a nuestro motor bien diseñado, la escena del juego apenas necesita saber que esto está ocurriendo.

Python

# soccer_rpg/scenes/match_scene.py
# ...

class MatchScene(Scene):
    # ...
    def update(self):
        # ... (lógica de juego existente) ...
        
        # Simplemente le decimos al gestor de audio que se actualice
        if self.sound_enabled:
            self.audio_manager.update_chants()
    
    def draw(self, surface):
        # Ya no necesitamos dibujar el texto del cántico aquí. El audio lo es todo.
        # ... (código de dibujado existente) ...
        pass
Clase 29: El Sonido de la IA - Síntesis de Voz y Concurrencia
Contenidos de la Clase
Discusión sobre Voz Sintética (20 min): Iniciar con la discusión sobre TTS, sus aplicaciones en juegos y los ejemplos mencionados, desde la accesibilidad hasta el diseño de personajes icónicos.

Introducción a gTTS (25 min):

Instalar la librería (pip install gTTS).

Mostrar un ejemplo simple y aislado: escribir un script corto que tome un texto, lo guarde como hola.mp3 y luego lo reproduzca con Pygame. Esto demuestra el flujo básico.

El Problema de la Concurrencia (35 min):

Ejecutar una llamada a la API de Gemini o a gTTS.save() directamente en el bucle update y mostrar cómo el juego se congela por completo.

Explicar por qué ocurre esto: el bucle principal debe esperar a que termine la lenta operación de red/disco.

Introducir threading como la solución. Explicar el concepto de un hilo de ejecución como un "mini-programa" que corre en paralelo al bucle principal sin bloquearlo.

Implementando el AudioManager Asíncrono (40 min):

Refactorizar el AudioManager paso a paso para crear la nueva lógica.

Crear la función _generate_and_synthesize_chant que contiene las operaciones lentas.

En update_chants, implementar el sistema de control:

Comprobar el cooldown.

Comprobar si el hilo anterior ha terminado (is_alive()).

Lanzar el nuevo hilo.

Comprobar si hay un nuevo archivo de audio listo para reproducir (new_chant_ready).

Cierre y Próximos Pasos
Resumen: Hemos dado un paso gigantesco en la complejidad y profesionalidad de nuestro motor. No solo generamos contenido con IA, sino que le damos voz en tiempo real sin sacrificar el rendimiento del juego. Hemos aprendido sobre síntesis de voz y hemos implementado una solución de concurrencia con hilos, una técnica avanzada indispensable para cualquier aplicación moderna.

Adelanto Final del Curso: "Con la capacidad de crear diálogos, juzgar poesía y ahora generar audio dinámicamente, nuestro motor está completo. La última etapa de nuestro viaje será la conclusión de los proyectos. En las clases finales, nos enfocaremos en pulir la jugabilidad de nuestros tres juegos, añadiendo los menús finales, pantallas de victoria/derrota, y empaquetando cada uno de ellos para que tengas un portafolio de tres juegos únicos y terminados, listos para mostrar al mundo."







avancemos!


¡Vamos allá! Es hora de atar todos los cabos sueltos y llevar nuestros proyectos a la línea de meta.

En esta clase final, nos enfocaremos en la conclusión y el pulido. El objetivo es tomar los prototipos funcionales que hemos construido y añadirles las características que los convierten en experiencias de juego completas: menús, condiciones de victoria claras, y un flujo de juego que guíe al jugador de principio a fin.

1. Discusión: ¿Qué Significa "Terminar" un Juego?
Un juego está "terminado" no cuando se le añaden todas las características posibles, sino cuando el bucle de jugabilidad principal es sólido y satisfactorio y el jugador puede experimentar el juego completo sin interrupciones ni callejones sin salida.

El Bucle de Jugabilidad (Gameplay Loop): Es la secuencia de acciones que el jugador repite.

En nuestro Plataformero: Saltar y moverse -> Recoger monedas -> Evitar enemigos -> Llegar al final del nivel.

En nuestro Fútbol Poético: Moverse en tiempo real -> Iniciar un duelo -> Resolver el duelo -> Ver la consecuencia táctica -> Repetir hasta el gol.

La Estructura de la Experiencia (Player Journey): ¿Cómo guiamos al jugador?

Menú Principal: La puerta de entrada. Debe ser claro y ofrecer las opciones básicas ("Jugar", "Salir").

El Juego en Sí: El núcleo de la experiencia.

Condiciones de Fin: Debe haber una forma clara de ganar (llegar al final, marcar 3 goles) o perder (quedarse sin vidas, recibir 3 goles).

Pantalla Final: Una pantalla de "Victoria" o "Game Over" que da cierre a la sesión y, idealmente, ofrece la opción de volver al menú para jugar de nuevo.

Ejemplos en juegos conocidos:

Cualquier juego de Arcade clásico (Pac-Man, Donkey Kong): Su estructura es la definición de un bucle completo. Tienen una pantalla de título (Menú), el juego principal donde el objetivo es claro (comer todos los puntos, salvar a la princesa), condiciones de fin claras (perder todas las vidas), y una pantalla de "Game Over" con la puntuación, que te incita a volver a intentarlo.

Journey 🏜️: Un ejemplo magistral de viaje del jugador. Comienza sin menús, directamente en el juego. El viaje tiene un principio, un desarrollo y un final narrativo muy claro y emotivo. Al terminar, vuelve suavemente a la pantalla de inicio, invitando a la rejugabilidad.

2. Estructura y Código: Aplicando el Pulido
No añadiremos nuevos módulos al motor. En su lugar, aplicaremos los que ya tenemos (SceneManager, UI, Button) a nuestros proyectos para construir las piezas que faltan.

Proyecto 1: Plataformero - La Meta Final
Objetivo: Añadir una "zona de victoria" al final del nivel.

platformer/entities.py (¡NUEVO!)
Python

# platformer/entities.py
from engine.game_object import GameObject

# ... (clases Player y Platform) ...

class Goal(GameObject):
    """Un objeto que representa el final del nivel."""
    def __init__(self, x, y):
        # Podríamos usar una imagen de una bandera, un portal, etc.
        super().__init__(x, y, 'platformer/assets/goal.png')
platformer/scenes/game_scene.py (Actualizado)
Python

# platformer/scenes/game_scene.py
# ...
from ..entities import Goal
from .victory_scene import VictoryScene # Necesitaremos crear esta escena

class GameScene(Scene):
    def __init__(self, scene_manager):
        # ...
        # Al final de la carga del nivel, añadimos la meta
        self.goal = Goal(1500, 352) # Ponerlo al final del nivel
        self.todos_los_sprites.add(self.goal)
        # ...

    def update(self):
        # ...
        # Comprobar si el jugador ha llegado a la meta
        if pygame.sprite.collide_rect(self.jugador, self.goal):
            self.scene_manager.go_to(VictoryScene, score=self.puntuacion)
Proyecto 2: Fútbol Poético - El Marcador y el Gol
Objetivo: Implementar un marcador, detectar goles y terminar el partido.

soccer_rpg/scenes/match_scene.py (Actualizado)
Python

# soccer_rpg/scenes/match_scene.py
# ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.score = {"A": 0, "B": 0}
        self.goal_a_rect = pygame.Rect(0, 250, 50, 100) # Portería izquierda
        self.goal_b_rect = pygame.Rect(750, 250, 50, 100) # Portería derecha
        # ...

    def update(self):
        # ...
        # En la fase de tiempo real, comprobar si el balón entra en una portería
        if self.ball.rect.colliderect(self.goal_a_rect):
            self.score["B"] += 1
            print("¡GOL DEL EQUIPO B!")
            self.reset_positions_after_goal()
        elif self.ball.rect.colliderect(self.goal_b_rect):
            self.score["A"] += 1
            print("¡GOL DEL EQUIPO A!")
            self.reset_positions_after_goal()
        
        # Comprobar condición de victoria
        if self.score["A"] >= 3 or self.score["B"] >= 3:
            # self.scene_manager.go_to(VictoryScene, score=self.score)
            pass
            
    def reset_positions_after_goal(self):
        """Reinicia la posición de los jugadores y el balón al centro."""
        # ... (lógica para recolocar a todos los jugadores y al balón) ...
        self.game_state = "REAL_TIME"

    def draw(self, surface):
        # ...
        # Usar nuestra UI del motor para dibujar el marcador
        # self.ui.draw_score(surface, self.score)
        pass
Clase 30: Conclusión de Proyectos y Pulido Final
Contenidos de la Clase
Discusión sobre el "Cierre" (20 min): Iniciar con la discusión sobre la importancia de tener un bucle de jugabilidad y un viaje del jugador completos. Usar los ejemplos de juegos arcade y Journey para ilustrar el concepto.

Implementando Menús y Escenas Finales (35 min):

Crear escenas genéricas de "Menú Principal" y "Pantalla de Victoria/Derrota" usando los componentes de nuestro motor (Scene, Button, UI).

Repasar el SceneManager y cómo se usa para conectar el flujo: Menú -> Juego -> Pantalla Final -> Menú.

Pulido del Plataformero (30 min):

Añadir la entidad Goal al final del nivel del plataformero.

Implementar la lógica de colisión en GameScene que detecta cuando el jugador toca la meta y transiciona a la pantalla de victoria, pasándole la puntuación final.

Pulido del Fútbol Poético (35 min):

Implementar el marcador y las porterías como pygame.Rect.

En el update de la MatchScene, añadir la lógica que comprueba si el balón colisiona con una de las porterías.

Si hay gol, actualizar el marcador, llamar a una función reset_positions_after_goal, y continuar el partido.

Añadir una condición de victoria (ej: el primero que marque 3 goles) que transicione a una pantalla final.

Cierre Final y Despedida del Curso
Resumen Definitivo: Hemos llegado al final de nuestro increíble viaje. No solo hemos aprendido a programar, sino que hemos aprendido a pensar como diseñadores de juegos. Hemos construido un motor robusto y, lo que es más importante, lo hemos utilizado para crear y terminar múltiples juegos, cada uno con su propia personalidad y mecánicas.

Tu Portafolio: Ahora tienes un portafolio tangible con tres proyectos únicos (Plataformero, Fútbol Poético, y el juego de Físicas/Billar). Cada uno demuestra un conjunto diferente de habilidades técnicas y creativas. Tienes el conocimiento para empaquetarlos con PyInstaller y mostrarlos con orgullo.

El Futuro es Tuyo: El desarrollo de juegos es un viaje de aprendizaje constante. Has construido una base sólida que te permitirá explorar cualquier género, cualquier mecánica y cualquier idea que se te ocurra. Has aprendido a aprender.















Clase 11: Diseño de Sonido y Gestión de Audio
Contenidos de la Clase
Discusión sobre Diseño de Sonido (20 min): Iniciar con la discusión y los ejemplos de juegos mencionados al principio. Enfatizar cómo el sonido guía, recompensa y advierte al jugador.

Creación del AudioManager (35 min):

Presentar la nueva clase AudioManager en el motor.

Explicar la importancia de pygame.mixer.pre_init para reducir la latencia del audio.

Detallar la lógica de la carga centralizada en un diccionario (self.sounds).

Explicar el concepto de canales de audio. Demostrar cómo pygame.mixer.set_num_channels y pygame.mixer.Channel(n) nos permite asignar tipos de sonidos a canales específicos para que una explosión no corte el sonido de un salto.

Integración en Arkanoid (35 min):

Refactorizar el archivo main.py para usar el nuevo gestor.

Crear una sección de "Carga de Assets" donde se llamará a audio_manager.load_sound() para todos los efectos de sonido del juego.

Recorrer el código y reemplazar todas las llamadas a .play() por audio_manager.play_sound('nombre_sonido', 'canal_apropiado').

Añadir sonidos a interacciones que antes eran silenciosas, como el rebote de la pelota en las paredes o la activación de un power-up.

Control de Volumen (20 min):

Mostrar cómo implementar y usar los métodos set_sfx_volume() y set_music_volume() en el AudioManager.

Como ejemplo práctico, implementar una funcionalidad simple para que las teclas + y - del teclado numérico suban y bajen el volumen general de los efectos de sonido.

Cierre y Próximos Pasos
Resumen: Hemos profesionalizado completamente el audio de nuestro juego. Con el AudioManager en nuestro motor, ahora tenemos un sistema centralizado, robusto y expandible para manejar música y efectos de sonido, controlar volúmenes y organizar la reproducción en canales. El juego ahora "habla" con el jugador de una forma mucho más rica.

Adelanto de la Clase 12: "Nuestro juego está casi completo, pero le falta algo fundamental para que otros puedan jugarlo: un menú principal y un sistema de guardado de puntuaciones. En la próxima clase, construiremos una escena de menú de inicio con opciones para 'Jugar' y 'Salir'. Además, aprenderemos a guardar la puntuación más alta en un archivo para que persista entre sesiones de juego, ¡dando a los jugadores un motivo para volver y superar su propio récord!"






Llegamos a una etapa crucial que transforma nuestro proyecto de un "juego que se ejecuta" a una "aplicación de juego completa".

En esta clase, construiremos la estructura para manejar diferentes escenas (como un menú principal y la escena de juego), y aprenderemos a guardar datos, como la puntuación más alta, para que persistan entre sesiones. Esto le da al jugador un objetivo a largo plazo: ¡superar su propio récord!

1. Discusión: La Máquina de Estados Finitos (Escenas)
¿Cómo hace un juego para mostrar un menú, luego el juego, luego una pantalla de opciones y volver al menú? La respuesta es una Máquina de Estados Finitos (Finite State Machine). No te asustes por el nombre, el concepto es simple:

El juego solo puede estar en un estado (o escena) a la vez.

Cada escena es responsable de su propia lógica, dibujado y eventos.

Existe un gestor de escenas que controla qué escena está activa y maneja la transición de una a otra.

Ejemplos en juegos conocidos:

Stardew Valley 🚜: Tiene decenas de escenas. El menú principal es una escena. El juego en tu granja es otra. Estar dentro de una tienda es una escena diferente, al igual que los festivales. El gestor de escenas se asegura de que cuando sales de la tienda, vuelvas exactamente al lugar donde estabas en la granja.

Cualquier juego de Lucha (Street Fighter, Mortal Kombat) 🥊: La pantalla de selección de personaje es una escena. El combate en sí es otra. La pantalla de "You Win/Lose" es una tercera. La transición entre ellas es lo que crea el flujo del juego.

Nuestro Arkanoid: Hasta ahora, hemos manejado estados ('jugando', 'game_over') con un if/elif. Esto funciona para una sola pantalla, pero se vuelve caótico si queremos añadir un menú. Vamos a profesionalizarlo creando un sistema de escenas en nuestro motor.

2. Estructura de Archivos Actualizada
Añadimos un archivo scene.py al motor y una carpeta data para guardar la puntuación.

/curso_pygame/
|
|-- engine/
|   |-- __init__.py
|   |-- game_object.py
|   |-- ui.py
|   |-- particle.py
|   |-- audio.py
|   |-- scene.py             # ¡NUEVO! Nuestro gestor de escenas.
|
|-- arkanoid/
|   |-- data/                # ¡NUEVO!
|   |   |-- scores.json      # Archivo para guardar la puntuación.
|   |
|   |-- scenes/              # ¡NUEVO! Organizamos nuestras escenas.
|   |   |-- __init__.py
|   |   |-- menu_scene.py
|   |   |-- game_scene.py
|   |
|   |-- main.py              # Ahora será mucho más simple.
|   |-- ... (el resto sin cambios)
|
|-- requirements.txt
3. El Código del Motor Actualizado
Este es el núcleo de nuestro nuevo sistema de flujo de juego.

engine/scene.py (¡NUEVO!)
Python

# engine/scene.py

class Scene:
    """Clase base para todas las escenas del juego."""
    def __init__(self, scene_manager):
        self.scene_manager = scene_manager

    def handle_events(self, events):
        """Maneja los eventos de la escena."""
        pass

    def update(self):
        """Actualiza la lógica de la escena."""
        pass

    def draw(self, surface):
        """Dibuja la escena en la superficie."""
        pass

class SceneManager:
    def __init__(self, initial_scene_class, screen):
        self.screen = screen
        self.go_to(initial_scene_class)

    def go_to(self, scene_class, **kwargs):
        """Transiciona a una nueva escena."""
        self.scene = scene_class(self, **kwargs)

    def handle_events(self, events):
        self.scene.handle_events(events)

    def update(self):
        self.scene.update()

    def draw(self):
        self.scene.draw(self.screen)
4. El Código del Juego Arkanoid Actualizado
Dividimos nuestro antiguo main.py en escenas específicas.

arkanoid/scenes/menu_scene.py (¡NUEVO!)
Python

# arkanoid/scenes/menu_scene.py
import pygame
from engine.scene import Scene
from .game_scene import GameScene # Importamos la escena de juego para la transición

class MenuScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        self.font_title = pygame.font.Font(None, 80)
        self.font_options = pygame.font.Font(None, 50)
        self.title = self.font_title.render("Arkanoid", True, (255, 255, 0))
        self.play_text = self.font_options.render("Presiona ESPACIO para Jugar", True, (255, 255, 255))
        self.quit_text = self.font_options.render("Presiona ESC para Salir", True, (255, 255, 255))

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    # Transicionamos a la escena del juego
                    self.scene_manager.go_to(GameScene)
                elif event.key == pygame.K_ESCAPE:
                    pygame.event.post(pygame.event.Event(pygame.QUIT))

    def draw(self, surface):
        surface.fill((10, 10, 20))
        surface.blit(self.title, self.title.get_rect(centerx=surface.get_width()/2, y=150))
        surface.blit(self.play_text, self.play_text.get_rect(centerx=surface.get_width()/2, y=350))
        surface.blit(self.quit_text, self.quit_text.get_rect(centerx=surface.get_width()/2, y=420))
arkanoid/scenes/game_scene.py (¡NUEVO!)
Este archivo ahora contiene toda la lógica que antes estaba en main.py.

Python

# arkanoid/scenes/game_scene.py
import pygame
import json
from engine.scene import Scene
# ... (imports de entidades, ui, audio_manager, etc.)

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        # Aquí va TODA la inicialización que antes estaba en main():
        # Cargar audio, UI, variables de juego (vidas, puntuación),
        # crear grupos de sprites, cargar el primer nivel, etc.
        self.load_high_score()

    def handle_events(self, events):
        # Aquí va el bucle de eventos que estaba en main()
        # (solo los eventos relevantes para el juego, como el reinicio)
        pass

    def update(self):
        # Aquí va toda la lógica de la sección UPDATE de main()
        # (actualizar sprites, colisiones, etc.)
        # Si el juego termina (game over o victoria), guardamos la puntuación
        if self.estado_juego in ['game_over', 'victoria']:
            self.save_high_score()
            # Podríamos transicionar a una escena de "Game Over" o de vuelta al menú
            # from .menu_scene import MenuScene
            # self.scene_manager.go_to(MenuScene)

    def draw(self, surface):
        # Aquí va toda la lógica de la sección DRAW de main()
        pass

    def save_high_score(self):
        if self.puntuacion > self.high_score:
            self.high_score = self.puntuacion
            with open('arkanoid/data/scores.json', 'w') as f:
                json.dump({'high_score': self.high_score}, f)

    def load_high_score(self):
        try:
            with open('arkanoid/data/scores.json', 'r') as f:
                data = json.load(f)
                self.high_score = data.get('high_score', 0)
        except (FileNotFoundError, json.JSONDecodeError):
            self.high_score = 0
(Nota: El código de GameScene es una refactorización directa del main.py de la clase anterior, organizado en los métodos __init__, handle_events, update y draw).

arkanoid/main.py (Simplificado)
El punto de entrada ahora es muy simple. Su única responsabilidad es iniciar Pygame, crear el gestor de escenas y correr el bucle principal.

Python

# arkanoid/main.py
import pygame
from engine.scene import SceneManager
from arkanoid.scenes.menu_scene import MenuScene

ANCHO, ALTO = 800, 600

def main():
    pygame.init()
    pantalla = pygame.display.set_mode((ANCHO, ALTO))
    pygame.display.set_caption("Arkanoid")
    reloj = pygame.time.Clock()

    scene_manager = SceneManager(MenuScene, pantalla)

    ejecutando = True
    while ejecutando:
        eventos = pygame.event.get()
        for evento in eventos:
            if evento.type == pygame.QUIT:
                ejecutando = False

        scene_manager.handle_events(eventos)
        scene_manager.update()
        scene_manager.draw()

        pygame.display.flip()
        reloj.tick(60)

    pygame.quit()

if __name__ == '__main__':
    main()
Clase 12: Menús, Escenas y Persistencia de Datos
Contenidos de la Clase
Discusión sobre Máquinas de Estado : Iniciar con la discusión y los ejemplos de juegos. Explicar por qué el sistema de if/elif se vuelve insostenible y cómo un sistema de escenas soluciona este problema de escalabilidad.

El Gestor de Escenas (30 min):

Presentar las clases Scene y SceneManager del motor. Scene es el "contrato" que toda escena debe cumplir (tener métodos handle_events, update, draw). SceneManager es el director de orquesta.

Mostrar cómo main.py se simplifica drásticamente, delegando toda la responsabilidad al gestor.

Creando la Escena de Menú (30 min):

Implementar la clase MenuScene. Es una escena simple que solo dibuja texto y escucha eventos de teclado para transicionar a otra escena (GameScene) o para salir del juego.

Explicar cómo se usa self.scene_manager.go_to(GameScene) para realizar la transición.

Persistencia de Datos con JSON :

Explicar qué es JSON (JavaScript Object Notation): un formato de texto ligero y legible para intercambiar datos. Es ideal para configuraciones o datos simples.

Implementar los métodos save_high_score y load_high_score en GameScene.

Detallar el uso de try/except en load_high_score para manejar el caso en que el archivo aún no exista la primera vez que se juega.

Cierre y Próximos Pasos
Resumen: ¡Hemos dado el salto final para convertir nuestro proyecto en una aplicación robusta! Nuestro motor ahora incluye un gestor de escenas que nos permite crear flujos de juego complejos (menús, opciones, juego, etc.) de forma limpia y ordenada. Además, hemos implementado la persistencia de datos, una característica esencial para cualquier juego que quiera ofrecer rejugabilidad y un sentido de logro.

Adelanto de la Clase 13: "Nuestro motor es potente, pero está muy enfocado en Arkanoid. ¿Cómo lo usaríamos para crear un juego completamente diferente, como un plataformero con scroll? En la próxima clase, comenzaremos un nuevo proyecto de juego. Crearemos un personaje que pueda saltar y moverse en un mundo más grande que la pantalla. Esto nos obligará a añadir a nuestro motor un sistema de cámara (scrolling) y una simulación de físicas básicas (gravedad)."




## Clase 13

Dejamos atrás el Arkanoid para aventurarnos en uno de los géneros más icónicos de los videojuegos: el plataformero. Esto nos obligará a añadir dos componentes fundamentales a nuestro motor: una simulación de físicas (gravedad y salto) y una cámara con scrolling para crear mundos más grandes que la pantalla.

1. Discusión: El "Feel" de un Plataformero
Antes de programar, ¿qué hace que saltar y correr en un juego se sienta bien? No es solo la lógica, es una serie de "trucos" de diseño que mejoran la experiencia del jugador.

Control en el aire: ¿Puede el jugador cambiar de dirección en mitad del salto? Juegos como Super Mario Bros. permiten un alto grado de control aéreo, haciéndolos más flexibles. Otros, como el Castlevania original, tienen un salto fijo y comprometido, exigiendo más precisión.

Peso y Gravedad: Una gravedad alta hace que el personaje se sienta pesado y los saltos sean cortos y rápidos. Una gravedad baja da una sensación de estar en la luna, con saltos largos y flotantes. El "feel" de Hollow Knight es muy diferente al de Super Meat Boy principalmente por esto.

"Coyote Time" ⏳: Este es un truco famoso. En la mayoría de los juegos, el jugador puede saltar durante una fracción de segundo después de haber salido de una plataforma. Se llama "tiempo de coyote" por el personaje del Correcaminos, que solo caía cuando se daba cuenta de que no había suelo. Esto hace que el juego se sienta más justo y que los saltos al borde de una plataforma no fallen por un píxel.

"Jump Buffering" ⌨️: Es lo contrario al "coyote time". El juego "recuerda" si el jugador pulsó el botón de salto un instante antes de tocar el suelo. Si lo hizo, el personaje salta automáticamente en cuanto aterriza. Esto evita la frustración de sentir que "¡le di al botón a tiempo!". Celeste es un maestro en el uso de estas técnicas para crear controles increíblemente responsivos.

Implementaremos las bases de la física, y estas técnicas avanzadas son el siguiente paso natural para pulir la jugabilidad.

2. Estructura de Archivos del Nuevo Proyecto
Creamos una nueva carpeta platformer al mismo nivel que arkanoid. El motor engine es compartido por ambos.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- camera.py            # ¡NUEVO! Nuestra cámara con scroll.
|
|-- arkanoid/                # Nuestro juego anterior sigue intacto.
|
|-- platformer/              # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- player.png
|   |   |-- ground_tile.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- game_scene.py
|   |
|   |-- main.py              # Punto de entrada para el plataformero.
|
|-- requirements.txt
3. El Código del Motor Actualizado
Añadimos uno de los componentes más importantes para juegos con mundos grandes.

engine/camera.py (¡NUEVO!)
Esta clase seguirá al jugador y gestionará qué porción del mundo se dibuja en la pantalla.

Python

# engine/camera.py
import pygame

class Camera:
    def __init__(self, width, height):
        self.camera_rect = pygame.Rect(0, 0, width, height)
        self.width = width
        self.height = height

    def apply(self, entity):
        """Aplica el desplazamiento de la cámara a la posición de una entidad."""
        return entity.rect.move(self.camera_rect.topleft)

    def update(self, target):
        """Actualiza la posición de la cámara para centrarse en el objetivo."""
        # El negativo de la posición del objetivo, más la mitad de la pantalla
        # para centrarlo.
        x = -target.rect.centerx + int(self.width / 2)
        y = -target.rect.centery + int(self.height / 2)

        # Aquí se podrían añadir límites para que la cámara no se salga del nivel.

        self.camera_rect = pygame.Rect(x, y, self.width, self.height)
La Lógica Clave: La cámara no mueve el mundo. El mundo está en coordenadas fijas. La cámara simplemente calcula un offset (desplazamiento). Al dibujar, cada objeto se desplaza por ese offset. Si la cámara se mueve a la derecha (x positivo), el mundo se dibuja desplazado a la izquierda (x negativo), creando la ilusión de movimiento.

4. El Código del Juego Plataformero
platformer/entities.py (¡NUEVO!)
Aquí definimos a nuestro jugador con físicas y a las plataformas.

Python

# platformer/entities.py
import pygame
from engine.game_object import GameObject

class Player(GameObject):
    def __init__(self, x, y):
        super().__init__(x, y, 'platformer/assets/player.png')
        self.velocidad = 5
        self.fuerza_salto = 12
        self.gravedad = 0.5
        
        self.vel_y = 0
        self.en_suelo = False

    def update(self, plataformas):
        # Movimiento horizontal
        teclas = pygame.key.get_pressed()
        if teclas[pygame.K_LEFT]:
            self.rect.x -= self.velocidad
        if teclas[pygame.K_RIGHT]:
            self.rect.x += self.velocidad

        # Aplicar gravedad
        if not self.en_suelo:
            self.vel_y += self.gravedad
        
        # Salto
        if teclas[pygame.K_SPACE] and self.en_suelo:
            self.vel_y = -self.fuerza_salto
        
        # Colisiones verticales
        self.en_suelo = False
        self.rect.y += self.vel_y
        colisiones_y = pygame.sprite.spritecollide(self, plataformas, False)
        for plat in colisiones_y:
            if self.vel_y > 0: # Cayendo
                self.rect.bottom = plat.rect.top
                self.en_suelo = True
                self.vel_y = 0
            elif self.vel_y < 0: # Saltando
                self.rect.top = plat.rect.bottom
                self.vel_y = 0

class Platform(GameObject):
    def __init__(self, x, y, image_path='platformer/assets/ground_tile.png'):
        super().__init__(x, y, image_path)
platformer/scenes/game_scene.py (¡NUEVO!)
Esta escena construye el nivel y orquesta el uso de la cámara.

Python

# platformer/scenes/game_scene.py
import pygame
from engine.scene import Scene
from engine.camera import Camera
from ..entities import Player, Platform

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        self.todos_los_sprites = pygame.sprite.Group()
        self.plataformas = pygame.sprite.Group()

        self.jugador = Player(100, 200)
        
        # Creación de un nivel simple
        # Suelo
        for i in range(20):
            p = Platform(i * 50, 400)
            self.plataformas.add(p)
        # Plataformas flotantes
        p1 = Platform(300, 300)
        p2 = Platform(400, 200)
        self.plataformas.add(p1, p2)
        
        self.todos_los_sprites.add(self.jugador, self.plataformas)
        
        self.camara = Camera(800, 600)

    def update(self):
        self.jugador.update(self.plataformas)
        self.camara.update(self.jugador)

    def draw(self, surface):
        surface.fill((135, 206, 235)) # Color cielo
        for sprite in self.todos_los_sprites:
            surface.blit(sprite.image, self.camara.apply(sprite))
platformer/main.py (Punto de Entrada)
Similar al de Arkanoid, pero inicia la escena de juego del plataformero.

Python

# platformer/main.py
import pygame
from engine.scene import SceneManager
from platformer.scenes.game_scene import GameScene

ANCHO, ALTO = 800, 600

def main():
    # ... (código de inicialización de pygame y bucle principal) ...
    # Es idéntico al main de Arkanoid, pero con una escena inicial diferente.
    scene_manager = SceneManager(GameScene, pantalla)
    # ... (resto del bucle) ...

if __name__ == '__main__':
    main()´


## Clase 13: Nuevo Género - Físicas de Plataformas y Scrolling
Contenidos de la Clase
Discusión sobre Plataformeros (20 min): Iniciar con la discusión sobre el "feel", el control aéreo, gravedad y los trucos como el "Coyote Time", usando los ejemplos de juegos mencionados.

Físicas de Salto y Gravedad (35 min):

Implementar la lógica dentro de la nueva clase Player.

Explicar el ciclo: aplicar gravedad constantemente, resetear la velocidad vertical al tocar el suelo, y aplicar una fuerza negativa (salto) solo si se está en el suelo.

Detallar la importancia de separar el manejo de colisiones en el eje Y del eje X para un comportamiento predecible.

La Cámara con Scrolling (35 min):

Presentar la clase Camera del motor. Explicar su rol de "ventana" al mundo del juego.

Detallar el método apply() y cómo traduce las coordenadas del mundo a coordenadas de pantalla.

Modificar el bucle de dibujado en GameScene para usar camara.apply(sprite) en cada objeto, creando el efecto de scroll.

Construcción del Nivel :

Mostrar cómo, al usar un grupo de sprites para las plataformas, podemos construir un nivel simple pero efectivo.

Probar el juego y ver cómo la cámara sigue al jugador a través del pequeño mundo que hemos creado.

## Discusión: El Mundo como un Mosaico (Tilemaps)
Casi todos los juegos 2D que conoces, desde los clásicos de 8-bits hasta muchos indies modernos, están construidos sobre una rejilla. Un Tilemap es un mosaico digital, una cuadrícula donde cada celda ("tile") contiene una pieza de un rompecabezas más grande (el nivel).

Eficiencia y Reusabilidad: En lugar de tener una imagen gigante para todo el nivel, se reutilizan pequeñas imágenes (los tiles). Un solo tile de "pasto" de 16x16 píxeles puede usarse miles de veces para crear una pradera inmensa, ahorrando una cantidad enorme de memoria y tiempo de desarrollo.

Diseño Rápido: Permite a los diseñadores de niveles (que pueden no ser programadores) crear, modificar y probar mundos de forma visual y rápida usando herramientas especializadas.

Flexibilidad: Los tiles no tienen por qué ser solo visuales. Un tile puede tener propiedades: "este tile es sólido", "este tile hace daño", "este tile es una escalera".

Ejemplos en juegos conocidos:

Pokémon (Rojo/Azul) 🔴🔵: El mundo entero es un tilemap. La hierba alta donde aparecen los Pokémon, los árboles que bloquean el camino, el suelo de las ciudades... todo son tiles colocados en una rejilla.

Shovel Knight ⚔️: Un homenaje moderno a los clásicos que demuestra un diseño de niveles magistral. Utiliza su tileset no solo para crear el terreno, sino también para comunicar sutilmente al jugador dónde puede ir y qué puede hacer.

Terraria / Starbound 🌳: Llevan el concepto al extremo. Son mundos gigantescos y modificables generados proceduralmente a partir de miles de tipos de tiles, cada uno con propiedades y físicas únicas.

Nuestra Herramienta: Tiled Map Editor

Usaremos Tiled (disponible en mapeditor.org), el editor de tilemaps gratuito más popular. El flujo de trabajo es:

Crear un "Tileset" a partir de una imagen que contiene todos nuestros tiles (pasto, tierra, etc.).

"Pintar" nuestro nivel en Tiled usando esos tiles.

Exportar el nivel como un archivo CSV (valores separados por comas). Este archivo será una simple matriz de números que nuestro motor aprenderá a leer.

### Estructura de Archivos Actualizada
Añadimos un módulo tilemap.py al motor y una carpeta levels a nuestro plataformero para los archivos CSV.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- tilemap.py           # ¡NUEVO! Nuestro cargador de tilemaps.
|
|-- platformer/
|   |-- assets/
|   |   |-- tileset.png      # Imagen con todos nuestros tiles.
|   |
|   |-- levels/              # ¡NUEVO!
|   |   |-- level_1.csv      # Nivel exportado desde Tiled.
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt

Ejemplo de level_1.csv
Un archivo CSV exportado de Tiled se verá así (donde -1 es un tile vacío y otros números son los IDs de los tiles):

Fragmento de código

-1,-1,-1,-1,-1,-1,...
-1,-1,-1,-1,-1,-1,...
-1,-1,-1,0,1,2,-1,...
0,1,1,1,1,1,1,1,2,...
3,4,4,4,4,4,4,4,5,...
3. El Código del Motor Actualizado
Este módulo es el corazón de nuestro nuevo sistema de niveles.

engine/tilemap.py (¡NUEVO!)
Python

# engine/tilemap.py
import pygame
import csv
from .game_object import GameObject

class Tile(GameObject):
    """Un objeto simple para representar un tile en el juego."""
    def __init__(self, x, y, image):
        # No llamamos a super() con una ruta, pasamos la imagen ya cargada.
        pygame.sprite.Sprite.__init__(self)
        self.image = image
        self.rect = self.image.get_rect(topleft=(x, y))

class TilemapLoader:
    def __init__(self, tileset_path, tile_size):
        self.tile_size = tile_size
        self.tileset_image = pygame.image.load(tileset_path).convert_alpha()
        self.tiles = self._slice_tileset()

    def _slice_tileset(self):
        """Corta la imagen del tileset en tiles individuales."""
        tiles = {}
        width, height = self.tileset_image.get_size()
        gid = 0
        for y in range(0, height, self.tile_size):
            for x in range(0, width, self.tile_size):
                tile_surface = self.tileset_image.subsurface((x, y, self.tile_size, self.tile_size))
                tiles[gid] = tile_surface
                gid += 1
        return tiles

    def load_level_from_csv(self, csv_path):
        """Lee un archivo CSV y crea un grupo de sprites de tiles."""
        plataformas = pygame.sprite.Group()
        map_data = []
        with open(csv_path) as f:
            reader = csv.reader(f)
            for row in reader:
                map_data.append(list(row))
        
        for j, row in enumerate(map_data):
            for i, tile_gid_str in enumerate(row):
                tile_gid = int(tile_gid_str)
                if tile_gid != -1: # -1 es el tile vacío en Tiled
                    if tile_gid in self.tiles:
                        tile_image = self.tiles[tile_gid]
                        x_pos = i * self.tile_size
                        y_pos = j * self.tile_size
                        tile_sprite = Tile(x_pos, y_pos, tile_image)
                        plataformas.add(tile_sprite)
        
        return plataformas
4. El Código del Juego Plataformero Actualizado
Reemplazamos la creación manual de plataformas por nuestro nuevo cargador.

platformer/scenes/game_scene.py (Grandes Cambios)
Python

# platformer/scenes/game_scene.py
import pygame
from engine.scene import Scene
from engine.camera import Camera
from engine.tilemap import TilemapLoader # Importamos nuestro cargador
from ..entities import Player # Platform ya no es necesaria aquí

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Carga del Nivel ---
        self.tilemap_loader = TilemapLoader('platformer/assets/tileset.png', 16) # Tamaño de tile de 16x16
        self.plataformas = self.tilemap_loader.load_level_from_csv('platformer/levels/level_1.csv')

        if not self.plataformas:
             raise ValueError("El nivel no se pudo cargar o está vacío.")

        # --- Creación de Sprites ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.jugador = Player(100, 200) # Posición inicial del jugador
        
        self.todos_los_sprites.add(self.jugador)
        self.todos_los_sprites.add(self.plataformas) # Añadimos el grupo entero

        self.camara = Camera(800, 600)
        # Podríamos calcular los límites del nivel aquí para la cámara

    def update(self):
        # La lógica de update no cambia
        self.jugador.update(self.plataformas)
        self.camara.update(self.jugador)

    def draw(self, surface):
        # La lógica de dibujado no cambia
        surface.fill((135, 206, 235))
        for sprite in self.todos_los_sprites:
            surface.blit(sprite.image, self.camara.apply(sprite))
Clase 14: Diseño de Niveles con Tilemaps
Contenidos de la Clase
Discusión sobre Diseño por Tiles (20 min): Iniciar con la discusión sobre las ventajas de los tilemaps y los ejemplos de juegos mencionados. Mostrar la interfaz de Tiled y cómo se "pinta" un nivel.

El Cargador de Tilemaps (35 min):

Presentar la nueva clase TilemapLoader en el motor.

Explicar el proceso en dos pasos: primero _slice_tileset corta la imagen principal en un diccionario de "sub-imágenes", cada una con un ID.

Luego, load_level_from_csv lee el archivo de nivel. Por cada número (ID) en el CSV, busca la imagen correspondiente en el diccionario y crea un Tile en la posición correcta.

Integración en la Escena del Juego (35 min):

Refactorizar por completo el __init__ de GameScene.

Eliminar la creación manual de Platform.

Instanciar el TilemapLoader y usarlo para generar el grupo de sprites self.plataformas.

Demostrar cómo cambiar el archivo CSV cargado (level_1.csv, level_2.csv, etc.) cambia completamente el juego sin tocar una línea de la lógica principal.

Límites de la Cámara (20 min):

Discutir y añadir una mejora a la clase Camera para que no se desplace más allá de los límites del nivel. Esto implica calcular el ancho y alto total del tilemap y restringir el camera_rect.x y camera_rect.y.

Cierre y Próximos Pasos
Resumen: Hemos dado uno de los pasos más importantes para crear juegos complejos y variados. Nuestro motor ahora tiene un cargador de tilemaps, lo que nos permite diseñar niveles de forma visual y externa al código. Esta separación entre diseño y lógica es fundamental en el desarrollo de juegos profesional.

Adelanto de la Clase 15: "Hemos construido mundos y personajes, pero la interacción es básica. En la próxima clase, añadiremos enemigos y coleccionables a nuestro plataformero. Programaremos la IA para un enemigo simple que patrulle las plataformas y crearemos monedas u otros objetos para que el jugador los recoja, dándole un objetivo más allá de simplemente llegar al final."





## Clase 15

En esta clase, añadiremos dos elementos cruciales a nuestro plataformero: enemigos con una inteligencia artificial (IA) simple que patrullan el terreno y coleccionables (monedas) que le dan al jugador un objetivo y una recompensa inmediata por explorar.

1. Discusión: El Dúo Dinámico - Desafío y Recompensa
Todo gran plataformero se equilibra sobre estos dos pilares. No existen el uno sin el otro.

El Desafío (Enemigos) 👹: Los enemigos cumplen varias funciones. Crean tensión y exigen habilidad. Sirven como "puertas" móviles que el jugador debe aprender a esquivar o neutralizar. Su comportamiento, por simple que sea, enseña al jugador las reglas del mundo.

La Recompensa (Coleccionables) 💎: Los coleccionables son la motivación. Guían al jugador a través del nivel, recompensan la curiosidad y la exploración, y proporcionan una métrica de éxito clara y satisfactoria (la puntuación).

Ejemplos en juegos conocidos:

Super Mario Bros. 🍄: El Goomba es el primer enemigo perfecto. Su IA es trivial: camina en línea recta. Pero enseña la lección más importante del juego: "saltar sobre los enemigos es bueno". Las monedas, a su vez, no solo dan puntos, sino que a menudo trazan la ruta óptima o más segura a través de un salto, guiando sutilmente al jugador.

Sonic the Hedgehog 🌀: Los anillos son un ejemplo brillante de un coleccionable que es también un mecanismo de defensa. Sirven como la "vida" del jugador. Los enemigos (los Badniks) son el desafío que amenaza con quitarle esos anillos. Esta simbiosis crea un bucle de jugabilidad de riesgo-recompensa constante.

Hollow Knight 🐞: Los enemigos tienen patrones de patrulla y ataque que el jugador debe aprender. Los Geo (la moneda del juego) que sueltan al morir son la recompensa directa por superar ese desafío, incentivando el combate.

Nosotros implementaremos una versión simple de esta dinámica: enemigos que patrullan y monedas que dan puntos.

2. Estructura de Archivos Actualizada
No necesitamos nuevos archivos en el motor, pero sí nuevos assets para nuestras entidades.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- platformer/
|   |-- assets/
|   |   |-- player.png
|   |   |-- ground_tile.png
|   |   |-- enemy.png        # ¡NUEVO!
|   |   |-- coin.png         # ¡NUEVO!
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Juego Plataformero Actualizado
Todo el trabajo de esta clase se centra en las entidades específicas de nuestro juego.

platformer/entities.py (Grandes Cambios)
Añadimos las clases Coin y Enemy.

Python

# platformer/entities.py
import pygame
from engine.game_object import GameObject

# --- Player y Platform (sin cambios) ---
class Player(GameObject):
    # ... (código existente) ...
    def reset(self, x, y):
        """Reinicia la posición del jugador."""
        self.rect.topleft = (x, y)
        self.vel_y = 0
        self.en_suelo = False

class Platform(GameObject):
    # ... (código existente) ...
    pass

# --- Coleccionable (¡NUEVO!) ---
class Coin(GameObject):
    def __init__(self, x, y):
        super().__init__(x, y, 'platformer/assets/coin.png')
        self.pos_inicial_y = y
        self.bob_speed = 0.1
        self.bob_range = 3
        self.angle = 0

    def update(self):
        # Animación de "bobbing" (flotar arriba y abajo)
        self.angle += self.bob_speed
        offset_y = pygame.math.Vector2(0, self.bob_range * pygame.math.sin(self.angle)).y
        self.rect.y = self.pos_inicial_y + offset_y

# --- Enemigo (¡NUEVO!) ---
class Enemy(GameObject):
    def __init__(self, x, y, patrol_range=100):
        super().__init__(x, y, 'platformer/assets/enemy.png')
        self.start_x = x
        self.patrol_range = patrol_range
        self.velocidad = 2
        self.direction = 1 # 1 para derecha, -1 para izquierda

    def update(self):
        # Mover el enemigo
        self.rect.x += self.velocidad * self.direction

        # Lógica de patrulla
        if self.rect.x >= self.start_x + self.patrol_range:
            self.direction = -1
        elif self.rect.x <= self.start_x:
            self.direction = 1
platformer/scenes/game_scene.py (Grandes Cambios)
Integramos las nuevas entidades y su lógica de colisión.

Python

# platformer/scenes/game_scene.py
import pygame
from engine.scene import Scene
from engine.camera import Camera
from engine.tilemap import TilemapLoader
from ..entities import Player, Coin, Enemy # Importamos las nuevas clases

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Carga del Nivel ---
        # ... (código de carga del tilemap sin cambios) ...

        # --- Creación de Sprites ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.plataformas = self.tilemap_loader.load_level_from_csv('platformer/levels/level_1.csv')
        
        # Nuevos grupos para las nuevas entidades
        self.monedas = pygame.sprite.Group()
        self.enemigos = pygame.sprite.Group()

        # Posicionamos las entidades (esto podría venir del tilemap en el futuro)
        self.jugador = Player(100, 200)
        self.posicion_inicial_jugador = (100, 200)

        c1 = Coin(200, 350)
        c2 = Coin(450, 150)
        self.monedas.add(c1, c2)

        e1 = Enemy(300, 352) # 352 para que quede justo sobre el suelo (400 - 48)
        self.enemigos.add(e1)
        
        self.todos_los_sprites.add(self.jugador, self.plataformas, self.monedas, self.enemigos)
        
        self.camara = Camera(800, 600)
        self.puntuacion = 0

    def update(self):
        # Actualizamos todos los sprites, incluyendo los nuevos
        self.jugador.update(self.plataformas)
        self.monedas.update()
        self.enemigos.update()
        self.camara.update(self.jugador)

        # --- Lógica de Colisiones ---
        # Jugador con monedas
        monedas_recogidas = pygame.sprite.spritecollide(self.jugador, self.monedas, True) # True para que la moneda se elimine
        if monedas_recogidas:
            self.puntuacion += len(monedas_recogidas) * 10
            # Aquí podríamos llamar a audio_manager.play_sound('coin')

        # Jugador con enemigos
        if pygame.sprite.spritecollide(self.jugador, self.enemigos, False):
            # Consecuencia: reiniciar la posición del jugador
            self.jugador.reset(self.posicion_inicial_jugador[0], self.posicion_inicial_jugador[1])
            # Aquí podríamos restar una vida y llamar a audio_manager.play_sound('hit')

    def draw(self, surface):
        # ... (código de dibujado sin cambios) ...
        # Podríamos añadir la puntuación a la UI




### Implementando Coleccionables (30 min):

Crear la clase Coin, heredando de GameObject.

Añadir una pequeña animación de "bobbing" (flotar) en su método update usando pygame.math.sin para darle vida y hacerla más atractiva visualmente.

En GameScene, crear un nuevo pygame.sprite.Group para las monedas y añadir la lógica de colisión con spritecollide, asegurándose de que el argumento dokill sea True.

IA de Patrulla para Enemigos :

Crear la clase Enemy, heredando de GameObject.

Implementar la IA de patrulla en su método update: el enemigo se mueve en una dirección hasta que alcanza un límite (start_x + patrol_range), momento en el que invierte su direction.

En GameScene, crear un grupo para los enemigos y añadir la lógica de colisión. Discutir diferentes posibles consecuencias de la colisión (reiniciar, perder vida, perder puntos, etc.) e implementar una de ellas.

Integración y Pruebas :

Poblar el nivel con varias instancias de Coin y Enemy.

Jugar y probar las interacciones. ¿Se siente justo? ¿Es la recompensa adecuada para el riesgo? Esta es una introducción al concepto de "playtesting" y balanceo del juego.

Cierre y Próximos Pasos
Resumen: ¡Nuestro mundo ahora está vivo! Hemos añadido los pilares de la jugabilidad de cualquier plataformero: enemigos que crean un desafío dinámico y coleccionables que recompensan al jugador. Hemos programado nuestra primera IA de patrulla y manejado las colisiones para crear consecuencias y recompensas.






excelente! vayamos por la última clase! De todas formas te comento que leugo de esta clase quiero agregar algunas clases más. luego de esta clase 16, quiero una clase o las clases que necesites par explicar bien, para usar alguna librería de f´sisica, y hacer algún juego tipo pool billar, algo que se vea el movimiento variando la f´sisica. luego quiero otra clase o calses, dónde hagamos un juego de aventura gráfica, y se siga enriqeuciendo el motor. y por último quiero ver algun juego de dialogos, tipo larry, o algo así si tienes otro ejemplo mejor, y que uses llamadas a una api de ia, y enseñes a dar indicaciones al prompt, para crear ese jeugo




## Clase 16 - Empaquetar nuestro juego.

1. Discusión: Los Toques Finales y el Salto al "EXE"
Antes de empaquetar, un juego necesita sentirse "terminado". Esto implica pulir los detalles que separan un prototipo de una experiencia completa.

Flujo Completo: ¿Tiene el juego una pantalla de inicio clara y una pantalla de victoria o fin de juego satisfactoria? El jugador nunca debe quedar atrapado en un callejón sin salida.

Consistencia: ¿La UI, los sonidos y el arte se sienten parte del mismo mundo?

Caza de Bugs (Bug Squashing): Jugar tu propio juego de principio a fin varias veces es la única forma de encontrar errores obvios que arruinarían la experiencia de otro jugador.

¿Por qué empaquetar?

Nadie que no sea desarrollador tiene Python o Pygame instalado. Para que tu amigo, tu familia o un jugador en internet pueda disfrutar de tu juego, no puedes simplemente enviarles los archivos .py. Necesitas crear un paquete autocontenido (un .exe en Windows, un .app en macOS) que incluya:

Tu código del juego.

Una versión del intérprete de Python.

Todas las librerías que usas (como Pygame).

Todos tus assets (imágenes, sonidos, niveles).

Nuestra Herramienta: PyInstaller

Usaremos PyInstaller, la herramienta estándar de la industria para esta tarea. Analiza tu proyecto y lo empaqueta todo en una carpeta o en un único archivo ejecutable.

2. Estructura de Archivos Actualizada
Añadimos un último módulo a nuestro motor, que es CRUCIAL para que PyInstaller encuentre nuestros assets.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- utils.py             # ¡NUEVO! Utilidades, como la ruta de los assets.
|
|-- platformer/
|   |-- ... (resto de la estructura)
|
|-- requirements.txt       # Añadimos pyinstaller
requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
3. El Código del Motor Actualizado
Este pequeño añadido es la clave para un empaquetado sin errores.

engine/utils.py (¡NUEVO!)
PyInstaller crea un ejecutable temporal en una carpeta extraña. Esta función nos devuelve la ruta correcta a nuestros assets, tanto si estamos desarrollando como si el juego está empaquetado.



# engine/utils.py
import sys
import os

def resource_path(relative_path):
    """ Obtiene la ruta absoluta al recurso, funciona para desarrollo y para PyInstaller """
    try:
        # PyInstaller crea una carpeta temporal y guarda la ruta en _MEIPASS
        base_path = sys._MEIPASS
    except Exception:
        # Si no estamos en PyInstaller, la base es el directorio del script principal
        base_path = os.path.abspath(".")

    return os.path.join(base_path, relative_path)
4. Adaptando el Código para Usar resource_path
Ahora debemos modificar TODAS las partes de nuestro código que cargan un archivo externo para que usen esta función.

Ejemplo en engine/audio.py (y en todas las demás cargas)
Python

# engine/audio.py
# ...
from .utils import resource_path # Importamos nuestra función

class AudioManager:
    # ...
    def load_sound(self, name, path):
        full_path = resource_path(path) # Usamos la función para obtener la ruta correcta
        try:
            self.sounds[name] = pygame.mixer.Sound(full_path)
        except pygame.error as e:
            print(f"Error al cargar el sonido: {full_path}\n{e}")

    def load_music(self, path):
        self.music_path = resource_path(path)
Deberás hacer este cambio en todas partes:

En GameObject al cargar la imagen.

En TilemapLoader al cargar el tileset y el archivo CSV.

En cualquier lugar donde hayas puesto una ruta a un archivo de assets/ o levels/.


El Proceso de Empaquetado (40 min):

Abrir una terminal en la carpeta raíz del proyecto (/curso_pygame/).

Ejecutar el comando de PyInstaller. Explicar cada parte del comando:

Bash

pyinstaller platformer/main.py --noconsole --name "PlatformerAdventure" --add-data "platformer/assets:assets" --add-data "platformer/levels:levels"
Desglose del Comando:

pyinstaller platformer/main.py: El comando base que apunta a nuestro punto de entrada.

--noconsole o --windowed: Esencial. Oculta la ventana de consola negra que aparece detrás de nuestro juego.

--name "PlatformerAdventure": Le da un nombre a nuestro ejecutable y a la carpeta de salida.

--add-data "origen:destino": La parte más importante. Le dice a PyInstaller: "Copia la carpeta platformer/assets y, dentro del paquete final, ponla en una carpeta llamada assets". Hacemos lo mismo para levels.

Revisar las carpetas que crea PyInstaller (build y dist). La carpeta dist/PlatformerAdventure es nuestro juego distribuible.

Pruebas y Pasos Finales:

Copiar la carpeta dist/PlatformerAdventure a otro lugar (como el Escritorio) y ejecutar el juego desde ahí para confirmar que funciona de forma independiente.

Discutir cómo comprimir esa carpeta en un .zip para compartirla.

Hablar brevemente de la opción --onefile de PyInstaller, que crea un único .exe pero es más lento para iniciar, y por qué el modo de carpeta es generalmente mejor para juegos.




## Clase 17

En esta sección, integraremos un motor de físicas profesional en nuestro propio motor. Para demostrar su poder, construiremos el esqueleto de un juego tipo billar o pool, donde las colisiones, el momento, la fricción y los rebotes no son animaciones, sino el resultado de una simulación en tiempo real.

1. Discusión: La Belleza del Caos Controlado (Juegos de Físicas)
Los juegos basados en físicas tienen un atractivo único. En lugar de seguir un guion, se basan en un sistema de reglas (las leyes de la física) que pueden producir resultados inesperados, divertidos y profundamente satisfactorios. Esto se conoce como jugabilidad emergente.

Simulación vs. Guion: En nuestro plataformero, el salto siempre tiene la misma altura. En un juego de físicas, el resultado de una acción depende de un complejo conjunto de factores iniciales. Una pequeña variación en el ángulo de un disparo en el billar puede cambiar drásticamente el resultado, haciendo que cada partida sea única.

Satisfacción Táctil: Hay una alegría intrínseca en dominar un sistema de físicas. Ver una torre desmoronarse en Angry Birds o meter una bola de rebote en el billar se siente bien porque no activaste una animación predefinida, sino que manipulaste una simulación que se comportó de forma creíble.

El Humor de la Falla: A menudo, la mayor diversión en estos juegos proviene de los fallos catastróficos. Un mal tiro que hunde la bola blanca, o un personaje que se tropieza de forma ridícula.

Ejemplos en juegos conocidos:

Angry Birds 🐦: El ejemplo perfecto de físicas simples (trayectorias parabólicas, estructuras de bloques) que crean un juego con una profundidad estratégica enorme.

World of Goo ⚫️: Un juego de puzles brillante construido enteramente sobre la física de conectar "bolas de goo" para crear estructuras flexibles y desvencijadas. El desafío es entender y explotar la simulación.

Human: Fall Flat / Gang Beasts 🤸: Estos juegos usan la "física de ragdoll" (muñeco de trapo) como su mecánica central de comedia y jugabilidad. Luchar contra los controles intencionadamente imprecisos de la simulación es el núcleo de la diversión.

Nuestra Herramienta: PyMunk

Usaremos PyMunk, una librería de Python que es un envoltorio (wrapper) para el potente y robusto motor de físicas 2D Chipmunk2D. PyMunk se encargará de todos los cálculos complejos de colisiones, momento, fricción y rotación.

2. Estructura de Archivos del Nuevo Proyecto
/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- physics_object.py    # ¡NUEVO! Clase base para objetos con físicas.
|
|-- billiards/               # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- ball_white.png
|   |   |-- ball_red.png
|   |   |-- table.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- game_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt       # Añadimos pymunk
requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
(Recuerda ejecutar pip install -r requirements.txt)

3. El Código del Motor Actualizado
Añadimos una clase base para integrar PyMunk de forma limpia con nuestra estructura GameObject.

engine/physics_object.py (¡NUEVO!)
Python

# engine/physics_object.py
import pygame
import pymunk
from .game_object import GameObject

class PhysicsObject(GameObject):
    """
    Clase base para objetos del juego controlados por el motor de físicas PyMunk.
    """
    def __init__(self, x, y, image_path, space, mass=1.0, radius=15, elasticity=0.8):
        # Llama a __init__ de GameObject para cargar la imagen y el rect inicial.
        super().__init__(x, y, image_path)

        # Creamos el cuerpo físico (Body)
        moment = pymunk.moment_for_circle(mass, 0, radius)
        self.body = pymunk.Body(mass, moment)
        self.body.position = x, y
        
        # Creamos la forma de colisión (Shape)
        self.shape = pymunk.Circle(self.body, radius)
        self.shape.elasticity = elasticity
        self.shape.friction = 0.5
        
        # Añadimos el cuerpo y la forma al espacio de simulación
        space.add(self.body, self.shape)

    def update(self):
        """
        Sincroniza la posición del sprite visual (rect) con la del cuerpo físico.
        El movimiento es calculado por PyMunk, nosotros solo lo reflejamos.
        """
        self.rect.center = self.body.position
4. El Código del Juego de Billar
billiards/entities.py (¡NUEVO!)
Definimos las bolas, que son objetos dinámicos.

Python

# billiards/entities.py
from engine.physics_object import PhysicsObject

class Ball(PhysicsObject):
    def __init__(self, x, y, image_path, space):
        # Pasamos los parámetros de físicas a la clase padre
        super().__init__(x, y, image_path, space, mass=1, radius=12, elasticity=0.95)
billiards/scenes/game_scene.py (¡NUEVO!)
Aquí es donde configuramos el mundo de PyMunk y manejamos la simulación.

Python

# billiards/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Ball

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # 1. Crear el Espacio de Físicas
        self.space = pymunk.Space()
        self.space.gravity = (0, 0) # Sin gravedad para una mesa de billar

        # --- Creación de Sprites y Cuerpos Físicos ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.bolas = []

        # Crear bolas
        bola_blanca = Ball(200, 300, 'billiards/assets/ball_white.png', self.space)
        self.todos_los_sprites.add(bola_blanca)
        self.bolas.append(bola_blanca)

        for i in range(3):
            bola_roja = Ball(600 + i*30, 300 + i*15, 'billiards/assets/ball_red.png', self.space)
            self.todos_los_sprites.add(bola_roja)
            self.bolas.append(bola_roja)

        # 2. Crear las Paredes Estáticas
        self.crear_paredes(800, 600)

    def crear_paredes(self, ancho, alto):
        """Crea cuerpos estáticos para los bordes de la mesa."""
        # Las paredes son segmentos estáticos, no necesitan un sprite
        bordes = [
            ((50, 50), (ancho - 50, 50)),         # Superior
            ((ancho - 50, 50), (ancho - 50, alto - 50)), # Derecha
            ((50, alto - 50), (ancho - 50, alto - 50)), # Inferior
            ((50, 50), (50, alto - 50))          # Izquierda
        ]
        for p1, p2 in bordes:
            segmento = pymunk.Segment(self.space.static_body, p1, p2, 5) # 5 de grosor
            segmento.elasticity = 0.95
            self.space.add(segmento)

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1: # Click izquierdo
                    # Aplicar un impulso a la bola blanca para simular un golpe
                    pos = pygame.mouse.get_pos()
                    bola_blanca = self.bolas[0]
                    fuerza = 500
                    direccion = pymunk.Vec2d(pos[0] - bola_blanca.body.position.x, pos[1] - bola_blanca.body.position.y).normalized()
                    bola_blanca.body.apply_impulse_at_local_point(fuerza * direccion)
    
    def update(self):
        # 3. Avanzar la Simulación de Físicas
        # ¡Esta línea reemplaza toda nuestra lógica de movimiento anterior!
        dt = 1.0 / 60.0 # Delta time
        self.space.step(dt)

        # Sincronizamos los sprites con los cuerpos físicos
        self.todos_los_sprites.update()

    def draw(self, surface):
        surface.fill((10, 80, 30)) # Color de la mesa
        self.todos_los_sprites.draw(surface)

## Clase 18: Módulo de Físicas con PyMunk
Contenidos de la Clase
Discusión sobre Jugabilidad Emergente (20 min): Iniciar con la discusión sobre juegos de físicas y los ejemplos mencionados. Enfatizar la diferencia entre un evento "scriptado" y uno "simulado".

Conceptos Clave de PyMunk (30 min):

Espacio (Space): El universo donde las leyes de la física aplican.

Cuerpo (Body): El objeto con propiedades como masa, posición y velocidad. Puede ser dinámico (se mueve, como una bola) o estático (no se mueve, como una pared).

Forma (Shape): La geometría de colisión del objeto (círculo, polígono). Se adjunta a un cuerpo.

Explicar cómo la combinación Body + Shape se añade al Space.

Integración con el Motor (35 min):

Presentar la nueva clase base PhysicsObject. Mostrar cómo super().__init__ ahora no solo carga una imagen, sino que también crea y configura un cuerpo y una forma en PyMunk.

Explicar el cambio de paradigma en el método update: ya no calculamos la posición, solo leemos la posición que el motor de físicas ha calculado para nosotros.

La Simulación en Acción (25 min):

Detallar la lógica de la GameScene. Mostrar cómo se configura el Space y se crean los objetos.

Explicar que la línea self.space.step(dt) es el verdadero "corazón" del bucle de actualización.

Implementar la interacción básica: aplicar un impulso a la bola blanca con el ratón para ver la simulación en acción.

Cierre y Próximos Pasos
Resumen: Hemos añadido una capacidad increíblemente potente a nuestro motor: la simulación de físicas 2D. Hemos aprendido a delegar todo el complejo cálculo de movimiento y colisiones a una librería especializada (PyMunk), permitiéndonos crear interacciones realistas y complejas con muy poco código.

Adelanto de la Clase 18: "Tenemos las físicas básicas, pero un juego de billar es más que eso. En la próxima clase, refinaremos la jugabilidad. Implementaremos la fricción y la resistencia angular para que las bolas se detengan y dejen de girar de forma realista. Crearemos los agujeros (troneras) y detectaremos cuándo una bola cae en ellos. Finalmente, estableceremos un sistema de turnos y reglas básicas del juego."



## Clase 18




1. Discusión: De la Simulación al Juego
Una simulación de físicas es fascinante, pero no es un juego. Un juego necesita reglas, objetivos y consecuencias.

Fricción y Amortiguación (Damping): ¿Por qué una bola de billar real se detiene? Por dos motivos: la fricción por deslizamiento contra el tapete y la resistencia del aire. En PyMunk, esto se simula con el parámetro damping. Un damping de 1.0 significa que no hay pérdida de velocidad (como en el espacio). Un valor ligeramente inferior, como 0.99, hará que los objetos pierdan un pequeño porcentaje de su velocidad en cada paso de la simulación, deteniéndose de forma natural.

Los Agujeros (Troneras): ¿Cómo detectamos que una bola ha caído en un agujero? Hay varias técnicas. Una muy efectiva es crear "sensores" en PyMunk. Un sensor es una forma de colisión que detecta cuándo algo entra en su área, pero no produce una respuesta física. Es como un fantasma que puede ver pero no tocar. Colocaremos sensores en la posición de las troneras para detectar las bolas.

Sistema de Turnos y Reglas: El billar es un juego por turnos. Necesitamos un sistema de estados que controle:

Estado de "Apuntado": El jugador puede preparar su tiro.

Estado de "Simulación": Las bolas se están moviendo. El control del jugador está desactivado.

Cambio de Turno: Cuando todas las bolas se detienen, se evalúa el resultado y se pasa al siguiente estado o jugador.

Ejemplos en juegos conocidos:

Pool Nation / 8 Ball Pool 🎱: Estos juegos digitales de billar tienen una física muy pulida. La sensación de la fricción, el efecto que se le puede dar a la bola (rotación) y la precisión de los rebotes son clave para su éxito.

Golf With Your Friends ⛳️: Aunque es un juego de golf, la mecánica es idéntica: un estado de apuntado, un golpe que inicia una simulación, y un estado de espera hasta que la bola se detiene.

Worms Armageddon 💣: Un clásico por turnos donde la física es la protagonista. Cada turno consiste en apuntar, disparar y luego observar las caóticas consecuencias hasta que todo se asienta y comienza el turno del siguiente jugador.

2. Estructura de Archivos Actualizada
No necesitamos nuevos archivos, pero sí nuevos assets para la mesa y los agujeros.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- billiards/
|   |-- assets/
|   |   |-- table.png        # Una imagen de la mesa completa puede servir de fondo
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Juego de Billar Actualizado
Refinamos la escena del juego para añadir las nuevas mecánicas.

billiards/entities.py (Sin Cambios)
Nuestra clase Ball ya es suficientemente buena para lo que necesitamos.

billiards/scenes/game_scene.py (Grandes Cambios)
Python

# billiards/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Ball

# Constantes para los tipos de colisión
COLLISION_TYPE_BALL = 1
COLLISION_TYPE_HOLE = 2

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Configuración del Espacio de Físicas ---
        self.space = pymunk.Space()
        self.space.damping = 0.98 # ¡NUEVO! Fricción para que las bolas se detengan

        # --- Estado del Juego ---
        self.game_state = "player_turn" # Estados: 'player_turn', 'simulating'
        self.cue_start_pos = None

        # --- Creación de Sprites y Cuerpos ---
        self.todos_los_sprites = pygame.sprite.Group()
        self.bolas = pygame.sprite.Group()

        # ... (código para crear las bolas, pero ahora les asignamos un tipo de colisión) ...
        # Ejemplo:
        bola_blanca = Ball(...)
        bola_blanca.shape.collision_type = COLLISION_TYPE_BALL
        self.bolas.add(bola_blanca)
        # ...

        # --- Creación de la Mesa y Agujeros ---
        self.crear_mesa(800, 600)
        
        # --- Gestor de Colisiones ---
        handler = self.space.add_collision_handler(COLLISION_TYPE_BALL, COLLISION_TYPE_HOLE)
        handler.begin = self.on_ball_in_hole

    def crear_mesa(self, ancho, alto):
        # ... (código para crear las paredes estáticas) ...

        # ¡NUEVO! Crear los agujeros como sensores
        self.agujeros = []
        hole_radius = 22
        posiciones_agujeros = [(50, 50), (ancho/2, 45), (ancho-50, 50), (50, alto-50), (ancho/2, alto-45), (ancho-50, alto-50)]
        for pos in posiciones_agujeros:
            hole_body = pymunk.Body(body_type=pymunk.Body.STATIC)
            hole_body.position = pos
            hole_shape = pymunk.Circle(hole_body, hole_radius)
            hole_shape.sensor = True # ¡Esto lo convierte en un sensor!
            hole_shape.collision_type = COLLISION_TYPE_HOLE
            self.space.add(hole_shape)
            self.agujeros.append(hole_shape)

    def on_ball_in_hole(self, arbiter, space, data):
        """Callback que se ejecuta cuando una bola entra en un agujero."""
        ball_shape = arbiter.shapes[0]
        # Buscamos el sprite correspondiente a la forma para eliminarlo
        for bola_sprite in self.bolas:
            if bola_sprite.shape == ball_shape:
                bola_sprite.kill() # Elimina el sprite de todos los grupos
                space.remove(ball_shape, ball_shape.body) # Elimina el cuerpo físico
        return True # Importante para que la colisión se procese

    def handle_events(self, events):
        if self.game_state == "player_turn":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    self.cue_start_pos = event.pos
                if event.type == pygame.MOUSEBUTTONUP and event.button == 1 and self.cue_start_pos:
                    end_pos = event.pos
                    bola_blanca = self.bolas.sprites()[0] # Asumimos que la blanca es la primera
                    
                    # Calculamos la fuerza y dirección inversas al arrastre del ratón
                    fuerza_x = (self.cue_start_pos[0] - end_pos[0]) * 5
                    fuerza_y = (self.cue_start_pos[1] - end_pos[1]) * 5
                    
                    bola_blanca.body.apply_impulse_at_local_point((fuerza_x, fuerza_y))
                    self.game_state = "simulating"
                    self.cue_start_pos = None

    def update(self):
        # Avanzar la simulación solo si las bolas se mueven
        if self.game_state == "simulating":
            dt = 1.0 / 60.0
            self.space.step(dt)
            self.todos_los_sprites.update()

            # Comprobar si todas las bolas se han detenido
            are_balls_moving = False
            for bola in self.bolas:
                if bola.body.velocity.length > 0.1: # Un pequeño umbral
                    are_balls_moving = True
                    break
            if not are_balls_moving:
                self.game_state = "player_turn"

    def draw(self, surface):
        surface.fill((10, 80, 30))
        # Dibujar los agujeros (opcional, para depuración)
        for hole in self.agujeros:
            pygame.draw.circle(surface, (0,0,0), hole.body.position, hole.radius)
        
        self.todos_los_sprites.draw(surface)

        # Dibujar la línea de apuntado
        if self.game_state == "player_turn" and self.cue_start_pos:
            pygame.draw.line(surface, (255, 255, 255), self.cue_start_pos, pygame.mouse.get_pos(), 2)

## Clase 19: Reglas y Pulido del Juego de Físicas
Contenidos de la Clase
Discusión sobre el "Jugo" de la Simulación (20 min): Iniciar con la discusión sobre fricción, rebote y el sistema de turnos. Usar los ejemplos para ilustrar cómo estas reglas convierten la simulación en un juego estructurado.

Fricción y Detección de Parada (30 min):

Introducir el concepto de space.damping y cómo simula la pérdida de energía en el sistema.

Implementar la lógica en el update para comprobar si todas las bolas se han detenido. Explicar por qué se usa un pequeño umbral (> 0.1) en lugar de == 0 para evitar problemas con la precisión de los números flotantes.

Implementar la máquina de estados player_turn / simulating.

Sensores y Gestores de Colisión (40 min):

Explicar el concepto de sensores en PyMunk y por qué son perfectos para los agujeros.

Introducir el gestor de colisiones (space.add_collision_handler). Explicar que es una forma de suscribirse a eventos de colisión entre tipos específicos de objetos.

Implementar la función on_ball_in_hole y mostrar cómo se usa para eliminar la bola tanto del espacio de simulación como de los grupos de sprites de Pygame.

Implementando el Golpeo y Turnos (20 min):

Refinar la lógica de handle_events para que el golpeo de la bola solo sea posible durante el player_turn.

Implementar un sistema simple de "arrastrar y soltar" con el ratón para determinar la fuerza y dirección del golpe.

Mostrar cómo el juego transiciona al estado simulating después del golpe, quitándole el control al jugador.

Cierre y Próximos Pasos
Resumen: Hemos transformado nuestra simulación de físicas en un juego funcional. Hemos implementado fricción para un movimiento realista, sensores para detectar objetivos, un gestor de colisiones para manejar las consecuencias, y un sistema de turnos que estructura la experiencia de juego.

## Clase 20

1. Discusión: El Arte de la Destrucción Calculada
Los juegos de lanzamiento de proyectiles son un género increíblemente popular y satisfactorio. Su bucle de jugabilidad es simple pero adictivo: apuntar, lanzar y observar las consecuencias.

Control Indirecto y Compromiso: A diferencia de un plataformero donde controlas al personaje en todo momento, aquí toda tu habilidad se concentra en un único momento: el lanzamiento. Una vez que el proyectil está en el aire, eres un mero espectador. Esto crea una tensión y una satisfacción enormes cuando el plan funciona.

La Física como Puzle: El verdadero objetivo no es solo lanzar, sino entender la simulación. ¿Cómo afectará la gravedad a la trayectoria? ¿Cómo se comportará una torre de bloques al ser golpeada en su base vs. en su cima? Cada nivel es un puzle de físicas.

La Recompensa del Caos: El "premio" en estos juegos es la destrucción. Ver una estructura cuidadosamente construida desmoronarse en una cascada de piezas, acompañado de efectos de sonido contundentes, es una de las experiencias más gratificantes en los videojuegos.

Ejemplos en juegos conocidos:

Worms Armageddon 💣: El abuelo del género por turnos. Añade variables como el viento y un arsenal de armas creativas, pero el núcleo sigue siendo el mismo: calcular la parábola perfecta para golpear a tu oponente.

Angry Birds 🐦: El rey indiscutible del género en móviles. Su genialidad radica en su simplicidad táctil ("arrastrar y soltar") y en la introducción de proyectiles con diferentes habilidades, lo que añade una capa de estrategia a la física básica.

Crush the Castle 🏰: Un precursor de Angry Birds que demuestra la pura satisfacción del concepto. Usando un trabuquete, el objetivo es simplemente derribar castillos. La alegría proviene únicamente de la física de la destrucción.

2. Estructura de Archivos del Nuevo Proyecto
Creamos una nueva carpeta slingshot para nuestro tercer juego. El motor engine sigue siendo el mismo, demostrando su increíble reutilización.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios, reutilizaremos todo)
|
|-- billiards/
|-- arkanoid/
|
|-- slingshot/             # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- projectile.png
|   |   |-- block.png
|   |   |-- target.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- game_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt       # (sin cambios, ya tenemos pygame y pymunk)
3. El Código del Juego de Lanzamiento
Aquí es donde aplicamos nuestros conocimientos de PyMunk en un nuevo contexto.

slingshot/entities.py (¡NUEVO!)
Definimos nuestros nuevos actores: el proyectil y los bloques que formarán las estructuras.

Python

# slingshot/entities.py
import pygame
import pymunk
from engine.physics_object import PhysicsObject

class Projectile(PhysicsObject):
    def __init__(self, x, y, space):
        # Un proyectil es esencialmente una bola de billar, pero en un entorno con gravedad.
        super().__init__(x, y, 'slingshot/assets/projectile.png', space, mass=5, radius=10, elasticity=0.6)
        # Lo mantenemos "dormido" al principio para que no caiga.
        self.body.sleep()

    def launch(self, impulse):
        """Activa el cuerpo y le aplica el impulso de lanzamiento."""
        self.body.activate()
        self.body.apply_impulse_at_local_point(impulse)

class Block(PhysicsObject):
    def __init__(self, x, y, width, height, space, mass=1.0):
        # Usamos un constructor personalizado en lugar del de la clase padre
        # para crear una forma poligonal (un rectángulo).
        pygame.sprite.Sprite.__init__(self)
        
        image_path = 'slingshot/assets/block.png'
        self.image = pygame.image.load(image_path).convert_alpha()
        self.image = pygame.transform.scale(self.image, (width, height))
        self.rect = self.image.get_rect(center=(x, y))

        # Creamos el cuerpo y la forma de la caja
        moment = pymunk.moment_for_box(mass, (width, height))
        self.body = pymunk.Body(mass, moment)
        self.body.position = x, y
        self.shape = pymunk.Poly.create_box(self.body, (width, height))
        self.shape.elasticity = 0.4
        self.shape.friction = 0.7
        
        space.add(self.body, self.shape)
    
    def update(self):
        # Necesitamos un update para manejar la rotación del sprite
        self.rect.center = self.body.position
        # PyMunk nos da el ángulo en radianes, Pygame lo necesita en grados.
        self.image_rotated = pygame.transform.rotate(self.image, -self.body.angle * 180 / 3.1415)
        self.rect = self.image_rotated.get_rect(center=self.rect.center)
    
    def draw(self, surface):
        surface.blit(self.image_rotated, self.rect)
Nota Importante: Hemos creado una clase Block que ahora maneja la rotación. Esto es un añadido clave para que los bloques se derrumben de forma realista.

slingshot/scenes/game_scene.py (¡NUEVO!)
Aquí reside la lógica principal de apuntado, lanzamiento y construcción de estructuras.

Python

# slingshot/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Projectile, Block

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        
        # --- Configuración del Espacio de Físicas ---
        self.space = pymunk.Space()
        self.space.gravity = (0, 500) # ¡La gravedad vuelve!

        # --- Estado del Juego ---
        self.game_state = "aiming" # 'aiming', 'simulating'
        self.start_drag_pos = None

        # --- Creación de Entidades ---
        self.todos_los_sprites = pygame.sprite.Group()
        
        # Proyectil en su posición de lanzamiento
        self.launch_pos = (150, 350)
        self.proyectil = Projectile(self.launch_pos[0], self.launch_pos[1], self.space)
        self.todos_los_sprites.add(self.proyectil)

        # Suelo estático
        suelo = pymunk.Segment(self.space.static_body, (0, 500), (800, 500), 5)
        suelo.friction = 1.0
        self.space.add(suelo)

        # Construir una estructura para destruir
        for i in range(3):
            block = Block(600, 484 - i * 32, 100, 30, self.space)
            self.todos_los_sprites.add(block)
        block = Block(600, 484 - 3 * 32, 30, 30, self.space)
        self.todos_los_sprites.add(block)


    def handle_events(self, events):
        if self.game_state == "aiming":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    self.start_drag_pos = event.pos
                if event.type == pygame.MOUSEBUTTONUP and event.button == 1 and self.start_drag_pos:
                    end_pos = event.pos
                    
                    fuerza_x = (self.start_drag_pos[0] - end_pos[0]) * 3
                    fuerza_y = (self.start_drag_pos[1] - end_pos[1]) * 3
                    
                    self.proyectil.launch((fuerza_x, fuerza_y))
                    self.game_state = "simulating"
                    self.start_drag_pos = None

    def update(self):
        if self.game_state == "simulating":
            dt = 1.0 / 60.0
            self.space.step(dt)
        
        self.todos_los_sprites.update()

    def draw(self, surface):
        surface.fill((190, 220, 255)) # Cielo
        pygame.draw.rect(surface, (100, 80, 70), (0, 500, 800, 100)) # Tierra
        
        # Dibujar la línea de apuntado
        if self.game_state == "aiming" and self.start_drag_pos:
            pygame.draw.line(surface, (0,0,0), self.launch_pos, self.start_drag_pos, 3)
            pygame.draw.line(surface, (255, 255, 255), self.start_drag_pos, pygame.mouse.get_pos(), 2)

        self.todos_los_sprites.draw(surface)

## Clase 21: Físicas de Proyectiles y Destrucción
Contenidos de la Clase
Discusión sobre Lanzamiento y Caos: Iniciar con la discusión sobre el bucle de jugabilidad de apuntar-lanzar-observar, usando los ejemplos de juegos mencionados.

Físicas con Gravedad :

Configurar el pymunk.Space con un vector de gravedad.

Crear la clase Block usando pymunk.Poly.create_box para las formas rectangulares.

Implementar la lógica de rotación en el update y draw de la clase Block para que las estructuras se derrumben de forma visualmente correcta.

El Mecanismo de Lanzamiento (40 min):

Implementar la lógica de "arrastrar y soltar" con el ratón para apuntar.

Explicar cómo el vector creado por el arrastre del ratón se traduce en un impulso para el proyectil.

Introducir la idea de "dormir" (body.sleep()) y "despertar" (body.activate()) los cuerpos de PyMunk para tener un proyectil que no se ve afectado por la física hasta que es lanzado.

Construyendo para Destruir :

Usar la nueva clase Block para apilar objetos y crear una torre simple.

Probar el juego, lanzar el proyectil y disfrutar de la satisfactoria destrucción de la estructura basada en la simulación de físicas.


1. Discusión: "Leyendo la Mente" del Jugador
Un buen diseño de juego a menudo consiste en anticipar lo que el jugador quiere hacer y darle las herramientas para que se sienta inteligente.

Información, no Adivinación: Forzar al jugador a adivinar la trayectoria de un proyectil puede ser frustrante. Una línea de predicción, como el icónico arco de puntos de Angry Birds, no le quita el desafío al juego; lo traslada. El desafío ya no es "adivinar la física", sino "usar la física para resolver el puzle". Le da al jugador la información que necesita para tomar decisiones estratégicas.

Feedback y Recompensa Claros: ¿Cómo sabe el jugador si lo está haciendo bien? Una puntuación que aumenta con la destrucción proporciona un feedback instantáneo y positivo. Un sonido gratificante y un número que sube son una de las recompensas más simples y efectivas en el diseño de juegos.

Minimizando la Fricción: ¿Qué pasa si el jugador hace un mal tiro? Forzarle a reiniciar todo el juego es una mala experiencia. Un simple botón de reinicio respeta el tiempo del jugador y le anima a experimentar y probar de nuevo inmediatamente, manteniendo el flujo del juego.

Ejemplos en juegos conocidos:

Angry Birds 🐦: El ejemplo por excelencia. La línea de puntos que muestra la trayectoria del pájaro anterior es una herramienta de aprendizaje y ajuste fundamental.

Golf With Your Friends / What the Golf? ⛳️: Muchos juegos de mini golf muestran una línea que predice el primer rebote de la pelota. No te muestra toda la solución, pero te da la información crucial para el primer paso.

Into the Breach 🤖: Un juego de estrategia por turnos donde la información es REY. El juego te dice exactamente qué van a hacer los enemigos en su turno. Tu trabajo no es adivinar, sino usar esa información perfecta para resolver el "puzle" del combate sin recibir daño. Es el extremo de la filosofía de "información sobre adivinación".

2. Estructura de Archivos Actualizada
No necesitamos nuevos archivos para esta clase. Todo el trabajo se realizará dentro de la escena de nuestro juego, demostrando cómo se puede construir una jugabilidad compleja sobre la base sólida de nuestro motor.

3. El Código del Juego de Lanzamiento Actualizado
Toda la magia de hoy ocurre en game_scene.py.

slingshot/entities.py (Pequeño Añadido)
Añadimos un método para reiniciar la posición del proyectil.

Python

# slingshot/entities.py
# ... (clase Block sin cambios) ...

class Projectile(PhysicsObject):
    # ... (__init__ y launch sin cambios) ...

    def reset(self, x, y):
        """Detiene el cuerpo, lo reinicia y lo pone a dormir."""
        self.body.velocity = 0, 0
        self.body.angular_velocity = 0
        self.body.position = x, y
        self.body.sleep()
slingshot/scenes/game_scene.py (Grandes Cambios)
Refactorizamos la escena para incluir la predicción de trayectoria, la puntuación y el reinicio.

Python

# slingshot/scenes/game_scene.py
import pygame
import pymunk
from engine.scene import Scene
from ..entities import Projectile, Block

class GameScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        self._setup_level()

    def _setup_level(self):
        """Configura o reinicia todas las entidades y el estado del nivel."""
        self.space = pymunk.Space()
        self.space.gravity = (0, 500)
        
        self.game_state = "aiming"
        self.start_drag_pos = None
        self.puntuacion = 0
        
        self.todos_los_sprites = pygame.sprite.Group()
        
        self.launch_pos = (150, 350)
        self.proyectil = Projectile(self.launch_pos[0], self.launch_pos[1], self.space)
        self.todos_los_sprites.add(self.proyectil)

        suelo = pymunk.Segment(self.space.static_body, (0, 500), (800, 500), 5)
        suelo.friction = 1.0
        self.space.add(suelo)

        # Construir estructura
        for i in range(3):
            block = Block(600, 484 - i * 32, 100, 30, self.space)
            self.todos_los_sprites.add(block)
        block = Block(600, 484 - 3 * 32, 30, 30, self.space)
        self.todos_los_sprites.add(block)

    def handle_events(self, events):
        # ... (lógica de MOUSEBUTTONDOWN y MOUSEBUTTONUP sin cambios) ...
        for event in events:
            if event.type == pygame.KEYDOWN and event.key == pygame.K_r:
                self._setup_level() # Reiniciar con la tecla 'R'

    def update(self):
        # ... (lógica de update sin cambios) ...
        if self.game_state == "simulating":
            # Podríamos añadir aquí una lógica de puntuación
            # por ejemplo, si un bloque cae por debajo de cierta altura.
            pass

    def _calculate_trajectory(self):
        """Ejecuta una simulación fantasma para predecir la trayectoria."""
        if not self.start_drag_pos:
            return []

        end_pos = pygame.mouse.get_pos()
        fuerza_x = (self.start_drag_pos[0] - end_pos[0]) * 3
        fuerza_y = (self.start_drag_pos[1] - end_pos[1]) * 3
        
        # Crear un cuerpo fantasma
        ghost_body = pymunk.Body(5, pymunk.moment_for_circle(5, 0, 10))
        ghost_body.position = self.proyectil.body.position
        ghost_body.apply_impulse_at_local_point((fuerza_x, fuerza_y))
        
        path = []
        for _ in range(100): # Simular 100 pasos en el futuro
            ghost_body.velocity += self.space.gravity * (1/60.0)
            ghost_body.position += ghost_body.velocity * (1/60.0)
            if _ % 5 == 0: # Guardar un punto cada 5 pasos
                path.append(ghost_body.position)
        return path

    def draw(self, surface):
        # ... (dibujado del fondo y los sprites) ...

        # Dibujar la línea de apuntado y la predicción
        if self.game_state == "aiming" and self.start_drag_pos:
            # Línea de fuerza
            pygame.draw.line(surface, (0,0,0), self.launch_pos, self.start_drag_pos, 3)
            
            # Línea de predicción
            path_points = self._calculate_trajectory()
            for point in path_points:
                pygame.draw.circle(surface, (255, 255, 255, 150), point, 3)

        # Dibujar UI (puntuación, etc.)
        font = pygame.font.Font(None, 36)
        score_text = font.render(f"Puntuación: {self.puntuacion}", True, (0,0,0))
        surface.blit(score_text, (10, 10))

## Clase 22: Feedback al Jugador y Cierre del Bucle
Contenidos de la Clase
Discusión sobre Calidad de Vida : Iniciar con la discusión sobre la importancia de la información, el feedback claro y la reducción de la frustración, usando los ejemplos de juegos mencionados.

Predicción de Trayectoria:

Explicar el concepto de una "simulación fantasma". No podemos usar nuestro space principal porque alteraría el estado del juego.

Implementar el método _calculate_trajectory. Detallar los pasos:

Calcular el impulso potencial basado en la posición actual del ratón.

Crear un pymunk.Body temporal y completamente separado.

Aplicarle el impulso.

Hacer un bucle que simule el paso del tiempo (body.position += body.velocity * dt) y aplique la gravedad manualmente.

Almacenar las posiciones resultantes en una lista.

En el método draw, recorrer esa lista de puntos para dibujar el arco de predicción.

Refactorización para Reinicio (25 min):

Explicar por qué es una mala práctica tener toda la lógica de creación de niveles en el __init__.

Crear el método _setup_level() y mover toda esa lógica a él. El __init__ ahora solo llama a este método.

En handle_events, añadir la escucha de una tecla (ej: 'R') que simplemente vuelve a llamar a _setup_level(), reiniciando el juego de forma limpia e instantánea.

Sistema de Puntuación Básico (25 min):

Discutir diferentes formas de puntuar: por bloques destruidos, por altura a la que caen, por golpear objetivos específicos.

Implementar la versión más simple: añadir una clase Target (objetivo) y dar puntos solo cuando el proyectil lo golpea.

Añadir el dibujado de la puntuación en la pantalla, completando el bucle de feedback para el jugador.


1. Discusión: El Cerebro sobre los Músculos
Las aventuras gráficas, también conocidas como juegos "Point-and-Click", ofrecen un tipo de desafío completamente diferente. La habilidad no reside en los reflejos, sino en la observación, la lógica y el pensamiento lateral.

Exploración y Observación: El mundo del juego es un diorama interactivo. El primer paso es siempre explorar cada rincón y "leer" el entorno. ¿Qué objetos parecen fuera de lugar? ¿Qué detalles en el fondo podrían ser una pista?

El Inventario como Caja de Herramientas: El jugador recoge objetos que son las "llaves" para resolver los puzles. La jugabilidad central a menudo gira en torno a la pregunta: "¿Qué objeto de mi inventario puedo usar aquí?".

La Lógica del Puzle: El corazón del género. Los puzles pueden ir desde lo simple y directo ("usar llave en puerta") hasta lo absurdamente creativo o cómico. La famosa "lógica de mono" de la saga Monkey Island es un ejemplo de puzles que requieren pensar de forma no convencional (ej: "usar pollo de goma con polea").

Narrativa y Diálogo: A diferencia de nuestros juegos anteriores, la historia es la fuerza motriz. Las conversaciones con otros personajes no son solo un adorno, sino una fuente vital de pistas, lore y, a menudo, puzles en sí mismas.

Ejemplos en juegos conocidos:

The Secret of Monkey Island 🐒: El referente absoluto. Famoso por su humor, sus puzles ingeniosos y su sistema de diálogo con insultos en duelos de espadas. Es el ejemplo perfecto de cómo la narrativa y la mecánica pueden fusionarse.

Broken Sword: La leyenda de los templarios ⚔️: Una aventura con un tono más serio y cinematográfico. Demuestra cómo el género puede contar historias de misterio e intriga, donde la investigación y la deducción son clave.

Thimbleweed Park 👾: Un homenaje moderno a los clásicos, creado por los mismos diseñadores de Monkey Island. Muestra que la fórmula sigue siendo increíblemente efectiva hoy en día, con una interfaz pulida y puzles complejos que involucran a múltiples personajes.

2. Estructura de Archivos del Nuevo Proyecto
Creamos una nueva carpeta adventure y añadimos componentes al motor para gestionar las nuevas mecánicas.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- hotspot.py           # ¡NUEVO! Para áreas interactivas.
|   |-- inventory.py         # ¡NUEVO! Nuestro sistema de inventario.
|
|-- adventure/               # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- room_office.png
|   |   |-- item_key.png
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- room_scene.py      # Clase base para nuestras habitaciones.
|   |   |-- office.py          # La primera habitación del juego.
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Motor Actualizado
Añadimos dos nuevos módulos para las mecánicas centrales de este género.

engine/hotspot.py (¡NUEVO!)
Un Hotspot es un rectángulo invisible en la pantalla. Cuando el jugador hace clic en él, se activa una acción.

Python

# engine/hotspot.py
import pygame

class Hotspot:
    def __init__(self, rect, callback, one_time=False):
        self.rect = pygame.Rect(rect)
        self.callback = callback
        self.one_time = one_time
        self.triggered = False

    def check_click(self, pos):
        """Comprueba si el clic ocurrió dentro del hotspot."""
        if self.rect.collidepoint(pos):
            if self.one_time and self.triggered:
                return False # Ya se ha activado una vez
            self.triggered = True
            return True
        return False
engine/inventory.py (¡NUEVO!)
Un sistema simple para gestionar y dibujar los objetos que el jugador posee.

Python

# engine/inventory.py
import pygame

class Inventory:
    def __init__(self):
        self.items = {} # Usamos un diccionario para acceder por nombre
        self.font = pygame.font.Font(None, 24)
        self.visible = True

    def add_item(self, name, image_path):
        """Añade un objeto al inventario."""
        if name not in self.items:
            self.items[name] = pygame.image.load(image_path).convert_alpha()
            print(f"Objeto añadido: {name}")

    def has_item(self, name):
        """Comprueba si un objeto está en el inventario."""
        return name in self.items

    def draw(self, surface):
        """Dibuja la barra de inventario."""
        if not self.visible:
            return
        
        # Dibuja un fondo para la barra de inventario
        bar_rect = pygame.Rect(0, surface.get_height() - 60, surface.get_width(), 60)
        pygame.draw.rect(surface, (20, 20, 40, 200), bar_rect)

        # Dibuja cada objeto
        x_offset = 10
        for name, image in self.items.items():
            surface.blit(image, (x_offset, surface.get_height() - 50))
            x_offset += image.get_width() + 10
4. El Código de la Aventura Gráfica
adventure/scenes/room_scene.py (¡NUEVO!)
Esta es una clase base que todas nuestras habitaciones heredarán. Gestiona un fondo, los hotspots y el inventario.

Python

# adventure/scenes/room_scene.py
from engine.scene import Scene

class RoomScene(Scene):
    def __init__(self, scene_manager, background_path, inventory):
        super().__init__(scene_manager)
        self.background = pygame.image.load(background_path).convert()
        self.hotspots = []
        self.inventory = inventory

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                for hotspot in self.hotspots:
                    if hotspot.check_click(event.pos):
                        hotspot.callback()

    def draw(self, surface):
        surface.blit(self.background, (0, 0))
        self.inventory.draw(surface)
adventure/scenes/office.py (¡NUEVO!)
Nuestra primera habitación. Define sus propios hotspots y la lógica de interacción.

Python

# adventure/scenes/office.py
import pygame
from .room_scene import RoomScene
from engine.hotspot import Hotspot

class OfficeScene(RoomScene):
    def __init__(self, scene_manager, inventory):
        super().__init__(scene_manager, 'adventure/assets/room_office.png', inventory)
        
        # Definimos los hotspots de esta habitación
        self.hotspots.append(Hotspot((300, 250, 50, 50), self.on_click_key, one_time=True))
        self.hotspots.append(Hotspot((500, 150, 100, 200), self.on_click_door))

    # --- Callbacks de los Hotspots ---
    def on_click_key(self):
        print("Has encontrado una llave.")
        self.inventory.add_item("llave", "adventure/assets/item_key.png")
        # Podríamos eliminar el hotspot o cambiar la imagen de fondo para que la llave no se vea

    def on_click_door(self):
        if self.inventory.has_item("llave"):
            print("Usas la llave y abres la puerta. ¡Has escapado!")
            # Aquí transicionaríamos a la siguiente escena/habitación
            # self.scene_manager.go_to(CorridorScene)
        else:
            print("La puerta está cerrada. Necesitas una llave.")

## Clase 23: Aventura Gráfica - Escenas, Hotspots e Inventario
Contenidos de la Clase
Discusión sobre el Género Narrativo (20 min): Iniciar con la discusión sobre los pilares del género (exploración, puzles, inventario) y los ejemplos de juegos clásicos y modernos.

Arquitectura Point-and-Click (35 min):

Presentar las nuevas clases del motor: Hotspot e Inventory.

Explicar el rol de un Hotspot como un área de clic invisible que dispara una función "callback".

Detallar la implementación del Inventory como un diccionario que almacena imágenes y tiene un método para dibujarse en pantalla.

Construyendo una Habitación Interactiva (45 min):

Introducir la clase base RoomScene, que encapsula la lógica común a todas las habitaciones (fondo, manejo de clics en hotspots, inventario).

Implementar la primera habitación, OfficeScene, que hereda de RoomScene.

Mostrar cómo se crean y añaden Hotspot a la lista de la habitación, asociando cada uno a un método específico (el callback).

Programar la lógica del puzle: on_click_key modifica el inventario, y on_click_door comprueba el estado del inventario para determinar el resultado.

Flujo de Juego y Pruebas (10 min):

Adaptar el main.py para que inicie la OfficeScene y le pase una instancia del Inventory.

Jugar la escena: hacer clic en la puerta (falla), hacer clic en la llave (éxito), y volver a hacer clic en la puerta (éxito).


1. Discusión: Más Allá del Clic - La Interfaz SCUMM
El "clic" es ambiguo. ¿Significa "mirar", "coger" o "hablar"? Los primeros juegos de aventura resolvían esto con un analizador de texto (parser), donde el jugador escribía "coger lámpara". La revolución llegó con la interfaz SCUMM (Script Creation Utility for Maniac Mansion) de LucasArts, que introdujo una cuadrícula de verbos en la pantalla.

Intención Explícita: El jugador primero selecciona un verbo ("Usar") y luego un objeto en el mundo ("puerta") o en su inventario ("llave"). Esto crea una estructura de "verbo + objeto" u "objeto + sobre objeto" que permite puzles mucho más complejos.

Reducción de la Frustración: Elimina el juego de "adivinar el verbo" que plagaba a los analizadores de texto. Las acciones posibles están claramente definidas.

El Cursor Inteligente: Una evolución posterior fue el "cursor inteligente" o la "rueda de acciones", como en Full Throttle o The Curse of Monkey Island. En lugar de una cuadrícula de verbos estática, al hacer clic en un objeto aparece un menú contextual con las acciones posibles. Este es el enfoque que adoptaremos, ya que es más moderno y menos intrusivo.

Ejemplos en juegos conocidos:

Maniac Mansion / Day of the Tentacle 🐙: Los ejemplos clásicos de la interfaz de verbos en la parte inferior de la pantalla. El jugador construye oraciones visualmente: "Usar" + "Hámster" + "en" + "Microondas".

Full Throttle 🏍️: Popularizó el menú de acciones contextual en forma de rueda. Al mantener pulsado el clic sobre un objeto, aparecía un icono de una calavera (usar/hablar), unos ojos (mirar) y una mano (coger).

Deponia 🌎: Una saga de aventuras moderna que utiliza una rueda de acciones similar, demostrando la vigencia del diseño. El jugador puede elegir entre "mirar", "usar/hablar" y "usar objeto del inventario".

2. Estructura de Archivos Actualizada
Añadimos un módulo cursor.py a nuestro motor para gestionar el estado y la apariencia de nuestro cursor inteligente.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- cursor.py            # ¡NUEVO! Nuestro cursor de acciones.
|
|-- adventure/
|   |-- assets/
|   |   |-- cursor_talk.png
|   |   |-- cursor_look.png
|   |   |-- cursor_use.png
|   |
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Motor Actualizado
engine/cursor.py (¡NUEVO!)
Este módulo gestionará el estado del cursor: ¿estamos en modo "mirar", "usar", o tenemos un objeto del inventario "cogido"?

Python

# engine/cursor.py
import pygame

class Cursor:
    def __init__(self):
        self.mode = 'default' # 'default', 'look', 'talk', 'use_item'
        self.selected_item = None # Nombre del objeto del inventario seleccionado
        
        # Cargamos las imágenes para cada modo
        self.cursors = {
            'default': pygame.mouse.get_cursor(),
            'look': pygame.image.load('adventure/assets/cursor_look.png').convert_alpha(),
            'talk': pygame.image.load('adventure/assets/cursor_talk.png').convert_alpha(),
        }
        pygame.mouse.set_visible(False) # Ocultamos el cursor del sistema

    def set_mode(self, mode, item_name=None):
        self.mode = mode
        self.selected_item = item_name
        print(f"Modo del cursor: {self.mode}, Objeto: {self.selected_item}")

    def draw(self, surface):
        """Dibuja el cursor personalizado en la posición del ratón."""
        pos = pygame.mouse.get_pos()
        if self.mode in self.cursors:
            cursor_image = self.cursors[self.mode]
            surface.blit(cursor_image, pos)
        # Si tenemos un objeto seleccionado, podríamos dibujarlo junto al cursor
        elif self.mode == 'use_item' and self.selected_item:
            # Lógica para dibujar la imagen del objeto en el cursor
            pass
4. El Código de la Aventura Gráfica Actualizado
Refactorizamos la escena y el inventario para que interactúen con el nuevo sistema de cursor.

engine/inventory.py (Actualizado)
Añadimos la lógica para que al hacer clic en un objeto del inventario, se "coja" con el cursor.

Python

# engine/inventory.py
import pygame

class Inventory:
    def __init__(self):
        self.items = {}
        self.item_rects = {} # Guardaremos los rects para detectar clics
        # ... (resto del __init__)

    def add_item(self, name, image_path):
        # ... (lógica existente) ...
        self.update_rects()

    def update_rects(self):
        """Actualiza las posiciones de los rects de los objetos."""
        self.item_rects.clear()
        x_offset = 10
        for name, image in self.items.items():
            item_rect = image.get_rect(topleft=(x_offset, pygame.display.get_surface().get_height() - 50))
            self.item_rects[name] = item_rect
            x_offset += image.get_width() + 10

    def check_click(self, pos, cursor):
        """Comprueba si se ha hecho clic en un objeto del inventario."""
        for name, rect in self.item_rects.items():
            if rect.collidepoint(pos):
                cursor.set_mode('use_item', item_name=name)
                return True
        return False

    def draw(self, surface):
        # ... (lógica de dibujado existente) ...
        # Dibuja cada objeto usando los rects guardados
        for name, rect in self.item_rects.items():
            surface.blit(self.items[name], rect)
adventure/scenes/office.py (Grandes Cambios)
La lógica de los hotspots ahora depende del modo del cursor.

Python

# adventure/scenes/office.py
# ... (imports) ...

class OfficeScene(RoomScene):
    # ... (__init__ sin cambios, define los hotspots) ...

    # Sobreescribimos handle_events para una lógica más compleja
    def handle_events(self, events):
        for event in events:
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos
                # Primero, comprobar si se ha hecho clic en el inventario
                if self.inventory.check_click(pos, self.scene_manager.cursor):
                    return # Si se coge un objeto, no interactuamos con el mundo

                # Si no, comprobar los hotspots de la escena
                for hotspot in self.hotspots:
                    if hotspot.check_click(pos):
                        # En lugar de llamar a un solo callback, pasamos el control
                        hotspot.callback(self.scene_manager.cursor.mode, self.scene_manager.cursor.selected_item)
                        # Reseteamos el cursor al modo por defecto después de cada acción
                        self.scene_manager.cursor.set_mode('default')
                        return

    # Los callbacks ahora reciben el modo del cursor y el objeto seleccionado
    def on_click_key(self, mode, item):
        if mode == 'default': # Clic normal
            print("Has encontrado una llave.")
            self.inventory.add_item("llave", "adventure/assets/item_key.png")
            # ...
    
    def on_click_door(self, mode, item):
        if mode == 'look':
            print("Es una puerta de madera robusta. Parece cerrada.")
        elif mode == 'use_item' and item == 'llave':
            print("Usas la llave y abres la puerta. ¡Has escapado!")
            # ...
        else: # Clic por defecto o usar otro objeto
            print("La puerta está cerrada.")

adventure/main.py (Pequeño Añadido)
El gestor de escenas ahora también debe crear y pasar el cursor.

Python

# adventure/main.py
# ...
from engine.cursor import Cursor
from engine.inventory import Inventory
from adventure.scenes.office import OfficeScene

def main():
    # ...
    cursor = Cursor()
    inventory = Inventory()
    # Pasamos el cursor y el inventario al gestor de escenas
    # (El gestor de escenas necesitará una pequeña modificación para guardar estos objetos
    # y pasarlos a cada nueva escena que cree)
    scene_manager = SceneManager(OfficeScene, pantalla, cursor=cursor, inventory=inventory)
    
    while ejecutando:
        # ...
        scene_manager.draw()
        cursor.draw(pantalla) # Dibujamos el cursor al final, por encima de todo
        pygame.display.flip()
    # ...


## Clase 24: Sistema de Verbos y Preparación para Diálogos
Contenidos de la Clase
Discusión sobre Interfaces de Aventura : Iniciar con la discusión sobre la evolución desde los parsers de texto hasta la interfaz SCUMM y los cursores inteligentes, usando los ejemplos de juegos mencionados.

Implementando el Cursor Inteligente :

Crear la clase Cursor en el motor. Explicar cómo gestiona un mode y un selected_item.

Implementar la lógica para ocultar el cursor del sistema y dibujar el cursor personalizado en cada frame.

Añadir la lógica al inventario para que, al hacer clic en un objeto, se llame a cursor.set_mode('use_item', ...).

Lógica de Interacción Basada en Verbos (45 min):

Refactorizar el handle_events de RoomScene o de la escena específica (OfficeScene).

El flujo de un clic ahora es:

¿Se hizo clic en la UI (inventario)? Si es así, actualizar el cursor y parar.

Si no, ¿se hizo clic en un hotspot?

Si es así, llamar al callback del hotspot, pasándole el cursor.mode y cursor.selected_item actuales.

Modificar los callbacks (on_click_door, etc.) para que contengan una estructura if/elif que reaccione de forma diferente según el modo del cursor.

Preparando el Terreno para la IA (10 min):

Discutir cómo este sistema de verbos se extiende naturalmente a los diálogos. Un personaje en el juego es simplemente otro Hotspot.

La acción "Hablar" sobre ese hotspot no resolverá un puzle, sino que iniciará una nueva "escena" o estado: la escena de diálogo.

Esta escena de diálogo será el componente que, en la próxima clase, conectaremos a una API de IA para generar las respuestas.








## Clase 25

Aprenderemos a diseñar la personalidad de un NPC, a comunicarnos con una API externa y a crear una interfaz de diálogo donde las conversaciones no están preescritas, sino que se generan en tiempo real.

1. Discusión: La Nueva Frontera de los NPCs
Tradicionalmente, los diálogos en los juegos se basan en un árbol de diálogo. Es una estructura con ramas donde cada opción que elige el jugador lleva a una respuesta preescrita por el desarrollador. Es efectivo, pero limitado y costoso de producir.

La IA Generativa rompe este paradigma:

Conversaciones Dinámicas: En lugar de seguir un guion, el NPC puede responder a preguntas abiertas, recordar partes de la conversación y reaccionar de formas inesperadas. El jugador puede escribir lo que quiera, no solo elegir de una lista.

Personalidad a través de Prompts: La "personalidad" del NPC no se programa con lógica if/else, sino que se diseña a través de un prompt de sistema. Este prompt es un conjunto de instrucciones que le damos a la IA para que actúe como un personaje específico.

Jugabilidad Emergente: Esto abre la puerta a nuevos tipos de puzles. Un puzle podría requerir que el jugador convenza al NPC, descubra una mentira en su discurso o le haga una pregunta tan específica que revele una pista oculta en su "conocimiento" de base.

Ejemplos y Potencial:

Juegos de Misterio 🕵️‍♀️: Imagina interrogar a un sospechoso que no tiene respuestas predefinidas. Podrías preguntarle sobre su coartada, sobre otros personajes o sobre un objeto que encontraste, y sus respuestas se generarían en tiempo real, manteniendo consistencia con su personalidad.

Simuladores Sociales 👨‍👩‍👧‍👦: NPCs en juegos como The Sims o Stardew Valley podrían tener conversaciones mucho más profundas y significativas, reaccionando a los eventos del juego de formas únicas.

Asistentes en el Juego 🤖: Un NPC podría actuar como un guía o un compañero que te da pistas sutiles y contextuales en lugar de respuestas directas, adaptándose a tu estilo de juego.

Nuestra Herramienta: La API de Gemini de Google

Usaremos la API de Gemini. Para ello, necesitarás:

Una cuenta de Google.

Crear un proyecto en Google AI Studio (aistudio.google.com).

Obtener una API Key gratuita. ¡Esta clave es secreta, no la compartas!

2. Estructura de Archivos Actualizada
Añadimos un módulo dialogue.py al motor y un archivo para la escena de diálogo en nuestro juego.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- dialogue.py          # ¡NUEVO! Gestor de diálogo y conexión con la API.
|
|-- adventure/
|   |-- scenes/
|   |   |-- ...
|   |   |-- dialogue_scene.py  # ¡NUEVO! La interfaz de conversación.
|   |
|   |-- config.py            # ¡NUEVO! Para guardar nuestra API key de forma segura.
|   |-- ... (resto)
|
|-- requirements.txt       # Añadimos google-generativeai
requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
google-generativeai
adventure/config.py (¡NUEVO Y MUY IMPORTANTE!)
Python

# adventure/config.py
# Pega tu API Key de Google AI Studio aquí.
# ¡NUNCA subas este archivo a un repositorio público como GitHub!
API_KEY = "TU_API_KEY_AQUI"
3. El Código del Motor Actualizado
engine/dialogue.py (¡NUEVO!)
Este módulo se encarga de la comunicación con la API de Gemini.

Python

# engine/dialogue.py
import google.generativeai as genai

class DialogueManager:
    def __init__(self, api_key, system_prompt):
        genai.configure(api_key=api_key)
        
        # Configuración del modelo
        generation_config = {
          "temperature": 0.9,
          "top_p": 1,
          "top_k": 1,
          "max_output_tokens": 2048,
        }
        
        self.model = genai.GenerativeModel(
            model_name="gemini-1.0-pro",
            generation_config=generation_config
        )
        
        # Iniciamos la conversación con el prompt del sistema
        self.conversation = self.model.start_chat(history=[
            {'role': 'user', 'parts': [system_prompt]},
            {'role': 'model', 'parts': ["Entendido. Estoy listo para actuar como este personaje."]}
        ])

    def send_message(self, user_input):
        """Envía el mensaje del jugador a la IA y devuelve la respuesta."""
        try:
            self.conversation.send_message(user_input)
            return self.conversation.last.text
        except Exception as e:
            print(f"Error en la API: {e}")
            return "Lo siento, mi mente está un poco confusa ahora mismo."
4. El Código de la Aventura Gráfica Actualizado
adventure/scenes/dialogue_scene.py (¡NUEVO!)
Esta es la interfaz visual para hablar con el NPC.

Python

# adventure/scenes/dialogue_scene.py
import pygame
from engine.scene import Scene
from engine.dialogue import DialogueManager
from ..config import API_KEY

class DialogueScene(Scene):
    def __init__(self, scene_manager, npc_prompt):
        super().__init__(scene_manager)
        self.dialogue_manager = DialogueManager(API_KEY, npc_prompt)
        
        self.history = [] # Historial de la conversación para mostrar en pantalla
        self.input_text = ""
        self.font = pygame.font.Font(None, 28)
        self.input_rect = pygame.Rect(50, 500, 700, 40)
        self.history.append("Te encuentras frente a un personaje. ¿Qué le dices?")

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RETURN:
                    if self.input_text:
                        # Añadimos la pregunta del jugador al historial
                        self.history.append(f"> {self.input_text}")
                        # Enviamos el mensaje a la IA
                        response = self.dialogue_manager.send_message(self.input_text)
                        # Añadimos la respuesta de la IA al historial
                        self.history.append(response)
                        self.input_text = "" # Limpiamos la entrada
                elif event.key == pygame.K_BACKSPACE:
                    self.input_text = self.input_text[:-1]
                else:
                    self.input_text += event.unicode
                
                # Limitar el historial para que no se salga de la pantalla
                if len(self.history) > 15:
                    self.history.pop(0)

    def draw(self, surface):
        surface.fill((30, 30, 50))
        
        # Dibujar historial
        y_offset = 20
        for line in self.history:
            text_surface = self.font.render(line, True, (255, 255, 255))
            surface.blit(text_surface, (20, y_offset))
            y_offset += 30
            
        # Dibujar caja de entrada
        pygame.draw.rect(surface, (80, 80, 100), self.input_rect)
        input_surface = self.font.render(self.input_text, True, (255, 255, 255))
        surface.blit(input_surface, (self.input_rect.x + 5, self.input_rect.y + 5))
adventure/scenes/office.py (Pequeño Añadido)
Ahora, un hotspot puede transicionar a la escena de diálogo.

Python

# adventure/scenes/office.py
# ...
from .dialogue_scene import DialogueScene

class OfficeScene(RoomScene):
    def __init__(self, scene_manager, inventory):
        # ...
        self.hotspots.append(Hotspot((100, 150, 150, 250), self.on_click_detective))

    def on_click_detective(self, mode, item):
        if mode == 'talk':
            # El "corazón" de la personalidad del NPC
            prompt = """
            Eres el Detective Jones, un investigador privado cínico y cansado del mundo.
            Estás en tu oficina investigando la desaparición de una valiosa estatuilla, 'El Halcón de Silicio'.
            Hablas con frases cortas y sarcásticas. No confías en nadie.
            Sabes que la estatuilla fue vista por última vez en el muelle 5.
            No reveles la pista del muelle 5 a menos que el jugador te pregunte específicamente por 'pistas' o 'dónde buscar'.
            Tu objetivo es ver si el jugador sabe algo, sin revelar demasiado.
            """
            self.scene_manager.go_to(DialogueScene, npc_prompt=prompt)
        elif mode == 'look':
            print("Es un detective con cara de pocos amigos. Mejor no molestarle... o sí.")
        # ...
## Clase 26: Diálogos con IA Generativa
Contenidos de la Clase
Discusión sobre IA en Juegos (20 min): Iniciar con la discusión sobre las limitaciones de los árboles de diálogo y el potencial revolucionario de la IA generativa, usando los ejemplos mencionados.

Configurando la API de Gemini:

Guiar a los alumnos a través del proceso de obtener una API Key en Google AI Studio.

Explicar la importancia de mantener la clave en un archivo config.py y nunca subirlo a un repositorio público.

Instalar la librería de Google (pip install google-generativeai).

El Arte de la Ingeniería de Prompts (40 min):

Presentar el DialogueManager y su system_prompt.

Explicar que el prompt es la parte más importante. Es donde "programamos" la personalidad. Desglosar un buen prompt:

Rol: "Eres el Detective Jones..."

Contexto: "...investigando la desaparición de 'El Halcón de Silicio'."

Personalidad/Tono: "...cínico y cansado. Hablas con frases cortas y sarcásticas."

Conocimiento Oculto: "Sabes que fue vista en el muelle 5."

Reglas/Condiciones: "No reveles la pista a menos que te pregunten por 'pistas'..."

Objetivo: "Tu objetivo es ver si el jugador sabe algo."

Creando la Interfaz de Diálogo (25 min):

Implementar la DialogueScene. Mostrar cómo gestiona un historial de texto y la entrada del teclado del jugador.

Conectar el handle_events para que al pulsar Enter, se envíe el texto al DialogueManager y se muestre la respuesta.

Conectar la OfficeScene para que al "hablar" con un personaje, se transicione a la DialogueScene, pasándole el prompt específico de ese personaje.







Sword Battle https://www.youtube.com/watch?v=rpP4VbzY7Ow

World Cup: https://www.youtube.com/watch?v=mPKoHVO5g3o